# Deslocamento vertical, dV

## Temperatura

### K=2

In [ ]:
# ==============================
# 0. Bibliotecas
# ==============================
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
from scipy.signal import savgol_filter
from sklearn.cluster import KMeans

# ==============================
# 1. Ler CSVs ASC, DESC e ORTHO
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"
ortho_v_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"
ortho_h_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_E_2019_2023_1/EGMS_L3_E27N18_100km_E_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)
ortho_v = pd.read_csv(ortho_v_file)
ortho_h = pd.read_csv(ortho_h_file)

# ==============================
# 2. Filtrar área de interesse
# ==============================
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

def filter_area(df):
    return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
              (df['easting'] >= este_min) & (df['easting'] <= este_max)]

asc = filter_area(asc)
desc = filter_area(desc)
ortho_v = filter_area(ortho_v)
ortho_h = filter_area(ortho_h)

# ==============================
# 3. Função melt para ASC/DESC
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]  # ajustar conforme colunas de datas
    long_df = df.melt(
        id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'],
        value_vars=disp_cols, var_name='date', value_name='disp'
    )
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)

common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(), desc_long['date'].max()),
    freq='MS'
)

# ==============================
# 4. Interpolação temporal linear
# ==============================
def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(
            pd.to_datetime(common_dates).astype(np.int64),
            group['date'].astype(np.int64),
            group['disp']
        )
        dfs.append(pd.DataFrame({
            'easting': x,
            'northing': y,
            'latitude': group['latitude'].iloc[0],
            'longitude': group['longitude'].iloc[0],
            'date': common_dates,
            'disp': interp,
            'incidence_angle': group['incidence_angle'].iloc[0],
            'track_angle': group['track_angle'].iloc[0]
        }))
    return pd.concat(dfs, ignore_index=True)

asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 5. Interpolação espacial IDW
# ==============================
def idw_interpolation_per_date(source_df, target_df, radius=150, power=2):
    out_list = []
    for date, src_group in source_df.groupby('date'):
        trg_group = target_df[target_df['date']==date].copy()
        if src_group.empty or trg_group.empty:
            continue
        src_points = np.array(list(zip(src_group['easting'], src_group['northing'])))
        trg_points = np.array(list(zip(trg_group['easting'], trg_group['northing'])))
        tree = cKDTree(src_points)
        dists, idxs = tree.query(trg_points, k=5, distance_upper_bound=radius)

        interpolated_disp = []
        interpolated_theta = []
        interpolated_alpha = []
        for dist, idx in zip(dists, idxs):
            mask = np.isfinite(dist)
            if not np.any(mask):
                interpolated_disp.append(np.nan)
                interpolated_theta.append(np.nan)
                interpolated_alpha.append(np.nan)
                continue
            weights = 1 / (dist[mask] ** power)
            interpolated_disp.append(np.sum(weights * src_group.iloc[idx[mask]]['disp']) / np.sum(weights))
            interpolated_theta.append(np.sum(weights * src_group.iloc[idx[mask]]['incidence_angle']) / np.sum(weights))
            interpolated_alpha.append(np.sum(weights * src_group.iloc[idx[mask]]['track_angle']) / np.sum(weights))

        trg_group['disp_idw'] = interpolated_disp
        trg_group['theta_desc'] = interpolated_theta
        trg_group['alpha_desc'] = interpolated_alpha
        out_list.append(trg_group)
    return pd.concat(out_list, ignore_index=True)

asc_interp = idw_interpolation_per_date(desc_interp, asc_interp)

# ==============================
# 6. Calcular β e γ
# ==============================
orbit_inclination = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orbit_inclination) * np.cos(np.deg2rad(asc_interp['latitude'])))
asc_interp['gamma'] = 0

# ==============================
# 7. Calcular dV e dH
# ==============================
def compute_dV_dH_real(row):
    θA = np.deg2rad(row['incidence_angle'])
    θD = np.deg2rad(row['theta_desc'])
    αA = np.deg2rad(row['track_angle'])
    αD = np.deg2rad(row['alpha_desc'])
    β = row['beta']
    γ = row['gamma']

    dASC_LOS = row['disp']
    dDESC_LOS = row['disp_idw']

    denom = (np.cos(θA)*np.sin(θD)*np.cos(β + γ) +
             np.cos(θD)*np.sin(θA)*np.cos(β - γ))

    dV = (dDESC_LOS*np.sin(θA)*np.cos(β - γ) +
          dASC_LOS*np.sin(θD)*np.cos(β + γ)) / denom
    dH = (dDESC_LOS*np.cos(θA) - dASC_LOS*np.cos(θD)) / denom
    return pd.Series({'dV': dV, 'dH': dH})

asc_interp[['dV','dH']] = asc_interp.apply(compute_dV_dH_real, axis=1)

# ==============================
# 8. Criar grelha centrada ORTHO
# ==============================
grid_size = 100
x_edges = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

asc_interp['cell_x'] = pd.cut(asc_interp['easting'], bins=x_edges_shifted, labels=False)
asc_interp['cell_y'] = pd.cut(asc_interp['northing'], bins=y_edges_shifted, labels=False)

# Remover NaNs antes de criar cell_id
asc_interp = asc_interp.dropna(subset=['cell_x','cell_y'])
asc_interp['cell_id'] = asc_interp['cell_x'].astype(int).astype(str) + "_" + asc_interp['cell_y'].astype(int).astype(str)

# Agrupar para ponto central de cada célula
points = asc_interp.groupby('cell_id').agg({'easting':'mean','northing':'mean'}).reset_index()
gdf_points = gpd.GeoDataFrame(
    points,
    geometry=gpd.points_from_xy(points['easting'], points['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

agg = asc_interp.groupby(['cell_x','cell_y','date']).agg(
    x_center=('easting','mean'),
    y_center=('northing','mean'),
    dV=('dV','mean'),
    dH=('dH','mean')
).reset_index()
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + "_" + agg['cell_y'].astype(int).astype(str)

# ==============================
# 9. Criar GeoDataFrame da grelha
# ==============================
grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({
            "cell_x": ix,
            "cell_y": iy,
            "cell_id": f"{ix}_{iy}",
            "geometry": box(x_edges_shifted[ix], y_edges_shifted[iy],
                            x_edges_shifted[ix+1], y_edges_shifted[iy+1])
        })
grid = gpd.GeoDataFrame(grid_data, crs="EPSG:3035").to_crs(epsg=3857)

# ==============================
# 10. Carregar temperatura
# ==============================
df_temp = pd.read_excel("data/alqueva_temp.xlsx")
df_temp['data'] = pd.to_datetime(df_temp['data'])
window = 365
df_temp['med_smooth'] = savgol_filter(df_temp['med'], window_length=window, polyorder=2)

# ==============================
# 11. Clustering de dV
# ==============================
agg_pivot = agg.pivot(index='cell_id', columns='date', values='dV').fillna(0)
k = 2
kmeans = KMeans(n_clusters=k, random_state=0)
cluster_labels = kmeans.fit_predict(agg_pivot)
cluster_df = pd.DataFrame({'cell_id': agg_pivot.index, 'cluster': cluster_labels})

# Mapear cores
cluster_colors = {i: color for i, color in enumerate(['red','green','blue','orange','purple'])}
grid_sel = grid.merge(cluster_df, on='cell_id', how='left')

import matplotlib.dates as mdates

# ==============================
# 14. Figura única: mapa + clusters
# ==============================
clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)

# Criar figura com 2 linhas: mapa + séries temporais
fig = plt.figure(figsize=(20, 18))
gs = fig.add_gridspec(2, n_clusters, height_ratios=[2, 1.2])

# ------------------------------
# Linha 1: Mapa com legenda técnica
# ------------------------------
ax_map = fig.add_subplot(gs[0, :])
grid.boundary.plot(ax=ax_map, color='lightgray', linewidth=0.5)
grid_sel.boundary.plot(ax=ax_map, color='black', linewidth=1, alpha=0.2)

# --- Células coloridas por cluster
for i, row in grid_sel.iterrows():
    if pd.notna(row['cluster']):
        gpd.GeoSeries([row['geometry']], crs=grid_sel.crs).plot(
            ax=ax_map,
            color=cluster_colors[int(row['cluster'])],
            alpha=0.4
        )

# --- Pontos centrais (ASC/DESC)
gdf_points.plot(ax=ax_map, color='white', edgecolor='black', markersize=40)

# Adicionar item de legenda para os pontos
ax_map.scatter([], [], marker='o', color='white', edgecolor='black', s=120,
               label='Pontos ASC/DESC')

# --- Adicionar basemap
ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)

# --- Legenda dos clusters
for cluster_id in clusters_present:
    color = cluster_colors[cluster_id]
    n_cells = len(cluster_df[cluster_df["cluster"] == cluster_id])
    ax_map.scatter([], [], color=color, alpha=0.6,
                   label=f'Cluster {cluster_id + 1} - {n_cells} células')

# # --- Caixa técnica (informações do processamento)
# textstr = '\n'.join((
#     f'Tamanho da grelha: {grid_size} x {grid_size} m',
#     f'Técnica de clustering: K-Means (k = {n_clusters})',
#     'Tipo de deslocamento: dV (vertical)',
#     'Base de dados: EGMS 2019–2023',
# ))
# ax_map.text(0.99, 0.01, textstr, transform=ax_map.transAxes,
#             fontsize=10, verticalalignment='bottom', horizontalalignment='right',
#             bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.7))

# --- Título e formatação
ax_map.set_title(
    f"Clusters de séries temporais de deslocamento vertical.\n"
    f"K-Means, K={k}.\n"
    f"Grelha {grid_size} m × {grid_size} m.\n"
    f"Correlação entre a série temporal média de cada cluster e a temperatura.",
    fontsize=16
)

ax_map.set_axis_off()
ax_map.legend(fontsize=10, loc='upper left')

# ------------------------------
# Linha 2: Séries temporais
# ------------------------------
# Escala comum para dV (mantida igual em todos os clusters)
dV_min = agg['dV'].min()
dV_max = agg['dV'].max()
dV_margin = (dV_max - dV_min) * 0.1  # margem de 10%

for idx, cluster_id in enumerate(clusters_present):
    ax = fig.add_subplot(gs[1, idx])
    
    # Selecionar células do cluster
    cluster_cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]

    # Plot das células individuais (cinzento)
    for cid in cluster_cells:
        ax.plot(cluster_data.columns, cluster_data.loc[cid], color='lightgray', alpha=0.7)

    # Média do cluster (linha colorida)
    cluster_mean_dV = cluster_data.mean(axis=0)
    cluster_color = cluster_colors[cluster_id]
    ax.plot(cluster_data.columns, cluster_mean_dV, color=cluster_color, linewidth=2.5,
            label=f'Média Cluster {cluster_id + 1}')

    # ---------- Temperatura média (achatada, mas com escala real) ----------
    ax2 = ax.twinx()

    temp_min = df_temp['med_smooth'].min()
    temp_max = df_temp['med_smooth'].max()

    # Normalizar temperatura para o intervalo de dV e achatar visualmente
    temp_visual = (df_temp['med_smooth'] - temp_min) / (temp_max - temp_min)
    temp_visual = temp_visual * (dV_max - dV_min) * 0.25 + (dV_max - (dV_max - dV_min) * 0.3)

    ax2.plot(df_temp['data'], temp_visual, color='black', linewidth=2.2, alpha=0.85, label='Temperatura média (°C)')
    ax2.set_ylabel("Temperatura (°C)", color='black')
    ax2.tick_params(axis='y', labelcolor='black')

    # Definir ticks reais para o eixo da direita
    temp_ticks_real = np.linspace(temp_min, temp_max, 6)
    temp_ticks_visual = (temp_ticks_real - temp_min) / (temp_max - temp_min)
    temp_ticks_visual = temp_ticks_visual * (dV_max - dV_min) * 0.25 + (dV_max - (dV_max - dV_min) * 0.3)
    ax2.set_yticks(temp_ticks_visual)
    ax2.set_yticklabels([f"{t:.0f}" for t in temp_ticks_real])

    # ---------- Eixos e estilo ----------
    ax.set_ylim(dV_min - dV_margin, dV_max + dV_margin)
    ax2.set_ylim(dV_min - dV_margin, dV_max + dV_margin)

    ax.set_title(f'Cluster {cluster_id + 1} - {len(cluster_cells)} células', fontsize=12)
    #ax.set_xlabel('Ano')
    ax.set_ylabel('dV (mm)')
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

    ax.grid(False)
    ax2.grid(False)

    # ---------- Legenda combinada ----------
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, fontsize=9, loc='upper left')

plt.tight_layout()
plt.show()

o mesmo mas com o recorte da barragem

In [ ]:
# ==============================
# SCRIPT COMPLETO: Clustering e plots apenas da barragem
# ==============================

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
from scipy.signal import savgol_filter
from sklearn.cluster import KMeans
import matplotlib.dates as mdates

# ==============================
# 0. Ler COS e definir barragem
# ==============================
COS_PATH = r"C:\projetos\analise_insar_ist\data\COS2023v1-S2-shp\COS2023v1-S2.shp"
cos = gpd.read_file(COS_PATH).to_crs(epsg=3035)
barragem = cos[cos['COS23_n4_L'] == 'Infraestruturas de produção de energia hídrica']

# ==============================
# 1. Ler CSVs ASC/DESC e filtrar área aproximada
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)

# Limites aproximados (para acelerar o processamento)
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

def filter_area(df):
    return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
              (df['easting'] >= este_min) & (df['easting'] <= este_max)]

asc = filter_area(asc)
desc = filter_area(desc)

# ==============================
# 2. Melt para ASC/DESC
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]  # ajustar conforme colunas de datas
    long_df = df.melt(
        id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'],
        value_vars=disp_cols, var_name='date', value_name='disp'
    )
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)

common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(), desc_long['date'].max()),
    freq='MS'
)

# ==============================
# 3. Interpolação temporal linear
# ==============================
def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(pd.to_datetime(common_dates).astype(np.int64),
                           group['date'].astype(np.int64), group['disp'])
        dfs.append(pd.DataFrame({
            'easting': x, 'northing': y,
            'latitude': group['latitude'].iloc[0],
            'longitude': group['longitude'].iloc[0],
            'date': common_dates,
            'disp': interp,
            'incidence_angle': group['incidence_angle'].iloc[0],
            'track_angle': group['track_angle'].iloc[0]
        }))
    return pd.concat(dfs, ignore_index=True)

asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 4. Interpolação espacial IDW
# ==============================
def idw_interpolation_per_date(source_df, target_df, radius=150, power=2):
    out_list = []
    for date, src_group in source_df.groupby('date'):
        trg_group = target_df[target_df['date']==date].copy()
        if src_group.empty or trg_group.empty:
            continue
        tree = cKDTree(list(zip(src_group['easting'], src_group['northing'])))
        dists, idxs = tree.query(list(zip(trg_group['easting'], trg_group['northing'])),
                                 k=5, distance_upper_bound=radius)
        interpolated_disp = []
        interpolated_theta = []
        interpolated_alpha = []
        for dist, idx in zip(dists, idxs):
            mask = np.isfinite(dist)
            if not np.any(mask):
                interpolated_disp.append(np.nan)
                interpolated_theta.append(np.nan)
                interpolated_alpha.append(np.nan)
                continue
            weights = 1 / (dist[mask] ** power)
            interpolated_disp.append(np.sum(weights * src_group.iloc[idx[mask]]['disp']) / np.sum(weights))
            interpolated_theta.append(np.sum(weights * src_group.iloc[idx[mask]]['incidence_angle']) / np.sum(weights))
            interpolated_alpha.append(np.sum(weights * src_group.iloc[idx[mask]]['track_angle']) / np.sum(weights))
        trg_group['disp_idw'] = interpolated_disp
        trg_group['theta_desc'] = interpolated_theta
        trg_group['alpha_desc'] = interpolated_alpha
        out_list.append(trg_group)
    return pd.concat(out_list, ignore_index=True)

asc_interp = idw_interpolation_per_date(desc_interp, asc_interp)

# ==============================
# 5. Calcular beta, gamma, dV, dH
# ==============================
orbit_inclination = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orbit_inclination) * np.cos(np.deg2rad(asc_interp['latitude'])))
asc_interp['gamma'] = 0

def compute_dV_dH_real(row):
    θA = np.deg2rad(row['incidence_angle'])
    θD = np.deg2rad(row['theta_desc'])
    αA = np.deg2rad(row['track_angle'])
    αD = np.deg2rad(row['alpha_desc'])
    β = row['beta']
    γ = row['gamma']
    dASC_LOS = row['disp']
    dDESC_LOS = row['disp_idw']
    denom = (np.cos(θA)*np.sin(θD)*np.cos(β + γ) +
             np.cos(θD)*np.sin(θA)*np.cos(β - γ))
    dV = (dDESC_LOS*np.sin(θA)*np.cos(β - γ) +
          dASC_LOS*np.sin(θD)*np.cos(β + γ)) / denom
    dH = (dDESC_LOS*np.cos(θA) - dASC_LOS*np.cos(θD)) / denom
    return pd.Series({'dV': dV, 'dH': dH})

asc_interp[['dV','dH']] = asc_interp.apply(compute_dV_dH_real, axis=1)

# ==============================
# 6. Criar grelha sobre asc_interp
# ==============================
grid_size = 100
x_edges = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

# Criar colunas cell_x, cell_y, cell_id
asc_interp['cell_x'] = pd.cut(asc_interp['easting'], bins=x_edges_shifted, labels=False)
asc_interp['cell_y'] = pd.cut(asc_interp['northing'], bins=y_edges_shifted, labels=False)
asc_interp = asc_interp.dropna(subset=['cell_x','cell_y'])
asc_interp['cell_id'] = asc_interp['cell_x'].astype(int).astype(str) + "_" + asc_interp['cell_y'].astype(int).astype(str)

# Criar GeoDataFrame da grelha
grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({
            'cell_x': ix,
            'cell_y': iy,
            'cell_id': f"{ix}_{iy}",
            'geometry': box(x_edges_shifted[ix], y_edges_shifted[iy],
                            x_edges_shifted[ix+1], y_edges_shifted[iy+1])
        })
grid = gpd.GeoDataFrame(grid_data, crs='EPSG:3035')

# Recorte à barragem
grid_barragem = gpd.overlay(grid, barragem, how='intersection').to_crs(epsg=3857)

# ==============================
# 7. Clustering apenas células da barragem
# ==============================
agg = asc_interp.groupby(['cell_x','cell_y','date']).agg(dV=('dV','mean')).reset_index()
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + '_' + agg['cell_y'].astype(int).astype(str)

# Considerar apenas células que intersectam a barragem
agg = agg[agg['cell_id'].isin(grid_barragem['cell_id'])]

agg_pivot = agg.pivot(index='cell_id', columns='date', values='dV').fillna(0)

k = 2
kmeans = KMeans(n_clusters=k, random_state=0)
cluster_labels = kmeans.fit_predict(agg_pivot)
cluster_df = pd.DataFrame({'cell_id': agg_pivot.index, 'cluster': cluster_labels})
cluster_colors = {i: color for i, color in enumerate(['red','green','blue','orange','purple'])}

grid_sel = grid_barragem.merge(cluster_df, on='cell_id', how='left')

# ==============================
# 8. Plot mapa + séries temporais
# ==============================
# Pontos centrais
points = asc_interp[asc_interp['cell_id'].isin(grid_barragem['cell_id'])].groupby('cell_id').agg(
    {'easting':'mean','northing':'mean'}).reset_index()
gdf_points = gpd.GeoDataFrame(
    points,
    geometry=gpd.points_from_xy(points['easting'], points['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

# Plot
fig = plt.figure(figsize=(20,18))
clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)
gs = fig.add_gridspec(2, n_clusters, height_ratios=[2, 1.2])

# Linha 1: mapa
ax_map = fig.add_subplot(gs[0, :])
grid_barragem.boundary.plot(ax=ax_map, color='lightgray', linewidth=0.5)
grid_sel.boundary.plot(ax=ax_map, color='black', linewidth=1, alpha=0.2)

for i, row in grid_sel.iterrows():
    if pd.notna(row['cluster']):
        gpd.GeoSeries([row['geometry']], crs=grid_sel.crs).plot(
            ax=ax_map, color=cluster_colors[int(row['cluster'])], alpha=0.4
        )

gdf_points.plot(ax=ax_map, color='white', edgecolor='black', markersize=40)
ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)
ax_map.set_axis_off()

# Linha 2: séries temporais
dV_min = agg['dV'].min()
dV_max = agg['dV'].max()
dV_margin = (dV_max - dV_min)*0.1

# Carregar temperatura
df_temp = pd.read_excel("data/alqueva_temp.xlsx")
df_temp['data'] = pd.to_datetime(df_temp['data'])
window = 365
df_temp['med_smooth'] = savgol_filter(df_temp['med'], window_length=window, polyorder=2)

for idx, cluster_id in enumerate(clusters_present):
    ax = fig.add_subplot(gs[1, idx])
    cluster_cells = cluster_df[cluster_df['cluster']==cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]

    for cid in cluster_cells:
        ax.plot(cluster_data.columns, cluster_data.loc[cid], color='lightgray', alpha=0.7)

    cluster_mean_dV = cluster_data.mean(axis=0)
    cluster_color = cluster_colors[cluster_id]
    ax.plot(cluster_data.columns, cluster_mean_dV, color=cluster_color, linewidth=2.5,
            label=f'Média Cluster {cluster_id+1}')

    # Temperatura
    ax2 = ax.twinx()
    temp_min = df_temp['med_smooth'].min()
    temp_max = df_temp['med_smooth'].max()
    temp_visual = (df_temp['med_smooth'] - temp_min) / (temp_max - temp_min)
    temp_visual = temp_visual*(dV_max-dV_min)*0.25 + (dV_max-(dV_max-dV_min)*0.3)
    ax2.plot(df_temp['data'], temp_visual, color='black', linewidth=2.2, alpha=0.85,
             label='Temperatura média (°C)')
    ax2.set_ylabel("Temperatura (°C)", color='black')
    ax2.tick_params(axis='y', labelcolor='black')

    # Eixos
    ax.set_ylim(dV_min-dV_margin, dV_max+dV_margin)
    ax2.set_ylim(dV_min-dV_margin, dV_max+dV_margin)
    ax.set_title(f'Cluster {cluster_id+1} - {len(cluster_cells)} células', fontsize=12)
    ax.set_ylabel('dV (mm)')
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.grid(False)
    ax2.grid(False)

    # Legenda combinada
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1+lines2, labels1+labels2, fontsize=9, loc='upper left')

plt.tight_layout()
plt.show()



In [ ]:
# ==============================
# SCRIPT COMPLETO: Clustering e plots apenas da barragem
# ==============================

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
from scipy.signal import savgol_filter
from sklearn.cluster import KMeans
import matplotlib.dates as mdates

# ==============================
# 0. Ler COS e definir barragem
# ==============================
COS_PATH = r"C:\projetos\analise_insar_ist\data\COS2023v1-S2-shp\COS2023v1-S2.shp"
cos = gpd.read_file(COS_PATH).to_crs(epsg=3035)
barragem = cos[cos['COS23_n4_L'] == 'Infraestruturas de produção de energia hídrica']

# ==============================
# 1. Ler CSVs ASC/DESC e filtrar área aproximada
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)

# Limites aproximados (para acelerar o processamento)
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

def filter_area(df):
    return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
              (df['easting'] >= este_min) & (df['easting'] <= este_max)]

asc = filter_area(asc)
desc = filter_area(desc)

# ==============================
# 2. Melt para ASC/DESC
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]  # ajustar conforme colunas de datas
    long_df = df.melt(
        id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'],
        value_vars=disp_cols, var_name='date', value_name='disp'
    )
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)

common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(), desc_long['date'].max()),
    freq='MS'
)

# ==============================
# 3. Interpolação temporal linear
# ==============================
def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(pd.to_datetime(common_dates).astype(np.int64),
                           group['date'].astype(np.int64), group['disp'])
        dfs.append(pd.DataFrame({
            'easting': x, 'northing': y,
            'latitude': group['latitude'].iloc[0],
            'longitude': group['longitude'].iloc[0],
            'date': common_dates,
            'disp': interp,
            'incidence_angle': group['incidence_angle'].iloc[0],
            'track_angle': group['track_angle'].iloc[0]
        }))
    return pd.concat(dfs, ignore_index=True)

asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 4. Interpolação espacial IDW
# ==============================
def idw_interpolation_per_date(source_df, target_df, radius=150, power=2):
    out_list = []
    for date, src_group in source_df.groupby('date'):
        trg_group = target_df[target_df['date']==date].copy()
        if src_group.empty or trg_group.empty:
            continue
        tree = cKDTree(list(zip(src_group['easting'], src_group['northing'])))
        dists, idxs = tree.query(list(zip(trg_group['easting'], trg_group['northing'])),
                                 k=5, distance_upper_bound=radius)
        interpolated_disp = []
        interpolated_theta = []
        interpolated_alpha = []
        for dist, idx in zip(dists, idxs):
            mask = np.isfinite(dist)
            if not np.any(mask):
                interpolated_disp.append(np.nan)
                interpolated_theta.append(np.nan)
                interpolated_alpha.append(np.nan)
                continue
            weights = 1 / (dist[mask] ** power)
            interpolated_disp.append(np.sum(weights * src_group.iloc[idx[mask]]['disp']) / np.sum(weights))
            interpolated_theta.append(np.sum(weights * src_group.iloc[idx[mask]]['incidence_angle']) / np.sum(weights))
            interpolated_alpha.append(np.sum(weights * src_group.iloc[idx[mask]]['track_angle']) / np.sum(weights))
        trg_group['disp_idw'] = interpolated_disp
        trg_group['theta_desc'] = interpolated_theta
        trg_group['alpha_desc'] = interpolated_alpha
        out_list.append(trg_group)
    return pd.concat(out_list, ignore_index=True)

asc_interp = idw_interpolation_per_date(desc_interp, asc_interp)

# ==============================
# 5. Calcular beta, gamma, dV, dH
# ==============================
orbit_inclination = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orbit_inclination) * np.cos(np.deg2rad(asc_interp['latitude'])))
asc_interp['gamma'] = 0

def compute_dV_dH_real(row):
    θA = np.deg2rad(row['incidence_angle'])
    θD = np.deg2rad(row['theta_desc'])
    αA = np.deg2rad(row['track_angle'])
    αD = np.deg2rad(row['alpha_desc'])
    β = row['beta']
    γ = row['gamma']
    dASC_LOS = row['disp']
    dDESC_LOS = row['disp_idw']
    denom = (np.cos(θA)*np.sin(θD)*np.cos(β + γ) +
             np.cos(θD)*np.sin(θA)*np.cos(β - γ))
    dV = (dDESC_LOS*np.sin(θA)*np.cos(β - γ) +
          dASC_LOS*np.sin(θD)*np.cos(β + γ)) / denom
    dH = (dDESC_LOS*np.cos(θA) - dASC_LOS*np.cos(θD)) / denom
    return pd.Series({'dV': dV, 'dH': dH})

asc_interp[['dV','dH']] = asc_interp.apply(compute_dV_dH_real, axis=1)

# ==============================
# 6. Criar grelha sobre asc_interp
# ==============================
grid_size = 100
x_edges = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

# Criar colunas cell_x, cell_y, cell_id
asc_interp['cell_x'] = pd.cut(asc_interp['easting'], bins=x_edges_shifted, labels=False)
asc_interp['cell_y'] = pd.cut(asc_interp['northing'], bins=y_edges_shifted, labels=False)
asc_interp = asc_interp.dropna(subset=['cell_x','cell_y'])
asc_interp['cell_id'] = asc_interp['cell_x'].astype(int).astype(str) + "_" + asc_interp['cell_y'].astype(int).astype(str)

# Criar GeoDataFrame da grelha
grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({
            'cell_x': ix,
            'cell_y': iy,
            'cell_id': f"{ix}_{iy}",
            'geometry': box(x_edges_shifted[ix], y_edges_shifted[iy],
                            x_edges_shifted[ix+1], y_edges_shifted[iy+1])
        })
grid = gpd.GeoDataFrame(grid_data, crs='EPSG:3035')

# Recorte à barragem
grid_barragem = gpd.overlay(grid, barragem, how='intersection').to_crs(epsg=3857)

# ==============================
# 7. Clustering apenas células da barragem
# ==============================
agg = asc_interp.groupby(['cell_x','cell_y','date']).agg(dV=('dV','mean')).reset_index()
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + '_' + agg['cell_y'].astype(int).astype(str)

# Considerar apenas células que intersectam a barragem
agg = agg[agg['cell_id'].isin(grid_barragem['cell_id'])]

agg_pivot = agg.pivot(index='cell_id', columns='date', values='dV').fillna(0)

k = 3
kmeans = KMeans(n_clusters=k, random_state=0)
cluster_labels = kmeans.fit_predict(agg_pivot)
cluster_df = pd.DataFrame({'cell_id': agg_pivot.index, 'cluster': cluster_labels})
cluster_colors = {i: color for i, color in enumerate(['red','green','blue','orange','purple'])}

grid_sel = grid_barragem.merge(cluster_df, on='cell_id', how='left')

# ==============================
# 8. Plot mapa + séries temporais
# ==============================
# Pontos centrais
points = asc_interp[asc_interp['cell_id'].isin(grid_barragem['cell_id'])].groupby('cell_id').agg(
    {'easting':'mean','northing':'mean'}).reset_index()
gdf_points = gpd.GeoDataFrame(
    points,
    geometry=gpd.points_from_xy(points['easting'], points['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

# Plot
fig = plt.figure(figsize=(20,18))
clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)
gs = fig.add_gridspec(2, n_clusters, height_ratios=[2, 1.2])

# Linha 1: mapa
ax_map = fig.add_subplot(gs[0, :])

# Grelha da barragem com cor azul claro, transparente
grid_barragem.plot(ax=ax_map, facecolor='none', edgecolor='white', linewidth=1, alpha=0.8)

# Células selecionadas com cluster
for i, row in grid_sel.iterrows():
    if pd.notna(row['cluster']):
        gpd.GeoSeries([row['geometry']], crs=grid_sel.crs).plot(
            ax=ax_map, color=cluster_colors[int(row['cluster'])], alpha=0.4
        )

# Pontos centrais
#gdf_points.plot(ax=ax_map, color='white', edgecolor='black', markersize=40)

# Basemap
ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)
ax_map.set_axis_off()


# Linha 2: séries temporais
dV_min = agg['dV'].min()
dV_max = agg['dV'].max()
dV_margin = (dV_max - dV_min)*0.1

# Carregar temperatura
df_temp = pd.read_excel("data/alqueva_temp.xlsx")
df_temp['data'] = pd.to_datetime(df_temp['data'])
window = 365
df_temp['med_smooth'] = savgol_filter(df_temp['med'], window_length=window, polyorder=2)

for idx, cluster_id in enumerate(clusters_present):
    ax = fig.add_subplot(gs[1, idx])
    cluster_cells = cluster_df[cluster_df['cluster']==cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]

    for cid in cluster_cells:
        ax.plot(cluster_data.columns, cluster_data.loc[cid], color='lightgray', alpha=0.7)

    cluster_mean_dV = cluster_data.mean(axis=0)
    cluster_color = cluster_colors[cluster_id]
    ax.plot(cluster_data.columns, cluster_mean_dV, color=cluster_color, linewidth=2.5,
            label=f'Média Cluster {cluster_id+1}')

    # ---------- Temperatura achatada, mas eixo com valores reais ----------
    ax2 = ax.twinx()

    temp_min = df_temp['med_smooth'].min()
    temp_max = df_temp['med_smooth'].max()

    # Normalizar temperatura para intervalo de dV (achatada visualmente)
    temp_visual = (df_temp['med_smooth'] - temp_min) / (temp_max - temp_min)
    temp_visual = temp_visual * (dV_max - dV_min) * 0.25 + (dV_max - (dV_max - dV_min) * 0.3)

    ax2.plot(df_temp['data'], temp_visual, color='black', linewidth=2, alpha=0.85, label='Temperatura média (°C)')
    ax2.set_ylabel("Temperatura (°C)", color='black')
    ax2.tick_params(axis='y', labelcolor='black')

    # Definir ticks reais do eixo direito, mas posicionados na escala achatada
    temp_ticks_real = np.linspace(temp_min, temp_max, 6)
    temp_ticks_visual = (temp_ticks_real - temp_min) / (temp_max - temp_min)
    temp_ticks_visual = temp_ticks_visual * (dV_max - dV_min) * 0.25 + (dV_max - (dV_max - dV_min) * 0.3)
    ax2.set_yticks(temp_ticks_visual)
    ax2.set_yticklabels([f"{t:.0f}" for t in temp_ticks_real])


    # Eixos
    ax.set_ylim(dV_min-dV_margin, dV_max+dV_margin)
    ax2.set_ylim(dV_min-dV_margin, dV_max+dV_margin)
    ax.set_title(f'Cluster {cluster_id+1} - {len(cluster_cells)} células', fontsize=12)
    ax.set_ylabel('dV (mm)')
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.grid(False)
    ax2.grid(False)

    # Legenda combinada
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1+lines2, labels1+labels2, fontsize=9, loc='upper left')

plt.tight_layout()
plt.show()


### K=3

In [ ]:
# ==============================
# 0. Bibliotecas
# ==============================
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
from scipy.signal import savgol_filter
from sklearn.cluster import KMeans

# ==============================
# 1. Ler CSVs ASC, DESC e ORTHO
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"
ortho_v_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"
ortho_h_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_E_2019_2023_1/EGMS_L3_E27N18_100km_E_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)
ortho_v = pd.read_csv(ortho_v_file)
ortho_h = pd.read_csv(ortho_h_file)

# ==============================
# 2. Filtrar área de interesse
# ==============================
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

def filter_area(df):
    return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
              (df['easting'] >= este_min) & (df['easting'] <= este_max)]

asc = filter_area(asc)
desc = filter_area(desc)
ortho_v = filter_area(ortho_v)
ortho_h = filter_area(ortho_h)

# ==============================
# 3. Função melt para ASC/DESC
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]  # ajustar conforme colunas de datas
    long_df = df.melt(
        id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'],
        value_vars=disp_cols, var_name='date', value_name='disp'
    )
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)

common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(), desc_long['date'].max()),
    freq='MS'
)

# ==============================
# 4. Interpolação temporal linear
# ==============================
def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(
            pd.to_datetime(common_dates).astype(np.int64),
            group['date'].astype(np.int64),
            group['disp']
        )
        dfs.append(pd.DataFrame({
            'easting': x,
            'northing': y,
            'latitude': group['latitude'].iloc[0],
            'longitude': group['longitude'].iloc[0],
            'date': common_dates,
            'disp': interp,
            'incidence_angle': group['incidence_angle'].iloc[0],
            'track_angle': group['track_angle'].iloc[0]
        }))
    return pd.concat(dfs, ignore_index=True)

asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 5. Interpolação espacial IDW
# ==============================
def idw_interpolation_per_date(source_df, target_df, radius=150, power=2):
    out_list = []
    for date, src_group in source_df.groupby('date'):
        trg_group = target_df[target_df['date']==date].copy()
        if src_group.empty or trg_group.empty:
            continue
        src_points = np.array(list(zip(src_group['easting'], src_group['northing'])))
        trg_points = np.array(list(zip(trg_group['easting'], trg_group['northing'])))
        tree = cKDTree(src_points)
        dists, idxs = tree.query(trg_points, k=5, distance_upper_bound=radius)

        interpolated_disp = []
        interpolated_theta = []
        interpolated_alpha = []
        for dist, idx in zip(dists, idxs):
            mask = np.isfinite(dist)
            if not np.any(mask):
                interpolated_disp.append(np.nan)
                interpolated_theta.append(np.nan)
                interpolated_alpha.append(np.nan)
                continue
            weights = 1 / (dist[mask] ** power)
            interpolated_disp.append(np.sum(weights * src_group.iloc[idx[mask]]['disp']) / np.sum(weights))
            interpolated_theta.append(np.sum(weights * src_group.iloc[idx[mask]]['incidence_angle']) / np.sum(weights))
            interpolated_alpha.append(np.sum(weights * src_group.iloc[idx[mask]]['track_angle']) / np.sum(weights))

        trg_group['disp_idw'] = interpolated_disp
        trg_group['theta_desc'] = interpolated_theta
        trg_group['alpha_desc'] = interpolated_alpha
        out_list.append(trg_group)
    return pd.concat(out_list, ignore_index=True)

asc_interp = idw_interpolation_per_date(desc_interp, asc_interp)

# ==============================
# 6. Calcular β e γ
# ==============================
orbit_inclination = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orbit_inclination) * np.cos(np.deg2rad(asc_interp['latitude'])))
asc_interp['gamma'] = 0

# ==============================
# 7. Calcular dV e dH
# ==============================
def compute_dV_dH_real(row):
    θA = np.deg2rad(row['incidence_angle'])
    θD = np.deg2rad(row['theta_desc'])
    αA = np.deg2rad(row['track_angle'])
    αD = np.deg2rad(row['alpha_desc'])
    β = row['beta']
    γ = row['gamma']

    dASC_LOS = row['disp']
    dDESC_LOS = row['disp_idw']

    denom = (np.cos(θA)*np.sin(θD)*np.cos(β + γ) +
             np.cos(θD)*np.sin(θA)*np.cos(β - γ))

    dV = (dDESC_LOS*np.sin(θA)*np.cos(β - γ) +
          dASC_LOS*np.sin(θD)*np.cos(β + γ)) / denom
    dH = (dDESC_LOS*np.cos(θA) - dASC_LOS*np.cos(θD)) / denom
    return pd.Series({'dV': dV, 'dH': dH})

asc_interp[['dV','dH']] = asc_interp.apply(compute_dV_dH_real, axis=1)

# ==============================
# 8. Criar grelha centrada ORTHO
# ==============================
grid_size = 100
x_edges = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

asc_interp['cell_x'] = pd.cut(asc_interp['easting'], bins=x_edges_shifted, labels=False)
asc_interp['cell_y'] = pd.cut(asc_interp['northing'], bins=y_edges_shifted, labels=False)

# Remover NaNs antes de criar cell_id
asc_interp = asc_interp.dropna(subset=['cell_x','cell_y'])
asc_interp['cell_id'] = asc_interp['cell_x'].astype(int).astype(str) + "_" + asc_interp['cell_y'].astype(int).astype(str)

# Agrupar para ponto central de cada célula
points = asc_interp.groupby('cell_id').agg({'easting':'mean','northing':'mean'}).reset_index()
gdf_points = gpd.GeoDataFrame(
    points,
    geometry=gpd.points_from_xy(points['easting'], points['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

agg = asc_interp.groupby(['cell_x','cell_y','date']).agg(
    x_center=('easting','mean'),
    y_center=('northing','mean'),
    dV=('dV','mean'),
    dH=('dH','mean')
).reset_index()
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + "_" + agg['cell_y'].astype(int).astype(str)

# ==============================
# 9. Criar GeoDataFrame da grelha
# ==============================
grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({
            "cell_x": ix,
            "cell_y": iy,
            "cell_id": f"{ix}_{iy}",
            "geometry": box(x_edges_shifted[ix], y_edges_shifted[iy],
                            x_edges_shifted[ix+1], y_edges_shifted[iy+1])
        })
grid = gpd.GeoDataFrame(grid_data, crs="EPSG:3035").to_crs(epsg=3857)

# ==============================
# 10. Carregar temperatura
# ==============================
df_temp = pd.read_excel("data/alqueva_temp.xlsx")
df_temp['data'] = pd.to_datetime(df_temp['data'])
window = 365
df_temp['med_smooth'] = savgol_filter(df_temp['med'], window_length=window, polyorder=2)

# ==============================
# 11. Clustering de dV
# ==============================
agg_pivot = agg.pivot(index='cell_id', columns='date', values='dV').fillna(0)
k = 3
kmeans = KMeans(n_clusters=k, random_state=0)
cluster_labels = kmeans.fit_predict(agg_pivot)
cluster_df = pd.DataFrame({'cell_id': agg_pivot.index, 'cluster': cluster_labels})

# Mapear cores
cluster_colors = {i: color for i, color in enumerate(['red','green','blue','orange','purple'])}
grid_sel = grid.merge(cluster_df, on='cell_id', how='left')

import matplotlib.dates as mdates

# ==============================
# 14. Figura única: mapa + clusters
# ==============================
clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)

# Criar figura com 2 linhas: mapa + séries temporais
fig = plt.figure(figsize=(20, 18))
gs = fig.add_gridspec(2, n_clusters, height_ratios=[2, 1.2])

# ------------------------------
# Linha 1: Mapa com legenda técnica
# ------------------------------
ax_map = fig.add_subplot(gs[0, :])
grid.boundary.plot(ax=ax_map, color='lightgray', linewidth=0.5)
grid_sel.boundary.plot(ax=ax_map, color='black', linewidth=1, alpha=0.2)

# --- Células coloridas por cluster
for i, row in grid_sel.iterrows():
    if pd.notna(row['cluster']):
        gpd.GeoSeries([row['geometry']], crs=grid_sel.crs).plot(
            ax=ax_map,
            color=cluster_colors[int(row['cluster'])],
            alpha=0.4
        )

# --- Pontos centrais (ASC/DESC)
gdf_points.plot(ax=ax_map, color='white', edgecolor='black', markersize=40)

# Adicionar item de legenda para os pontos
ax_map.scatter([], [], marker='o', color='white', edgecolor='black', s=120,
               label='Pontos ASC/DESC')

# --- Adicionar basemap
ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)

# --- Legenda dos clusters
for cluster_id in clusters_present:
    color = cluster_colors[cluster_id]
    n_cells = len(cluster_df[cluster_df["cluster"] == cluster_id])
    ax_map.scatter([], [], color=color, alpha=0.6,
                   label=f'Cluster {cluster_id + 1} - {n_cells} células')

# # --- Caixa técnica (informações do processamento)
# textstr = '\n'.join((
#     f'Tamanho da grelha: {grid_size} x {grid_size} m',
#     f'Técnica de clustering: K-Means (k = {n_clusters})',
#     'Tipo de deslocamento: dV (vertical)',
#     'Base de dados: EGMS 2019–2023',
# ))
# ax_map.text(0.99, 0.01, textstr, transform=ax_map.transAxes,
#             fontsize=10, verticalalignment='bottom', horizontalalignment='right',
#             bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.7))

# --- Título e formatação
ax_map.set_title(
    f"Clusters de séries temporais de deslocamento vertical.\n"
    f"K-Means, K={k}.\n"
    f"Grelha {grid_size} m × {grid_size} m.\n"
    f"Correlação entre a série temporal média de cada cluster e a temperatura.",
    fontsize=16
)
ax_map.set_axis_off()
ax_map.legend(fontsize=10, loc='upper left')

# ------------------------------
# Linha 2: Séries temporais
# ------------------------------
# Escala comum para dV (mantida igual em todos os clusters)
dV_min = agg['dV'].min()
dV_max = agg['dV'].max()
dV_margin = (dV_max - dV_min) * 0.1  # margem de 10%

for idx, cluster_id in enumerate(clusters_present):
    ax = fig.add_subplot(gs[1, idx])
    
    # Selecionar células do cluster
    cluster_cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]

    # Plot das células individuais (cinzento)
    for cid in cluster_cells:
        ax.plot(cluster_data.columns, cluster_data.loc[cid], color='lightgray', alpha=0.7)

    # Média do cluster (linha colorida)
    cluster_mean_dV = cluster_data.mean(axis=0)
    cluster_color = cluster_colors[cluster_id]
    ax.plot(cluster_data.columns, cluster_mean_dV, color=cluster_color, linewidth=2.5,
            label=f'Média Cluster {cluster_id + 1}')

    # ---------- Temperatura média (achatada, mas com escala real) ----------
    ax2 = ax.twinx()

    temp_min = df_temp['med_smooth'].min()
    temp_max = df_temp['med_smooth'].max()

    # Normalizar temperatura para o intervalo de dV e achatar visualmente
    temp_visual = (df_temp['med_smooth'] - temp_min) / (temp_max - temp_min)
    temp_visual = temp_visual * (dV_max - dV_min) * 0.25 + (dV_max - (dV_max - dV_min) * 0.3)

    ax2.plot(df_temp['data'], temp_visual, color='black', linewidth=2.2, alpha=0.85, label='Temperatura média (°C)')
    ax2.set_ylabel("Temperatura (°C)", color='black')
    ax2.tick_params(axis='y', labelcolor='black')

    # Definir ticks reais para o eixo da direita
    temp_ticks_real = np.linspace(temp_min, temp_max, 6)
    temp_ticks_visual = (temp_ticks_real - temp_min) / (temp_max - temp_min)
    temp_ticks_visual = temp_ticks_visual * (dV_max - dV_min) * 0.25 + (dV_max - (dV_max - dV_min) * 0.3)
    ax2.set_yticks(temp_ticks_visual)
    ax2.set_yticklabels([f"{t:.0f}" for t in temp_ticks_real])

    # ---------- Eixos e estilo ----------
    ax.set_ylim(dV_min - dV_margin, dV_max + dV_margin)
    ax2.set_ylim(dV_min - dV_margin, dV_max + dV_margin)

    ax.set_title(f'Cluster {cluster_id + 1} - {len(cluster_cells)} células', fontsize=12)
    #ax.set_xlabel('Ano')
    ax.set_ylabel('dV (mm)')
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

    ax.grid(False)
    ax2.grid(False)

    # ---------- Legenda combinada ----------
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, fontsize=9, loc='upper left')

plt.tight_layout()
plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
import numpy as np

# 1. Definições Globais de Estilo
LINE_WIDTH = 0.4
LINE_COLOR = '#333333' 
MY_CMAP = plt.cm.RdBu # RdBu padrão: Azul é Positivo (+1), Vermelho é Negativo (-1)

sns.set_context("paper", font_scale=0.9)
sns.set_style("white")
plt.rcParams['axes.linewidth'] = LINE_WIDTH

# --- Dados (Certifica-te que estas variáveis existem no teu ambiente) ---
CLUSTER_ALVO_ID = 0 
c_cells = cluster_df[cluster_df['cluster'] == CLUSTER_ALVO_ID]['cell_id']
dV_mean = agg_pivot.loc[c_cells].mean()
df_plot = pd.DataFrame({
    'dV': dV_mean,
    'Temp': df_temp.set_index('data')['med_smooth'].reindex(dV_mean.index),
    'Niv': df_nivel.set_index('data')['nivel_smooth'].reindex(dV_mean.index),
    'Prec': df_prec.set_index('data')['prec'].reindex(dV_mean.index),
    'P_Ac': df_prec.set_index('data')['prec_acum'].reindex(dV_mean.index),
    'P_An': df_prec.set_index('data')['prec_acum_anual'].reindex(dV_mean.index)
}).dropna()

cols = df_plot.columns.tolist()
n_vars = len(cols)

# 2. Criar o PairGrid
g = sns.PairGrid(df_plot, vars=cols, diag_sharey=False, height=1.0, aspect=1.0)

# --- Funções de Desenho ---
def cor_func(x, y, **kwargs):
    r, p = pearsonr(x, y)
    ax = plt.gca()
    # Mapeamento: r=1 -> Azul, r=-1 -> Vermelho
    color_val = (r + 1) / 2
    facecolor = MY_CMAP(color_val)
    ax.set_facecolor((*facecolor[:3], 0.5))
    ax.annotate(f"{r:.2f}", xy=(0.5, 0.5), xycoords=ax.transAxes, 
                ha='center', va='center', fontsize=8, fontweight='normal')

g.map_diag(sns.histplot, kde=True, color="#2c3e50", alpha=0.2, edgecolor='white', linewidth=0.3, line_kws={'linewidth': 0.7})
# --- PARTE INFERIOR: Scatter Plots ---
g.map_lower(sns.regplot, ci=None, 
            scatter_kws={
                's': 3,                # Tamanho do ponto
                'alpha': 1.0,          # Opacidade total (sem transparência)
                'color': 'black',      # Cor base
                'facecolor': 'black',  # Preenchimento preto sólido
                'edgecolor': 'black',  # Contorno preto sólido
                'linewidths': 0        # Remove qualquer largura de linha de bordo para evitar reflexos
            }, 
            line_kws={'color': 'red', 'linewidth': 0.7})
g.map_upper(cor_func)

# 3. Uniformização Total das Linhas e Padding
for i in range(n_vars):
    for j in range(n_vars):
        ax = g.axes[i, j]
        ax.set_xlabel(""); ax.set_ylabel("")
        
        # Ajuste de Padding interno (Respiro de 25%)
        if i != j:
            x_min, x_max = df_plot[cols[j]].min(), df_plot[cols[j]].max()
            y_min, y_max = df_plot[cols[i]].min(), df_plot[cols[i]].max()
            x_range, y_range = x_max - x_min, y_max - y_min
            ax.set_xlim(x_min - 0.3 * x_range, x_max + 0.3 * x_range)
            ax.set_ylim(y_min - 0.3 * y_range, y_max + 0.3 * y_range)

        # Labels apenas nas extremidades
        if j != 0: ax.set_yticklabels([])
        if i != n_vars - 1: ax.set_xticklabels([])
        
        ax.tick_params(labelsize=6, direction='in', pad=1, width=LINE_WIDTH, color=LINE_COLOR)
        
        # Forçar todas as linhas da grelha
        for edge in ['top', 'bottom', 'left', 'right']:
            ax.spines[edge].set_visible(True)
            ax.spines[edge].set_linewidth(LINE_WIDTH)
            ax.spines[edge].set_color(LINE_COLOR)
        
        # Identificação na Diagonal em Vermelho (Sem Bold)
        if i == j:
            ax.annotate(cols[i], xy=(0.05, 0.90), xycoords='axes fraction', 
                        ha='left', va='top', fontsize=8, fontweight='normal', color='red',
                        bbox=dict(facecolor='white', alpha=0.4, edgecolor='none', pad=0))

# 4. Ajuste de Layout para Colagem
plt.subplots_adjust(hspace=0, wspace=0, left=0.1, right=0.85, bottom=0.18, top=0.95)

# 5. Colorbar Corrigida (Azul=Positivo, Vermelho=Negativo)
cax = g.fig.add_axes([0.87, 0.18, 0.02, 0.77]) 
sm = plt.cm.ScalarMappable(cmap=MY_CMAP, norm=plt.Normalize(-1, 1))
cbar = g.fig.colorbar(sm, cax=cax)
cbar.set_ticks([-1, 0, 1])
cbar.outline.set_linewidth(LINE_WIDTH)
cbar.outline.set_edgecolor(LINE_COLOR)
cbar.set_label('Correlation coefficient', rotation=270, labelpad=15, fontsize=9)
cbar.ax.tick_params(labelsize=7, width=LINE_WIDTH, color=LINE_COLOR)

# 6. Legenda Inferior (Recuperada e Colada)
ax_table = g.fig.add_axes([0.1, 0.06, 0.75, 0.12]) 
ax_table.axis('off')

legend_data = [
    ["dV: Vertical Displacement", "Temp: Temperature", "Niv: Reservoir Level"],
    ["Prec: Daily Precipitation", "P_Ac: Accumulated Prec.", "P_An: Annual Acc. Prec."]
]

table = ax_table.table(cellText=legend_data, loc='upper center', cellLoc='left')
table.auto_set_font_size(False)
table.set_fontsize(8)
table.scale(1, 1.4) 

for (row, col), cell in table.get_celld().items():
    cell.set_linewidth(LINE_WIDTH)
    cell.set_edgecolor(LINE_COLOR)
    cell.get_text().set_fontweight('normal')

# 7. Finalização
plt.savefig("Matriz_Correlacao_Final_Final.png", dpi=600, bbox_inches='tight')
plt.show()

### K=4

In [ ]:
# ==============================
# 0. Bibliotecas
# ==============================
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
from scipy.signal import savgol_filter
from sklearn.cluster import KMeans

# ==============================
# 1. Ler CSVs ASC, DESC e ORTHO
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"
ortho_v_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"
ortho_h_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_E_2019_2023_1/EGMS_L3_E27N18_100km_E_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)
ortho_v = pd.read_csv(ortho_v_file)
ortho_h = pd.read_csv(ortho_h_file)

# ==============================
# 2. Filtrar área de interesse
# ==============================
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

def filter_area(df):
    return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
              (df['easting'] >= este_min) & (df['easting'] <= este_max)]

asc = filter_area(asc)
desc = filter_area(desc)
ortho_v = filter_area(ortho_v)
ortho_h = filter_area(ortho_h)

# ==============================
# 3. Função melt para ASC/DESC
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]  # ajustar conforme colunas de datas
    long_df = df.melt(
        id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'],
        value_vars=disp_cols, var_name='date', value_name='disp'
    )
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)

common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(), desc_long['date'].max()),
    freq='MS'
)

# ==============================
# 4. Interpolação temporal linear
# ==============================
def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(
            pd.to_datetime(common_dates).astype(np.int64),
            group['date'].astype(np.int64),
            group['disp']
        )
        dfs.append(pd.DataFrame({
            'easting': x,
            'northing': y,
            'latitude': group['latitude'].iloc[0],
            'longitude': group['longitude'].iloc[0],
            'date': common_dates,
            'disp': interp,
            'incidence_angle': group['incidence_angle'].iloc[0],
            'track_angle': group['track_angle'].iloc[0]
        }))
    return pd.concat(dfs, ignore_index=True)

asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 5. Interpolação espacial IDW
# ==============================
def idw_interpolation_per_date(source_df, target_df, radius=150, power=2):
    out_list = []
    for date, src_group in source_df.groupby('date'):
        trg_group = target_df[target_df['date']==date].copy()
        if src_group.empty or trg_group.empty:
            continue
        src_points = np.array(list(zip(src_group['easting'], src_group['northing'])))
        trg_points = np.array(list(zip(trg_group['easting'], trg_group['northing'])))
        tree = cKDTree(src_points)
        dists, idxs = tree.query(trg_points, k=5, distance_upper_bound=radius)

        interpolated_disp = []
        interpolated_theta = []
        interpolated_alpha = []
        for dist, idx in zip(dists, idxs):
            mask = np.isfinite(dist)
            if not np.any(mask):
                interpolated_disp.append(np.nan)
                interpolated_theta.append(np.nan)
                interpolated_alpha.append(np.nan)
                continue
            weights = 1 / (dist[mask] ** power)
            interpolated_disp.append(np.sum(weights * src_group.iloc[idx[mask]]['disp']) / np.sum(weights))
            interpolated_theta.append(np.sum(weights * src_group.iloc[idx[mask]]['incidence_angle']) / np.sum(weights))
            interpolated_alpha.append(np.sum(weights * src_group.iloc[idx[mask]]['track_angle']) / np.sum(weights))

        trg_group['disp_idw'] = interpolated_disp
        trg_group['theta_desc'] = interpolated_theta
        trg_group['alpha_desc'] = interpolated_alpha
        out_list.append(trg_group)
    return pd.concat(out_list, ignore_index=True)

asc_interp = idw_interpolation_per_date(desc_interp, asc_interp)

# ==============================
# 6. Calcular β e γ
# ==============================
orbit_inclination = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orbit_inclination) * np.cos(np.deg2rad(asc_interp['latitude'])))
asc_interp['gamma'] = 0

# ==============================
# 7. Calcular dV e dH
# ==============================
def compute_dV_dH_real(row):
    θA = np.deg2rad(row['incidence_angle'])
    θD = np.deg2rad(row['theta_desc'])
    αA = np.deg2rad(row['track_angle'])
    αD = np.deg2rad(row['alpha_desc'])
    β = row['beta']
    γ = row['gamma']

    dASC_LOS = row['disp']
    dDESC_LOS = row['disp_idw']

    denom = (np.cos(θA)*np.sin(θD)*np.cos(β + γ) +
             np.cos(θD)*np.sin(θA)*np.cos(β - γ))

    dV = (dDESC_LOS*np.sin(θA)*np.cos(β - γ) +
          dASC_LOS*np.sin(θD)*np.cos(β + γ)) / denom
    dH = (dDESC_LOS*np.cos(θA) - dASC_LOS*np.cos(θD)) / denom
    return pd.Series({'dV': dV, 'dH': dH})

asc_interp[['dV','dH']] = asc_interp.apply(compute_dV_dH_real, axis=1)

# ==============================
# 8. Criar grelha centrada ORTHO
# ==============================
grid_size = 100
x_edges = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

asc_interp['cell_x'] = pd.cut(asc_interp['easting'], bins=x_edges_shifted, labels=False)
asc_interp['cell_y'] = pd.cut(asc_interp['northing'], bins=y_edges_shifted, labels=False)

# Remover NaNs antes de criar cell_id
asc_interp = asc_interp.dropna(subset=['cell_x','cell_y'])
asc_interp['cell_id'] = asc_interp['cell_x'].astype(int).astype(str) + "_" + asc_interp['cell_y'].astype(int).astype(str)

# Agrupar para ponto central de cada célula
points = asc_interp.groupby('cell_id').agg({'easting':'mean','northing':'mean'}).reset_index()
gdf_points = gpd.GeoDataFrame(
    points,
    geometry=gpd.points_from_xy(points['easting'], points['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

agg = asc_interp.groupby(['cell_x','cell_y','date']).agg(
    x_center=('easting','mean'),
    y_center=('northing','mean'),
    dV=('dV','mean'),
    dH=('dH','mean')
).reset_index()
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + "_" + agg['cell_y'].astype(int).astype(str)

# ==============================
# 9. Criar GeoDataFrame da grelha
# ==============================
grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({
            "cell_x": ix,
            "cell_y": iy,
            "cell_id": f"{ix}_{iy}",
            "geometry": box(x_edges_shifted[ix], y_edges_shifted[iy],
                            x_edges_shifted[ix+1], y_edges_shifted[iy+1])
        })
grid = gpd.GeoDataFrame(grid_data, crs="EPSG:3035").to_crs(epsg=3857)

# ==============================
# 10. Carregar temperatura
# ==============================
df_temp = pd.read_excel("data/alqueva_temp.xlsx")
df_temp['data'] = pd.to_datetime(df_temp['data'])
window = 365
df_temp['med_smooth'] = savgol_filter(df_temp['med'], window_length=window, polyorder=2)

# ==============================
# 11. Clustering de dV
# ==============================
agg_pivot = agg.pivot(index='cell_id', columns='date', values='dV').fillna(0)
k = 4
kmeans = KMeans(n_clusters=k, random_state=0)
cluster_labels = kmeans.fit_predict(agg_pivot)
cluster_df = pd.DataFrame({'cell_id': agg_pivot.index, 'cluster': cluster_labels})

# Mapear cores
cluster_colors = {i: color for i, color in enumerate(['red','green','blue','orange','purple'])}
grid_sel = grid.merge(cluster_df, on='cell_id', how='left')

import matplotlib.dates as mdates

# ==============================
# 14. Figura única: mapa + clusters
# ==============================
clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)

# Criar figura com 2 linhas: mapa + séries temporais
fig = plt.figure(figsize=(20, 18))
gs = fig.add_gridspec(2, n_clusters, height_ratios=[2, 1.2])

# ------------------------------
# Linha 1: Mapa com legenda técnica
# ------------------------------
ax_map = fig.add_subplot(gs[0, :])
grid.boundary.plot(ax=ax_map, color='lightgray', linewidth=0.5)
grid_sel.boundary.plot(ax=ax_map, color='black', linewidth=1, alpha=0.2)

# --- Células coloridas por cluster
for i, row in grid_sel.iterrows():
    if pd.notna(row['cluster']):
        gpd.GeoSeries([row['geometry']], crs=grid_sel.crs).plot(
            ax=ax_map,
            color=cluster_colors[int(row['cluster'])],
            alpha=0.4
        )

# --- Pontos centrais (ASC/DESC)
gdf_points.plot(ax=ax_map, color='white', edgecolor='black', markersize=40)

# Adicionar item de legenda para os pontos
ax_map.scatter([], [], marker='o', color='white', edgecolor='black', s=120,
               label='Pontos ASC/DESC')

# --- Adicionar basemap
ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)

# --- Legenda dos clusters
for cluster_id in clusters_present:
    color = cluster_colors[cluster_id]
    n_cells = len(cluster_df[cluster_df["cluster"] == cluster_id])
    ax_map.scatter([], [], color=color, alpha=0.6,
                   label=f'Cluster {cluster_id + 1} - {n_cells} células')

# # --- Caixa técnica (informações do processamento)
# textstr = '\n'.join((
#     f'Tamanho da grelha: {grid_size} x {grid_size} m',
#     f'Técnica de clustering: K-Means (k = {n_clusters})',
#     'Tipo de deslocamento: dV (vertical)',
#     'Base de dados: EGMS 2019–2023',
# ))
# ax_map.text(0.99, 0.01, textstr, transform=ax_map.transAxes,
#             fontsize=10, verticalalignment='bottom', horizontalalignment='right',
#             bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.7))

# --- Título e formatação
ax_map.set_title(
    f"Clusters de séries temporais de deslocamento vertical.\n"
    f"K-Means, K={k}.\n"
    f"Grelha {grid_size} m × {grid_size} m.\n"
    f"Correlação entre a série temporal média de cada cluster e a temperatura.",
    fontsize=16
)
ax_map.set_axis_off()
ax_map.legend(fontsize=10, loc='upper left')

# ------------------------------
# Linha 2: Séries temporais
# ------------------------------
# Escala comum para dV (mantida igual em todos os clusters)
dV_min = agg['dV'].min()
dV_max = agg['dV'].max()
dV_margin = (dV_max - dV_min) * 0.1  # margem de 10%

for idx, cluster_id in enumerate(clusters_present):
    ax = fig.add_subplot(gs[1, idx])
    
    # Selecionar células do cluster
    cluster_cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]

    # Plot das células individuais (cinzento)
    for cid in cluster_cells:
        ax.plot(cluster_data.columns, cluster_data.loc[cid], color='lightgray', alpha=0.7)

    # Média do cluster (linha colorida)
    cluster_mean_dV = cluster_data.mean(axis=0)
    cluster_color = cluster_colors[cluster_id]
    ax.plot(cluster_data.columns, cluster_mean_dV, color=cluster_color, linewidth=2.5,
            label=f'Média Cluster {cluster_id + 1}')

    # ---------- Temperatura média (achatada, mas com escala real) ----------
    ax2 = ax.twinx()

    temp_min = df_temp['med_smooth'].min()
    temp_max = df_temp['med_smooth'].max()

    # Normalizar temperatura para o intervalo de dV e achatar visualmente
    temp_visual = (df_temp['med_smooth'] - temp_min) / (temp_max - temp_min)
    temp_visual = temp_visual * (dV_max - dV_min) * 0.25 + (dV_max - (dV_max - dV_min) * 0.3)

    ax2.plot(df_temp['data'], temp_visual, color='black', linewidth=2.2, alpha=0.85, label='Temperatura média (°C)')
    ax2.set_ylabel("Temperatura (°C)", color='black')
    ax2.tick_params(axis='y', labelcolor='black')

    # Definir ticks reais para o eixo da direita
    temp_ticks_real = np.linspace(temp_min, temp_max, 6)
    temp_ticks_visual = (temp_ticks_real - temp_min) / (temp_max - temp_min)
    temp_ticks_visual = temp_ticks_visual * (dV_max - dV_min) * 0.25 + (dV_max - (dV_max - dV_min) * 0.3)
    ax2.set_yticks(temp_ticks_visual)
    ax2.set_yticklabels([f"{t:.0f}" for t in temp_ticks_real])

    # ---------- Eixos e estilo ----------
    ax.set_ylim(dV_min - dV_margin, dV_max + dV_margin)
    ax2.set_ylim(dV_min - dV_margin, dV_max + dV_margin)

    ax.set_title(f'Cluster {cluster_id + 1} - {len(cluster_cells)} células', fontsize=12)
    #ax.set_xlabel('Ano')
    ax.set_ylabel('dV (mm)')
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

    ax.grid(False)
    ax2.grid(False)

    # ---------- Legenda combinada ----------
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, fontsize=9, loc='upper left')

plt.tight_layout()
plt.show()

### K=5

In [ ]:
# ==============================
# 0. Bibliotecas
# ==============================
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
from scipy.signal import savgol_filter
from sklearn.cluster import KMeans

# ==============================
# 1. Ler CSVs ASC, DESC e ORTHO
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"
ortho_v_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"
ortho_h_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_E_2019_2023_1/EGMS_L3_E27N18_100km_E_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)
ortho_v = pd.read_csv(ortho_v_file)
ortho_h = pd.read_csv(ortho_h_file)

# ==============================
# 2. Filtrar área de interesse
# ==============================
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

def filter_area(df):
    return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
              (df['easting'] >= este_min) & (df['easting'] <= este_max)]

asc = filter_area(asc)
desc = filter_area(desc)
ortho_v = filter_area(ortho_v)
ortho_h = filter_area(ortho_h)

# ==============================
# 3. Função melt para ASC/DESC
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]  # ajustar conforme colunas de datas
    long_df = df.melt(
        id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'],
        value_vars=disp_cols, var_name='date', value_name='disp'
    )
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)

common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(), desc_long['date'].max()),
    freq='MS'
)

# ==============================
# 4. Interpolação temporal linear
# ==============================
def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(
            pd.to_datetime(common_dates).astype(np.int64),
            group['date'].astype(np.int64),
            group['disp']
        )
        dfs.append(pd.DataFrame({
            'easting': x,
            'northing': y,
            'latitude': group['latitude'].iloc[0],
            'longitude': group['longitude'].iloc[0],
            'date': common_dates,
            'disp': interp,
            'incidence_angle': group['incidence_angle'].iloc[0],
            'track_angle': group['track_angle'].iloc[0]
        }))
    return pd.concat(dfs, ignore_index=True)

asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 5. Interpolação espacial IDW
# ==============================
def idw_interpolation_per_date(source_df, target_df, radius=150, power=2):
    out_list = []
    for date, src_group in source_df.groupby('date'):
        trg_group = target_df[target_df['date']==date].copy()
        if src_group.empty or trg_group.empty:
            continue
        src_points = np.array(list(zip(src_group['easting'], src_group['northing'])))
        trg_points = np.array(list(zip(trg_group['easting'], trg_group['northing'])))
        tree = cKDTree(src_points)
        dists, idxs = tree.query(trg_points, k=5, distance_upper_bound=radius)

        interpolated_disp = []
        interpolated_theta = []
        interpolated_alpha = []
        for dist, idx in zip(dists, idxs):
            mask = np.isfinite(dist)
            if not np.any(mask):
                interpolated_disp.append(np.nan)
                interpolated_theta.append(np.nan)
                interpolated_alpha.append(np.nan)
                continue
            weights = 1 / (dist[mask] ** power)
            interpolated_disp.append(np.sum(weights * src_group.iloc[idx[mask]]['disp']) / np.sum(weights))
            interpolated_theta.append(np.sum(weights * src_group.iloc[idx[mask]]['incidence_angle']) / np.sum(weights))
            interpolated_alpha.append(np.sum(weights * src_group.iloc[idx[mask]]['track_angle']) / np.sum(weights))

        trg_group['disp_idw'] = interpolated_disp
        trg_group['theta_desc'] = interpolated_theta
        trg_group['alpha_desc'] = interpolated_alpha
        out_list.append(trg_group)
    return pd.concat(out_list, ignore_index=True)

asc_interp = idw_interpolation_per_date(desc_interp, asc_interp)

# ==============================
# 6. Calcular β e γ
# ==============================
orbit_inclination = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orbit_inclination) * np.cos(np.deg2rad(asc_interp['latitude'])))
asc_interp['gamma'] = 0

# ==============================
# 7. Calcular dV e dH
# ==============================
def compute_dV_dH_real(row):
    θA = np.deg2rad(row['incidence_angle'])
    θD = np.deg2rad(row['theta_desc'])
    αA = np.deg2rad(row['track_angle'])
    αD = np.deg2rad(row['alpha_desc'])
    β = row['beta']
    γ = row['gamma']

    dASC_LOS = row['disp']
    dDESC_LOS = row['disp_idw']

    denom = (np.cos(θA)*np.sin(θD)*np.cos(β + γ) +
             np.cos(θD)*np.sin(θA)*np.cos(β - γ))

    dV = (dDESC_LOS*np.sin(θA)*np.cos(β - γ) +
          dASC_LOS*np.sin(θD)*np.cos(β + γ)) / denom
    dH = (dDESC_LOS*np.cos(θA) - dASC_LOS*np.cos(θD)) / denom
    return pd.Series({'dV': dV, 'dH': dH})

asc_interp[['dV','dH']] = asc_interp.apply(compute_dV_dH_real, axis=1)

# ==============================
# 8. Criar grelha centrada ORTHO
# ==============================
grid_size = 100
x_edges = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

asc_interp['cell_x'] = pd.cut(asc_interp['easting'], bins=x_edges_shifted, labels=False)
asc_interp['cell_y'] = pd.cut(asc_interp['northing'], bins=y_edges_shifted, labels=False)

# Remover NaNs antes de criar cell_id
asc_interp = asc_interp.dropna(subset=['cell_x','cell_y'])
asc_interp['cell_id'] = asc_interp['cell_x'].astype(int).astype(str) + "_" + asc_interp['cell_y'].astype(int).astype(str)

# Agrupar para ponto central de cada célula
points = asc_interp.groupby('cell_id').agg({'easting':'mean','northing':'mean'}).reset_index()
gdf_points = gpd.GeoDataFrame(
    points,
    geometry=gpd.points_from_xy(points['easting'], points['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

agg = asc_interp.groupby(['cell_x','cell_y','date']).agg(
    x_center=('easting','mean'),
    y_center=('northing','mean'),
    dV=('dV','mean'),
    dH=('dH','mean')
).reset_index()
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + "_" + agg['cell_y'].astype(int).astype(str)

# ==============================
# 9. Criar GeoDataFrame da grelha
# ==============================
grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({
            "cell_x": ix,
            "cell_y": iy,
            "cell_id": f"{ix}_{iy}",
            "geometry": box(x_edges_shifted[ix], y_edges_shifted[iy],
                            x_edges_shifted[ix+1], y_edges_shifted[iy+1])
        })
grid = gpd.GeoDataFrame(grid_data, crs="EPSG:3035").to_crs(epsg=3857)

# ==============================
# 10. Carregar temperatura
# ==============================
df_temp = pd.read_excel("data/alqueva_temp.xlsx")
df_temp['data'] = pd.to_datetime(df_temp['data'])
window = 365
df_temp['med_smooth'] = savgol_filter(df_temp['med'], window_length=window, polyorder=2)

# ==============================
# 11. Clustering de dV
# ==============================
agg_pivot = agg.pivot(index='cell_id', columns='date', values='dV').fillna(0)
k = 5
kmeans = KMeans(n_clusters=k, random_state=0)
cluster_labels = kmeans.fit_predict(agg_pivot)
cluster_df = pd.DataFrame({'cell_id': agg_pivot.index, 'cluster': cluster_labels})

# Mapear cores
cluster_colors = {i: color for i, color in enumerate(['red','green','blue','orange','purple'])}
grid_sel = grid.merge(cluster_df, on='cell_id', how='left')

import matplotlib.dates as mdates

# ==============================
# 14. Figura única: mapa + clusters
# ==============================
clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)

# Criar figura com 2 linhas: mapa + séries temporais
fig = plt.figure(figsize=(20, 18))
gs = fig.add_gridspec(2, n_clusters, height_ratios=[2, 1.2])

# ------------------------------
# Linha 1: Mapa com legenda técnica
# ------------------------------
ax_map = fig.add_subplot(gs[0, :])
grid.boundary.plot(ax=ax_map, color='lightgray', linewidth=0.5)
grid_sel.boundary.plot(ax=ax_map, color='black', linewidth=1, alpha=0.2)

# --- Células coloridas por cluster
for i, row in grid_sel.iterrows():
    if pd.notna(row['cluster']):
        gpd.GeoSeries([row['geometry']], crs=grid_sel.crs).plot(
            ax=ax_map,
            color=cluster_colors[int(row['cluster'])],
            alpha=0.4
        )

# --- Pontos centrais (ASC/DESC)
gdf_points.plot(ax=ax_map, color='white', edgecolor='black', markersize=40)

# Adicionar item de legenda para os pontos
ax_map.scatter([], [], marker='o', color='white', edgecolor='black', s=120,
               label='Pontos ASC/DESC')

# --- Adicionar basemap
ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)

# --- Legenda dos clusters
for cluster_id in clusters_present:
    color = cluster_colors[cluster_id]
    n_cells = len(cluster_df[cluster_df["cluster"] == cluster_id])
    ax_map.scatter([], [], color=color, alpha=0.6,
                   label=f'Cluster {cluster_id + 1} - {n_cells} células')

# # --- Caixa técnica (informações do processamento)
# textstr = '\n'.join((
#     f'Tamanho da grelha: {grid_size} x {grid_size} m',
#     f'Técnica de clustering: K-Means (k = {n_clusters})',
#     'Tipo de deslocamento: dV (vertical)',
#     'Base de dados: EGMS 2019–2023',
# ))
# ax_map.text(0.99, 0.01, textstr, transform=ax_map.transAxes,
#             fontsize=10, verticalalignment='bottom', horizontalalignment='right',
#             bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.7))

# --- Título e formatação
ax_map.set_title(
    f"Clusters de séries temporais de deslocamento vertical.\n"
    f"K-Means, K={k}.\n"
    f"Grelha {grid_size} m × {grid_size} m.\n"
    f"Correlação entre a série temporal média de cada cluster e a temperatura.",
    fontsize=16
)
ax_map.set_axis_off()
ax_map.legend(fontsize=10, loc='upper left')

# ------------------------------
# Linha 2: Séries temporais
# ------------------------------
# Escala comum para dV (mantida igual em todos os clusters)
dV_min = agg['dV'].min()
dV_max = agg['dV'].max()
dV_margin = (dV_max - dV_min) * 0.1  # margem de 10%

for idx, cluster_id in enumerate(clusters_present):
    ax = fig.add_subplot(gs[1, idx])
    
    # Selecionar células do cluster
    cluster_cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]

    # Plot das células individuais (cinzento)
    for cid in cluster_cells:
        ax.plot(cluster_data.columns, cluster_data.loc[cid], color='lightgray', alpha=0.7)

    # Média do cluster (linha colorida)
    cluster_mean_dV = cluster_data.mean(axis=0)
    cluster_color = cluster_colors[cluster_id]
    ax.plot(cluster_data.columns, cluster_mean_dV, color=cluster_color, linewidth=2.5,
            label=f'Média Cluster {cluster_id + 1}')

    # ---------- Temperatura média (achatada, mas com escala real) ----------
    ax2 = ax.twinx()

    temp_min = df_temp['med_smooth'].min()
    temp_max = df_temp['med_smooth'].max()

    # Normalizar temperatura para o intervalo de dV e achatar visualmente
    temp_visual = (df_temp['med_smooth'] - temp_min) / (temp_max - temp_min)
    temp_visual = temp_visual * (dV_max - dV_min) * 0.25 + (dV_max - (dV_max - dV_min) * 0.3)

    ax2.plot(df_temp['data'], temp_visual, color='black', linewidth=2.2, alpha=0.85, label='Temperatura média (°C)')
    ax2.set_ylabel("Temperatura (°C)", color='black')
    ax2.tick_params(axis='y', labelcolor='black')

    # Definir ticks reais para o eixo da direita
    temp_ticks_real = np.linspace(temp_min, temp_max, 6)
    temp_ticks_visual = (temp_ticks_real - temp_min) / (temp_max - temp_min)
    temp_ticks_visual = temp_ticks_visual * (dV_max - dV_min) * 0.25 + (dV_max - (dV_max - dV_min) * 0.3)
    ax2.set_yticks(temp_ticks_visual)
    ax2.set_yticklabels([f"{t:.0f}" for t in temp_ticks_real])

    # ---------- Eixos e estilo ----------
    ax.set_ylim(dV_min - dV_margin, dV_max + dV_margin)
    ax2.set_ylim(dV_min - dV_margin, dV_max + dV_margin)

    ax.set_title(f'Cluster {cluster_id + 1} - {len(cluster_cells)} células', fontsize=12)
    #ax.set_xlabel('Ano')
    ax.set_ylabel('dV (mm)')
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

    ax.grid(False)
    ax2.grid(False)

    # ---------- Legenda combinada ----------
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, fontsize=9, loc='upper left')

plt.tight_layout()
plt.show()

## Nível da albufeira

### K=2

In [ ]:
# ==============================
# 0. Bibliotecas
# ==============================
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
from scipy.signal import savgol_filter
from sklearn.cluster import KMeans
import matplotlib.dates as mdates

# ==============================
# 1. Ler CSVs ASC, DESC e ORTHO
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"
ortho_v_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"
ortho_h_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_E_2019_2023_1/EGMS_L3_E27N18_100km_E_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)
ortho_v = pd.read_csv(ortho_v_file)
ortho_h = pd.read_csv(ortho_h_file)

# ==============================
# 2. Filtrar área de interesse
# ==============================
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

def filter_area(df):
    return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
              (df['easting'] >= este_min) & (df['easting'] <= este_max)]

asc = filter_area(asc)
desc = filter_area(desc)
ortho_v = filter_area(ortho_v)
ortho_h = filter_area(ortho_h)

# ==============================
# 3. Função melt para ASC/DESC
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]  # ajustar conforme colunas de datas
    long_df = df.melt(
        id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'],
        value_vars=disp_cols, var_name='date', value_name='disp'
    )
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)

common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(), desc_long['date'].max()),
    freq='MS'
)

# ==============================
# 4. Interpolação temporal linear
# ==============================
def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(
            pd.to_datetime(common_dates).astype(np.int64),
            group['date'].astype(np.int64),
            group['disp']
        )
        dfs.append(pd.DataFrame({
            'easting': x,
            'northing': y,
            'latitude': group['latitude'].iloc[0],
            'longitude': group['longitude'].iloc[0],
            'date': common_dates,
            'disp': interp,
            'incidence_angle': group['incidence_angle'].iloc[0],
            'track_angle': group['track_angle'].iloc[0]
        }))
    return pd.concat(dfs, ignore_index=True)

asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 5. Interpolação espacial IDW
# ==============================
def idw_interpolation_per_date(source_df, target_df, radius=150, power=2):
    out_list = []
    for date, src_group in source_df.groupby('date'):
        trg_group = target_df[target_df['date']==date].copy()
        if src_group.empty or trg_group.empty:
            continue
        src_points = np.array(list(zip(src_group['easting'], src_group['northing'])))
        trg_points = np.array(list(zip(trg_group['easting'], trg_group['northing'])))
        tree = cKDTree(src_points)
        dists, idxs = tree.query(trg_points, k=5, distance_upper_bound=radius)

        interpolated_disp = []
        interpolated_theta = []
        interpolated_alpha = []
        for dist, idx in zip(dists, idxs):
            mask = np.isfinite(dist)
            if not np.any(mask):
                interpolated_disp.append(np.nan)
                interpolated_theta.append(np.nan)
                interpolated_alpha.append(np.nan)
                continue
            weights = 1 / (dist[mask] ** power)
            interpolated_disp.append(np.sum(weights * src_group.iloc[idx[mask]]['disp']) / np.sum(weights))
            interpolated_theta.append(np.sum(weights * src_group.iloc[idx[mask]]['incidence_angle']) / np.sum(weights))
            interpolated_alpha.append(np.sum(weights * src_group.iloc[idx[mask]]['track_angle']) / np.sum(weights))

        trg_group['disp_idw'] = interpolated_disp
        trg_group['theta_desc'] = interpolated_theta
        trg_group['alpha_desc'] = interpolated_alpha
        out_list.append(trg_group)
    return pd.concat(out_list, ignore_index=True)

asc_interp = idw_interpolation_per_date(desc_interp, asc_interp)

# ==============================
# 6. Calcular β e γ
# ==============================
orbit_inclination = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orbit_inclination) * np.cos(np.deg2rad(asc_interp['latitude'])))
asc_interp['gamma'] = 0

# ==============================
# 7. Calcular dV e dH
# ==============================
def compute_dV_dH_real(row):
    θA = np.deg2rad(row['incidence_angle'])
    θD = np.deg2rad(row['theta_desc'])
    αA = np.deg2rad(row['track_angle'])
    αD = np.deg2rad(row['alpha_desc'])
    β = row['beta']
    γ = row['gamma']

    dASC_LOS = row['disp']
    dDESC_LOS = row['disp_idw']

    denom = (np.cos(θA)*np.sin(θD)*np.cos(β + γ) +
             np.cos(θD)*np.sin(θA)*np.cos(β - γ))

    dV = (dDESC_LOS*np.sin(θA)*np.cos(β - γ) +
          dASC_LOS*np.sin(θD)*np.cos(β + γ)) / denom
    dH = (dDESC_LOS*np.cos(θA) - dASC_LOS*np.cos(θD)) / denom
    return pd.Series({'dV': dV, 'dH': dH})

asc_interp[['dV','dH']] = asc_interp.apply(compute_dV_dH_real, axis=1)

# ==============================
# 8. Criar grelha centrada ORTHO
# ==============================
grid_size = 100
x_edges = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

asc_interp['cell_x'] = pd.cut(asc_interp['easting'], bins=x_edges_shifted, labels=False)
asc_interp['cell_y'] = pd.cut(asc_interp['northing'], bins=y_edges_shifted, labels=False)
asc_interp = asc_interp.dropna(subset=['cell_x','cell_y'])
asc_interp['cell_id'] = asc_interp['cell_x'].astype(int).astype(str) + "_" + asc_interp['cell_y'].astype(int).astype(str)

points = asc_interp.groupby('cell_id').agg({'easting':'mean','northing':'mean'}).reset_index()
gdf_points = gpd.GeoDataFrame(
    points,
    geometry=gpd.points_from_xy(points['easting'], points['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

agg = asc_interp.groupby(['cell_x','cell_y','date']).agg(
    x_center=('easting','mean'),
    y_center=('northing','mean'),
    dV=('dV','mean'),
    dH=('dH','mean')
).reset_index()
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + "_" + agg['cell_y'].astype(int).astype(str)

grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({
            "cell_x": ix,
            "cell_y": iy,
            "cell_id": f"{ix}_{iy}",
            "geometry": box(x_edges_shifted[ix], y_edges_shifted[iy],
                            x_edges_shifted[ix+1], y_edges_shifted[iy+1])
        })
grid = gpd.GeoDataFrame(grid_data, crs="EPSG:3035").to_crs(epsg=3857)

# ==============================
# 9. Carregar temperatura e nível
# ==============================
df_temp = pd.read_excel("data/alqueva_temp.xlsx")
df_temp['data'] = pd.to_datetime(df_temp['data'])
window = 365
df_temp['med_smooth'] = savgol_filter(df_temp['med'], window_length=window, polyorder=2)

df_nivel = pd.read_excel("data/alqueva_nivel.xlsx")
df_nivel['data'] = pd.to_datetime(df_nivel['data'])
df_nivel['nivel_smooth'] = savgol_filter(df_nivel['nivel'], window_length=window, polyorder=2)

# ==============================
# 10. Clustering de dV
# ==============================
agg_pivot = agg.pivot(index='cell_id', columns='date', values='dV').fillna(0)
k = 2
kmeans = KMeans(n_clusters=k, random_state=0)
cluster_labels = kmeans.fit_predict(agg_pivot)
cluster_df = pd.DataFrame({'cell_id': agg_pivot.index, 'cluster': cluster_labels})
cluster_colors = {i: color for i, color in enumerate(['red','green','blue','orange','purple'])}
grid_sel = grid.merge(cluster_df, on='cell_id', how='left')

# ==============================
# 11. Figura única: mapa + clusters + nível
# ==============================
clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)

fig = plt.figure(figsize=(20, 18))
gs = fig.add_gridspec(2, n_clusters, height_ratios=[2, 1.2])

# ------------------------------
# Linha 1: Mapa com legenda técnica
# ------------------------------
ax_map = fig.add_subplot(gs[0, :])
grid.boundary.plot(ax=ax_map, color='lightgray', linewidth=0.5)
grid_sel.boundary.plot(ax=ax_map, color='black', linewidth=1, alpha=0.2)

# --- Células coloridas por cluster
for i, row in grid_sel.iterrows():
    if pd.notna(row['cluster']):
        gpd.GeoSeries([row['geometry']], crs=grid_sel.crs).plot(
            ax=ax_map,
            color=cluster_colors[int(row['cluster'])],
            alpha=0.4
        )

# --- Pontos centrais (ASC/DESC)
gdf_points.plot(ax=ax_map, color='white', edgecolor='black', markersize=40)

# Adicionar item de legenda para os pontos
ax_map.scatter([], [], marker='o', color='white', edgecolor='black', s=120,
               label='Pontos ASC/DESC')

# --- Adicionar basemap
ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)

# --- Legenda dos clusters
for cluster_id in clusters_present:
    color = cluster_colors[cluster_id]
    n_cells = len(cluster_df[cluster_df["cluster"] == cluster_id])
    ax_map.scatter([], [], color=color, alpha=0.6,
                   label=f'Cluster {cluster_id + 1} - {n_cells} células')

# # --- Caixa técnica (informações do processamento)
# textstr = '\n'.join((
#     f'Tamanho da grelha: {grid_size} x {grid_size} m',
#     f'Técnica de clustering: K-Means (k = {n_clusters})',
#     'Tipo de deslocamento: dV (vertical)',
#     'Base de dados: EGMS 2019–2023',
# ))
# ax_map.text(0.99, 0.01, textstr, transform=ax_map.transAxes,
#             fontsize=10, verticalalignment='bottom', horizontalalignment='right',
#             bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.7))

# --- Título e formatação
ax_map.set_title(
    f"Clusters de séries temporais de deslocamento vertical.\n"
    f"K-Means, K={k}.\n"
    f"Grelha {grid_size} m × {grid_size} m.\n"
    f"Correlação entre a série temporal média de cada cluster e o nível da albufeira.",
    fontsize=16
)
ax_map.set_axis_off()
ax_map.legend(fontsize=10, loc='upper left')

# --- Linha 2: Séries temporais com nível ---
dV_min = agg['dV'].min()
dV_max = agg['dV'].max()
dV_margin = (dV_max - dV_min) * 0.1

nivel_min = df_nivel['nivel_smooth'].min()
nivel_max = df_nivel['nivel_smooth'].max()

for idx, cluster_id in enumerate(clusters_present):
    ax = fig.add_subplot(gs[1, idx])
    cluster_cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]

    for cid in cluster_cells:
        ax.plot(cluster_data.columns, cluster_data.loc[cid], color='lightgray', alpha=0.7)

    cluster_mean_dV = cluster_data.mean(axis=0)
    cluster_color = cluster_colors[cluster_id]
    ax.plot(cluster_data.columns, cluster_mean_dV, color=cluster_color, linewidth=2.5,
            label=f'Média Cluster {cluster_id + 1}')

    ax2 = ax.twinx()
    nivel_visual = (df_nivel['nivel_smooth'] - nivel_min) / (nivel_max - nivel_min)
    nivel_visual = nivel_visual * (dV_max - dV_min) * 0.25 + (dV_max - (dV_max - dV_min) * 0.3)
    ax2.plot(df_nivel['data'], nivel_visual, color='black', linewidth=2.2, alpha=0.85, label='Nível da albufeira (m)')
    ax2.set_ylabel("Nível da albufeira (m)", color='black')
    ax2.tick_params(axis='y', labelcolor='black')

    nivel_ticks_real = np.linspace(nivel_min, nivel_max, 6)
    nivel_ticks_visual = (nivel_ticks_real - nivel_min) / (nivel_max - nivel_min)
    nivel_ticks_visual = nivel_ticks_visual * (dV_max - dV_min) * 0.25 + (dV_max - (dV_max - dV_min) * 0.3)
    ax2.set_yticks(nivel_ticks_visual)
    ax2.set_yticklabels([f"{t:.1f}" for t in nivel_ticks_real])

    ax.set_ylim(dV_min - dV_margin, dV_max + dV_margin)
    ax2.set_ylim(dV_min - dV_margin, dV_max + dV_margin)

    ax.set_title(f'Cluster {cluster_id + 1} - {len(cluster_cells)} células', fontsize=12)
    ax.set_ylabel('dV (mm)')
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

    ax.grid(False)
    ax2.grid(False)

    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, fontsize=9, loc='upper left')

plt.tight_layout()
plt.show()

### K=3

In [ ]:
# ==============================
# 0. Bibliotecas
# ==============================
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
from scipy.signal import savgol_filter
from sklearn.cluster import KMeans
import matplotlib.dates as mdates

# ==============================
# 1. Ler CSVs ASC, DESC e ORTHO
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"
ortho_v_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"
ortho_h_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_E_2019_2023_1/EGMS_L3_E27N18_100km_E_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)
ortho_v = pd.read_csv(ortho_v_file)
ortho_h = pd.read_csv(ortho_h_file)

# ==============================
# 2. Filtrar área de interesse
# ==============================
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

def filter_area(df):
    return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
              (df['easting'] >= este_min) & (df['easting'] <= este_max)]

asc = filter_area(asc)
desc = filter_area(desc)
ortho_v = filter_area(ortho_v)
ortho_h = filter_area(ortho_h)

# ==============================
# 3. Função melt para ASC/DESC
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]  # ajustar conforme colunas de datas
    long_df = df.melt(
        id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'],
        value_vars=disp_cols, var_name='date', value_name='disp'
    )
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)

common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(), desc_long['date'].max()),
    freq='MS'
)

# ==============================
# 4. Interpolação temporal linear
# ==============================
def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(
            pd.to_datetime(common_dates).astype(np.int64),
            group['date'].astype(np.int64),
            group['disp']
        )
        dfs.append(pd.DataFrame({
            'easting': x,
            'northing': y,
            'latitude': group['latitude'].iloc[0],
            'longitude': group['longitude'].iloc[0],
            'date': common_dates,
            'disp': interp,
            'incidence_angle': group['incidence_angle'].iloc[0],
            'track_angle': group['track_angle'].iloc[0]
        }))
    return pd.concat(dfs, ignore_index=True)

asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 5. Interpolação espacial IDW
# ==============================
def idw_interpolation_per_date(source_df, target_df, radius=150, power=2):
    out_list = []
    for date, src_group in source_df.groupby('date'):
        trg_group = target_df[target_df['date']==date].copy()
        if src_group.empty or trg_group.empty:
            continue
        src_points = np.array(list(zip(src_group['easting'], src_group['northing'])))
        trg_points = np.array(list(zip(trg_group['easting'], trg_group['northing'])))
        tree = cKDTree(src_points)
        dists, idxs = tree.query(trg_points, k=5, distance_upper_bound=radius)

        interpolated_disp = []
        interpolated_theta = []
        interpolated_alpha = []
        for dist, idx in zip(dists, idxs):
            mask = np.isfinite(dist)
            if not np.any(mask):
                interpolated_disp.append(np.nan)
                interpolated_theta.append(np.nan)
                interpolated_alpha.append(np.nan)
                continue
            weights = 1 / (dist[mask] ** power)
            interpolated_disp.append(np.sum(weights * src_group.iloc[idx[mask]]['disp']) / np.sum(weights))
            interpolated_theta.append(np.sum(weights * src_group.iloc[idx[mask]]['incidence_angle']) / np.sum(weights))
            interpolated_alpha.append(np.sum(weights * src_group.iloc[idx[mask]]['track_angle']) / np.sum(weights))

        trg_group['disp_idw'] = interpolated_disp
        trg_group['theta_desc'] = interpolated_theta
        trg_group['alpha_desc'] = interpolated_alpha
        out_list.append(trg_group)
    return pd.concat(out_list, ignore_index=True)

asc_interp = idw_interpolation_per_date(desc_interp, asc_interp)

# ==============================
# 6. Calcular β e γ
# ==============================
orbit_inclination = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orbit_inclination) * np.cos(np.deg2rad(asc_interp['latitude'])))
asc_interp['gamma'] = 0

# ==============================
# 7. Calcular dV e dH
# ==============================
def compute_dV_dH_real(row):
    θA = np.deg2rad(row['incidence_angle'])
    θD = np.deg2rad(row['theta_desc'])
    αA = np.deg2rad(row['track_angle'])
    αD = np.deg2rad(row['alpha_desc'])
    β = row['beta']
    γ = row['gamma']

    dASC_LOS = row['disp']
    dDESC_LOS = row['disp_idw']

    denom = (np.cos(θA)*np.sin(θD)*np.cos(β + γ) +
             np.cos(θD)*np.sin(θA)*np.cos(β - γ))

    dV = (dDESC_LOS*np.sin(θA)*np.cos(β - γ) +
          dASC_LOS*np.sin(θD)*np.cos(β + γ)) / denom
    dH = (dDESC_LOS*np.cos(θA) - dASC_LOS*np.cos(θD)) / denom
    return pd.Series({'dV': dV, 'dH': dH})

asc_interp[['dV','dH']] = asc_interp.apply(compute_dV_dH_real, axis=1)

# ==============================
# 8. Criar grelha centrada ORTHO
# ==============================
grid_size = 100
x_edges = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

asc_interp['cell_x'] = pd.cut(asc_interp['easting'], bins=x_edges_shifted, labels=False)
asc_interp['cell_y'] = pd.cut(asc_interp['northing'], bins=y_edges_shifted, labels=False)
asc_interp = asc_interp.dropna(subset=['cell_x','cell_y'])
asc_interp['cell_id'] = asc_interp['cell_x'].astype(int).astype(str) + "_" + asc_interp['cell_y'].astype(int).astype(str)

points = asc_interp.groupby('cell_id').agg({'easting':'mean','northing':'mean'}).reset_index()
gdf_points = gpd.GeoDataFrame(
    points,
    geometry=gpd.points_from_xy(points['easting'], points['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

agg = asc_interp.groupby(['cell_x','cell_y','date']).agg(
    x_center=('easting','mean'),
    y_center=('northing','mean'),
    dV=('dV','mean'),
    dH=('dH','mean')
).reset_index()
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + "_" + agg['cell_y'].astype(int).astype(str)

grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({
            "cell_x": ix,
            "cell_y": iy,
            "cell_id": f"{ix}_{iy}",
            "geometry": box(x_edges_shifted[ix], y_edges_shifted[iy],
                            x_edges_shifted[ix+1], y_edges_shifted[iy+1])
        })
grid = gpd.GeoDataFrame(grid_data, crs="EPSG:3035").to_crs(epsg=3857)

# ==============================
# 9. Carregar temperatura e nível
# ==============================
df_temp = pd.read_excel("data/alqueva_temp.xlsx")
df_temp['data'] = pd.to_datetime(df_temp['data'])
window = 365
df_temp['med_smooth'] = savgol_filter(df_temp['med'], window_length=window, polyorder=2)

df_nivel = pd.read_excel("data/alqueva_nivel.xlsx")
df_nivel['data'] = pd.to_datetime(df_nivel['data'])
df_nivel['nivel_smooth'] = savgol_filter(df_nivel['nivel'], window_length=window, polyorder=2)

# ==============================
# 10. Clustering de dV
# ==============================
agg_pivot = agg.pivot(index='cell_id', columns='date', values='dV').fillna(0)
k = 3
kmeans = KMeans(n_clusters=k, random_state=0)
cluster_labels = kmeans.fit_predict(agg_pivot)
cluster_df = pd.DataFrame({'cell_id': agg_pivot.index, 'cluster': cluster_labels})
cluster_colors = {i: color for i, color in enumerate(['red','green','blue','orange','purple'])}
grid_sel = grid.merge(cluster_df, on='cell_id', how='left')

# ==============================
# 11. Figura única: mapa + clusters + nível
# ==============================
clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)

fig = plt.figure(figsize=(20, 18))
gs = fig.add_gridspec(2, n_clusters, height_ratios=[2, 1.2])

# ------------------------------
# Linha 1: Mapa com legenda técnica
# ------------------------------
ax_map = fig.add_subplot(gs[0, :])
grid.boundary.plot(ax=ax_map, color='lightgray', linewidth=0.5)
grid_sel.boundary.plot(ax=ax_map, color='black', linewidth=1, alpha=0.2)

# --- Células coloridas por cluster
for i, row in grid_sel.iterrows():
    if pd.notna(row['cluster']):
        gpd.GeoSeries([row['geometry']], crs=grid_sel.crs).plot(
            ax=ax_map,
            color=cluster_colors[int(row['cluster'])],
            alpha=0.4
        )

# --- Pontos centrais (ASC/DESC)
gdf_points.plot(ax=ax_map, color='white', edgecolor='black', markersize=40)

# Adicionar item de legenda para os pontos
ax_map.scatter([], [], marker='o', color='white', edgecolor='black', s=120,
               label='Pontos ASC/DESC')

# --- Adicionar basemap
ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)

# --- Legenda dos clusters
for cluster_id in clusters_present:
    color = cluster_colors[cluster_id]
    n_cells = len(cluster_df[cluster_df["cluster"] == cluster_id])
    ax_map.scatter([], [], color=color, alpha=0.6,
                   label=f'Cluster {cluster_id + 1} - {n_cells} células')

# # --- Caixa técnica (informações do processamento)
# textstr = '\n'.join((
#     f'Tamanho da grelha: {grid_size} x {grid_size} m',
#     f'Técnica de clustering: K-Means (k = {n_clusters})',
#     'Tipo de deslocamento: dV (vertical)',
#     'Base de dados: EGMS 2019–2023',
# ))
# ax_map.text(0.99, 0.01, textstr, transform=ax_map.transAxes,
#             fontsize=10, verticalalignment='bottom', horizontalalignment='right',
#             bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.7))

# --- Título e formatação
ax_map.set_title(
    f"Clusters de séries temporais de deslocamento vertical.\n"
    f"K-Means, K={k}.\n"
    f"Grelha {grid_size} m × {grid_size} m.\n"
    f"Correlação entre a série temporal média de cada cluster e o nível da albufeira.",
    fontsize=16
)
ax_map.set_axis_off()
ax_map.legend(fontsize=10, loc='upper left')

# --- Linha 2: Séries temporais com nível ---
dV_min = agg['dV'].min()
dV_max = agg['dV'].max()
dV_margin = (dV_max - dV_min) * 0.1

nivel_min = df_nivel['nivel_smooth'].min()
nivel_max = df_nivel['nivel_smooth'].max()

for idx, cluster_id in enumerate(clusters_present):
    ax = fig.add_subplot(gs[1, idx])
    cluster_cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]

    for cid in cluster_cells:
        ax.plot(cluster_data.columns, cluster_data.loc[cid], color='lightgray', alpha=0.7)

    cluster_mean_dV = cluster_data.mean(axis=0)
    cluster_color = cluster_colors[cluster_id]
    ax.plot(cluster_data.columns, cluster_mean_dV, color=cluster_color, linewidth=2.5,
            label=f'Média Cluster {cluster_id + 1}')

    ax2 = ax.twinx()
    nivel_visual = (df_nivel['nivel_smooth'] - nivel_min) / (nivel_max - nivel_min)
    nivel_visual = nivel_visual * (dV_max - dV_min) * 0.25 + (dV_max - (dV_max - dV_min) * 0.3)
    ax2.plot(df_nivel['data'], nivel_visual, color='black', linewidth=2.2, alpha=0.85, label='Nível da albufeira (m)')
    ax2.set_ylabel("Nível da albufeira (m)", color='black')
    ax2.tick_params(axis='y', labelcolor='black')

    nivel_ticks_real = np.linspace(nivel_min, nivel_max, 6)
    nivel_ticks_visual = (nivel_ticks_real - nivel_min) / (nivel_max - nivel_min)
    nivel_ticks_visual = nivel_ticks_visual * (dV_max - dV_min) * 0.25 + (dV_max - (dV_max - dV_min) * 0.3)
    ax2.set_yticks(nivel_ticks_visual)
    ax2.set_yticklabels([f"{t:.1f}" for t in nivel_ticks_real])

    ax.set_ylim(dV_min - dV_margin, dV_max + dV_margin)
    ax2.set_ylim(dV_min - dV_margin, dV_max + dV_margin)

    ax.set_title(f'Cluster {cluster_id + 1} - {len(cluster_cells)} células', fontsize=12)
    ax.set_ylabel('dV (mm)')
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

    ax.grid(False)
    ax2.grid(False)

    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, fontsize=9, loc='upper left')

plt.tight_layout()
plt.show()

## Precipitação total

### K=2

In [ ]:
# ==============================
# 0. Bibliotecas
# ==============================
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
from scipy.signal import savgol_filter
from sklearn.cluster import KMeans

# ==============================
# 1. Ler CSVs ASC, DESC e ORTHO
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"
ortho_v_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"
ortho_h_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_E_2019_2023_1/EGMS_L3_E27N18_100km_E_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)
ortho_v = pd.read_csv(ortho_v_file)
ortho_h = pd.read_csv(ortho_h_file)

# ==============================
# 2. Filtrar área de interesse
# ==============================
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

def filter_area(df):
    return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
              (df['easting'] >= este_min) & (df['easting'] <= este_max)]

asc = filter_area(asc)
desc = filter_area(desc)
ortho_v = filter_area(ortho_v)
ortho_h = filter_area(ortho_h)

# ==============================
# 3. Função melt para ASC/DESC
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]  # ajustar conforme colunas de datas
    long_df = df.melt(
        id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'],
        value_vars=disp_cols, var_name='date', value_name='disp'
    )
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)

common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(), desc_long['date'].max()),
    freq='MS'
)

# ==============================
# 4. Interpolação temporal linear
# ==============================
def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(
            pd.to_datetime(common_dates).astype(np.int64),
            group['date'].astype(np.int64),
            group['disp']
        )
        dfs.append(pd.DataFrame({
            'easting': x,
            'northing': y,
            'latitude': group['latitude'].iloc[0],
            'longitude': group['longitude'].iloc[0],
            'date': common_dates,
            'disp': interp,
            'incidence_angle': group['incidence_angle'].iloc[0],
            'track_angle': group['track_angle'].iloc[0]
        }))
    return pd.concat(dfs, ignore_index=True)

asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 5. Interpolação espacial IDW
# ==============================
def idw_interpolation_per_date(source_df, target_df, radius=150, power=2):
    out_list = []
    for date, src_group in source_df.groupby('date'):
        trg_group = target_df[target_df['date']==date].copy()
        if src_group.empty or trg_group.empty:
            continue
        src_points = np.array(list(zip(src_group['easting'], src_group['northing'])))
        trg_points = np.array(list(zip(trg_group['easting'], trg_group['northing'])))
        tree = cKDTree(src_points)
        dists, idxs = tree.query(trg_points, k=5, distance_upper_bound=radius)

        interpolated_disp = []
        interpolated_theta = []
        interpolated_alpha = []
        for dist, idx in zip(dists, idxs):
            mask = np.isfinite(dist)
            if not np.any(mask):
                interpolated_disp.append(np.nan)
                interpolated_theta.append(np.nan)
                interpolated_alpha.append(np.nan)
                continue
            weights = 1 / (dist[mask] ** power)
            interpolated_disp.append(np.sum(weights * src_group.iloc[idx[mask]]['disp']) / np.sum(weights))
            interpolated_theta.append(np.sum(weights * src_group.iloc[idx[mask]]['incidence_angle']) / np.sum(weights))
            interpolated_alpha.append(np.sum(weights * src_group.iloc[idx[mask]]['track_angle']) / np.sum(weights))

        trg_group['disp_idw'] = interpolated_disp
        trg_group['theta_desc'] = interpolated_theta
        trg_group['alpha_desc'] = interpolated_alpha
        out_list.append(trg_group)
    return pd.concat(out_list, ignore_index=True)

asc_interp = idw_interpolation_per_date(desc_interp, asc_interp)

# ==============================
# 6. Calcular β e γ
# ==============================
orbit_inclination = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orbit_inclination) * np.cos(np.deg2rad(asc_interp['latitude'])))
asc_interp['gamma'] = 0

# ==============================
# 7. Calcular dV e dH
# ==============================
def compute_dV_dH_real(row):
    θA = np.deg2rad(row['incidence_angle'])
    θD = np.deg2rad(row['theta_desc'])
    αA = np.deg2rad(row['track_angle'])
    αD = np.deg2rad(row['alpha_desc'])
    β = row['beta']
    γ = row['gamma']

    dASC_LOS = row['disp']
    dDESC_LOS = row['disp_idw']

    denom = (np.cos(θA)*np.sin(θD)*np.cos(β + γ) +
             np.cos(θD)*np.sin(θA)*np.cos(β - γ))

    dV = (dDESC_LOS*np.sin(θA)*np.cos(β - γ) +
          dASC_LOS*np.sin(θD)*np.cos(β + γ)) / denom
    dH = (dDESC_LOS*np.cos(θA) - dASC_LOS*np.cos(θD)) / denom
    return pd.Series({'dV': dV, 'dH': dH})

asc_interp[['dV','dH']] = asc_interp.apply(compute_dV_dH_real, axis=1)

# ==============================
# 8. Criar grelha centrada ORTHO
# ==============================
grid_size = 100
x_edges = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

asc_interp['cell_x'] = pd.cut(asc_interp['easting'], bins=x_edges_shifted, labels=False)
asc_interp['cell_y'] = pd.cut(asc_interp['northing'], bins=y_edges_shifted, labels=False)

# Remover NaNs antes de criar cell_id
asc_interp = asc_interp.dropna(subset=['cell_x','cell_y'])
asc_interp['cell_id'] = asc_interp['cell_x'].astype(int).astype(str) + "_" + asc_interp['cell_y'].astype(int).astype(str)

# Agrupar para ponto central de cada célula
points = asc_interp.groupby('cell_id').agg({'easting':'mean','northing':'mean'}).reset_index()
gdf_points = gpd.GeoDataFrame(
    points,
    geometry=gpd.points_from_xy(points['easting'], points['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

agg = asc_interp.groupby(['cell_x','cell_y','date']).agg(
    x_center=('easting','mean'),
    y_center=('northing','mean'),
    dV=('dV','mean'),
    dH=('dH','mean')
).reset_index()
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + "_" + agg['cell_y'].astype(int).astype(str)

# ==============================
# 9. Criar GeoDataFrame da grelha
# ==============================
grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({
            "cell_x": ix,
            "cell_y": iy,
            "cell_id": f"{ix}_{iy}",
            "geometry": box(x_edges_shifted[ix], y_edges_shifted[iy],
                            x_edges_shifted[ix+1], y_edges_shifted[iy+1])
        })
grid = gpd.GeoDataFrame(grid_data, crs="EPSG:3035").to_crs(epsg=3857)

# ==============================
# 10. Carregar precipitação
# ==============================
df_prec = pd.read_excel("data/prec.xlsx")
df_prec['data'] = pd.to_datetime(df_prec['data'])
# (sem suavização)

# ==============================
# 11. Clustering de dV
# ==============================
agg_pivot = agg.pivot(index='cell_id', columns='date', values='dV').fillna(0)
k = 2
kmeans = KMeans(n_clusters=k, random_state=0)
cluster_labels = kmeans.fit_predict(agg_pivot)
cluster_df = pd.DataFrame({'cell_id': agg_pivot.index, 'cluster': cluster_labels})

# Mapear cores
cluster_colors = {i: color for i, color in enumerate(['red','green','blue','orange','purple'])}
grid_sel = grid.merge(cluster_df, on='cell_id', how='left')

import matplotlib.dates as mdates

# ==============================
# 14. Figura única: mapa + clusters
# ==============================
clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)

# Criar figura com 2 linhas: mapa + séries temporais
fig = plt.figure(figsize=(20, 18))
gs = fig.add_gridspec(2, n_clusters, height_ratios=[2, 1.2])

# ------------------------------
# Linha 1: Mapa com legenda técnica
# ------------------------------
ax_map = fig.add_subplot(gs[0, :])
grid.boundary.plot(ax=ax_map, color='lightgray', linewidth=0.5)
grid_sel.boundary.plot(ax=ax_map, color='black', linewidth=1, alpha=0.2)

# --- Células coloridas por cluster
for i, row in grid_sel.iterrows():
    if pd.notna(row['cluster']):
        gpd.GeoSeries([row['geometry']], crs=grid_sel.crs).plot(
            ax=ax_map,
            color=cluster_colors[int(row['cluster'])],
            alpha=0.4
        )

# --- Pontos centrais (ASC/DESC)
gdf_points.plot(ax=ax_map, color='white', edgecolor='black', markersize=40)

# Adicionar item de legenda para os pontos
ax_map.scatter([], [], marker='o', color='white', edgecolor='black', s=120,
               label='Pontos ASC/DESC')

# --- Adicionar basemap
ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)

# --- Legenda dos clusters
for cluster_id in clusters_present:
    color = cluster_colors[cluster_id]
    n_cells = len(cluster_df[cluster_df["cluster"] == cluster_id])
    ax_map.scatter([], [], color=color, alpha=0.6,
                   label=f'Cluster {cluster_id + 1} - {n_cells} células')

# --- Caixa técnica (informações do processamento)
# textstr = '\n'.join((
#     f'Tamanho da grelha: {grid_size} x {grid_size} m',
#     f'Técnica de clustering: K-Means (k = {n_clusters})',
#     'Tipo de deslocamento: dV (vertical)',
#     'Base de dados: EGMS 2019–2023',
# ))
# ax_map.text(0.99, 0.01, textstr, transform=ax_map.transAxes,
#             fontsize=10, verticalalignment='bottom', horizontalalignment='right',
#             bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.7))

# --- Título e formatação
ax_map.set_title(
    f"Clusters de séries temporais de deslocamento vertical.\n"
    f"K-Means, K={k}.\n"
    f"Grelha {grid_size} m × {grid_size} m.\n"
    f"Correlação entre a série temporal média de cada cluster e a precipitação total.",
    fontsize=16
)
ax_map.set_axis_off()
ax_map.legend(fontsize=10, loc='upper left')

# ==============================
# Linha 2: Séries temporais (ajustada para precipitação)
# ==============================
dV_min = agg['dV'].min()
dV_max = agg['dV'].max()
dV_margin = (dV_max - dV_min) * 0.1

for idx, cluster_id in enumerate(clusters_present):
    ax = fig.add_subplot(gs[1, idx])
    
    cluster_cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]

    # --- Série dV ---
    for cid in cluster_cells:
        ax.plot(cluster_data.columns, cluster_data.loc[cid], color='lightgray', alpha=0.7)
    
    cluster_mean_dV = cluster_data.mean(axis=0)
    cluster_color = cluster_colors[cluster_id]
    ax.plot(cluster_data.columns, cluster_mean_dV, color=cluster_color, linewidth=2.5,
            label=f'Média Cluster {cluster_id + 1}')

    # --- Precipitação total como barras ---
    ax2 = ax.twinx()
    ax2.bar(df_prec['data'], df_prec['prec'], width=20, color='blue', alpha=0.3, label='Precipitação (mm)')
    ax2.set_ylabel("Precipitação (mm)", color='blue')
    ax2.tick_params(axis='y', labelcolor='blue')

    # --- Estética ---
    ax.set_ylim(dV_min - dV_margin, dV_max + dV_margin)
    ax2.set_ylim(0, df_prec['prec'].max() * 1.1)

    ax.set_title(f'Cluster {cluster_id + 1} - {len(cluster_cells)} células', fontsize=12)
    ax.set_ylabel('dV (mm)')
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

    ax.grid(False)
    ax2.grid(False)

    # --- Legenda combinada ---
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, fontsize=9, loc='upper left')

plt.tight_layout()
plt.show()

### K=3

In [ ]:
# ==============================
# 0. Bibliotecas
# ==============================
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
from scipy.signal import savgol_filter
from sklearn.cluster import KMeans

# ==============================
# 1. Ler CSVs ASC, DESC e ORTHO
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"
ortho_v_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"
ortho_h_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_E_2019_2023_1/EGMS_L3_E27N18_100km_E_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)
ortho_v = pd.read_csv(ortho_v_file)
ortho_h = pd.read_csv(ortho_h_file)

# ==============================
# 2. Filtrar área de interesse
# ==============================
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

def filter_area(df):
    return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
              (df['easting'] >= este_min) & (df['easting'] <= este_max)]

asc = filter_area(asc)
desc = filter_area(desc)
ortho_v = filter_area(ortho_v)
ortho_h = filter_area(ortho_h)

# ==============================
# 3. Função melt para ASC/DESC
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]  # ajustar conforme colunas de datas
    long_df = df.melt(
        id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'],
        value_vars=disp_cols, var_name='date', value_name='disp'
    )
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)

common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(), desc_long['date'].max()),
    freq='MS'
)

# ==============================
# 4. Interpolação temporal linear
# ==============================
def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(
            pd.to_datetime(common_dates).astype(np.int64),
            group['date'].astype(np.int64),
            group['disp']
        )
        dfs.append(pd.DataFrame({
            'easting': x,
            'northing': y,
            'latitude': group['latitude'].iloc[0],
            'longitude': group['longitude'].iloc[0],
            'date': common_dates,
            'disp': interp,
            'incidence_angle': group['incidence_angle'].iloc[0],
            'track_angle': group['track_angle'].iloc[0]
        }))
    return pd.concat(dfs, ignore_index=True)

asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 5. Interpolação espacial IDW
# ==============================
def idw_interpolation_per_date(source_df, target_df, radius=150, power=2):
    out_list = []
    for date, src_group in source_df.groupby('date'):
        trg_group = target_df[target_df['date']==date].copy()
        if src_group.empty or trg_group.empty:
            continue
        src_points = np.array(list(zip(src_group['easting'], src_group['northing'])))
        trg_points = np.array(list(zip(trg_group['easting'], trg_group['northing'])))
        tree = cKDTree(src_points)
        dists, idxs = tree.query(trg_points, k=5, distance_upper_bound=radius)

        interpolated_disp = []
        interpolated_theta = []
        interpolated_alpha = []
        for dist, idx in zip(dists, idxs):
            mask = np.isfinite(dist)
            if not np.any(mask):
                interpolated_disp.append(np.nan)
                interpolated_theta.append(np.nan)
                interpolated_alpha.append(np.nan)
                continue
            weights = 1 / (dist[mask] ** power)
            interpolated_disp.append(np.sum(weights * src_group.iloc[idx[mask]]['disp']) / np.sum(weights))
            interpolated_theta.append(np.sum(weights * src_group.iloc[idx[mask]]['incidence_angle']) / np.sum(weights))
            interpolated_alpha.append(np.sum(weights * src_group.iloc[idx[mask]]['track_angle']) / np.sum(weights))

        trg_group['disp_idw'] = interpolated_disp
        trg_group['theta_desc'] = interpolated_theta
        trg_group['alpha_desc'] = interpolated_alpha
        out_list.append(trg_group)
    return pd.concat(out_list, ignore_index=True)

asc_interp = idw_interpolation_per_date(desc_interp, asc_interp)

# ==============================
# 6. Calcular β e γ
# ==============================
orbit_inclination = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orbit_inclination) * np.cos(np.deg2rad(asc_interp['latitude'])))
asc_interp['gamma'] = 0

# ==============================
# 7. Calcular dV e dH
# ==============================
def compute_dV_dH_real(row):
    θA = np.deg2rad(row['incidence_angle'])
    θD = np.deg2rad(row['theta_desc'])
    αA = np.deg2rad(row['track_angle'])
    αD = np.deg2rad(row['alpha_desc'])
    β = row['beta']
    γ = row['gamma']

    dASC_LOS = row['disp']
    dDESC_LOS = row['disp_idw']

    denom = (np.cos(θA)*np.sin(θD)*np.cos(β + γ) +
             np.cos(θD)*np.sin(θA)*np.cos(β - γ))

    dV = (dDESC_LOS*np.sin(θA)*np.cos(β - γ) +
          dASC_LOS*np.sin(θD)*np.cos(β + γ)) / denom
    dH = (dDESC_LOS*np.cos(θA) - dASC_LOS*np.cos(θD)) / denom
    return pd.Series({'dV': dV, 'dH': dH})

asc_interp[['dV','dH']] = asc_interp.apply(compute_dV_dH_real, axis=1)

# ==============================
# 8. Criar grelha centrada ORTHO
# ==============================
grid_size = 100
x_edges = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

asc_interp['cell_x'] = pd.cut(asc_interp['easting'], bins=x_edges_shifted, labels=False)
asc_interp['cell_y'] = pd.cut(asc_interp['northing'], bins=y_edges_shifted, labels=False)

# Remover NaNs antes de criar cell_id
asc_interp = asc_interp.dropna(subset=['cell_x','cell_y'])
asc_interp['cell_id'] = asc_interp['cell_x'].astype(int).astype(str) + "_" + asc_interp['cell_y'].astype(int).astype(str)

# Agrupar para ponto central de cada célula
points = asc_interp.groupby('cell_id').agg({'easting':'mean','northing':'mean'}).reset_index()
gdf_points = gpd.GeoDataFrame(
    points,
    geometry=gpd.points_from_xy(points['easting'], points['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

agg = asc_interp.groupby(['cell_x','cell_y','date']).agg(
    x_center=('easting','mean'),
    y_center=('northing','mean'),
    dV=('dV','mean'),
    dH=('dH','mean')
).reset_index()
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + "_" + agg['cell_y'].astype(int).astype(str)

# ==============================
# 9. Criar GeoDataFrame da grelha
# ==============================
grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({
            "cell_x": ix,
            "cell_y": iy,
            "cell_id": f"{ix}_{iy}",
            "geometry": box(x_edges_shifted[ix], y_edges_shifted[iy],
                            x_edges_shifted[ix+1], y_edges_shifted[iy+1])
        })
grid = gpd.GeoDataFrame(grid_data, crs="EPSG:3035").to_crs(epsg=3857)

# ==============================
# 10. Carregar precipitação
# ==============================
df_prec = pd.read_excel("data/prec.xlsx")
df_prec['data'] = pd.to_datetime(df_prec['data'])
# (sem suavização)

# ==============================
# 11. Clustering de dV
# ==============================
agg_pivot = agg.pivot(index='cell_id', columns='date', values='dV').fillna(0)
k = 3
kmeans = KMeans(n_clusters=k, random_state=0)
cluster_labels = kmeans.fit_predict(agg_pivot)
cluster_df = pd.DataFrame({'cell_id': agg_pivot.index, 'cluster': cluster_labels})

# Mapear cores
cluster_colors = {i: color for i, color in enumerate(['red','green','blue','orange','purple'])}
grid_sel = grid.merge(cluster_df, on='cell_id', how='left')

import matplotlib.dates as mdates

# ==============================
# 14. Figura única: mapa + clusters
# ==============================
clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)

# Criar figura com 2 linhas: mapa + séries temporais
fig = plt.figure(figsize=(20, 18))
gs = fig.add_gridspec(2, n_clusters, height_ratios=[2, 1.2])

# ------------------------------
# Linha 1: Mapa com legenda técnica
# ------------------------------
ax_map = fig.add_subplot(gs[0, :])
grid.boundary.plot(ax=ax_map, color='lightgray', linewidth=0.5)
grid_sel.boundary.plot(ax=ax_map, color='black', linewidth=1, alpha=0.2)

# --- Células coloridas por cluster
for i, row in grid_sel.iterrows():
    if pd.notna(row['cluster']):
        gpd.GeoSeries([row['geometry']], crs=grid_sel.crs).plot(
            ax=ax_map,
            color=cluster_colors[int(row['cluster'])],
            alpha=0.4
        )

# --- Pontos centrais (ASC/DESC)
gdf_points.plot(ax=ax_map, color='white', edgecolor='black', markersize=40)

# Adicionar item de legenda para os pontos
ax_map.scatter([], [], marker='o', color='white', edgecolor='black', s=120,
               label='Pontos ASC/DESC')

# --- Adicionar basemap
ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)

# --- Legenda dos clusters
for cluster_id in clusters_present:
    color = cluster_colors[cluster_id]
    n_cells = len(cluster_df[cluster_df["cluster"] == cluster_id])
    ax_map.scatter([], [], color=color, alpha=0.6,
                   label=f'Cluster {cluster_id + 1} - {n_cells} células')

# --- Caixa técnica (informações do processamento)
# textstr = '\n'.join((
#     f'Tamanho da grelha: {grid_size} x {grid_size} m',
#     f'Técnica de clustering: K-Means (k = {n_clusters})',
#     'Tipo de deslocamento: dV (vertical)',
#     'Base de dados: EGMS 2019–2023',
# ))
# ax_map.text(0.99, 0.01, textstr, transform=ax_map.transAxes,
#             fontsize=10, verticalalignment='bottom', horizontalalignment='right',
#             bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.7))

# --- Título e formatação
ax_map.set_title(
    f"Clusters de séries temporais de deslocamento vertical.\n"
    f"K-Means, K={k}.\n"
    f"Grelha {grid_size} m × {grid_size} m.\n"
    f"Correlação entre a série temporal média de cada cluster e a precipitação total.",
    fontsize=16
)
ax_map.set_axis_off()
ax_map.legend(fontsize=10, loc='upper left')

# ==============================
# Linha 2: Séries temporais (ajustada para precipitação)
# ==============================
dV_min = agg['dV'].min()
dV_max = agg['dV'].max()
dV_margin = (dV_max - dV_min) * 0.1

for idx, cluster_id in enumerate(clusters_present):
    ax = fig.add_subplot(gs[1, idx])
    
    cluster_cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]

    # --- Série dV ---
    for cid in cluster_cells:
        ax.plot(cluster_data.columns, cluster_data.loc[cid], color='lightgray', alpha=0.7)
    
    cluster_mean_dV = cluster_data.mean(axis=0)
    cluster_color = cluster_colors[cluster_id]
    ax.plot(cluster_data.columns, cluster_mean_dV, color=cluster_color, linewidth=2.5,
            label=f'Média Cluster {cluster_id + 1}')

    # --- Precipitação total como barras ---
    ax2 = ax.twinx()
    ax2.bar(df_prec['data'], df_prec['prec'], width=20, color='blue', alpha=0.3, label='Precipitação (mm)')
    ax2.set_ylabel("Precipitação (mm)", color='blue')
    ax2.tick_params(axis='y', labelcolor='blue')

    # --- Estética ---
    ax.set_ylim(dV_min - dV_margin, dV_max + dV_margin)
    ax2.set_ylim(0, df_prec['prec'].max() * 1.1)

    ax.set_title(f'Cluster {cluster_id + 1} - {len(cluster_cells)} células', fontsize=12)
    ax.set_ylabel('dV (mm)')
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

    ax.grid(False)
    ax2.grid(False)

    # --- Legenda combinada ---
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, fontsize=9, loc='upper left')

plt.tight_layout()
plt.show()

In [ ]:
# ==============================
# 0. Bibliotecas
# ==============================
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
from scipy.signal import savgol_filter
from sklearn.cluster import KMeans

# ==============================
# 1. Ler CSVs ASC, DESC e ORTHO
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"
ortho_v_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"
ortho_h_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_E_2019_2023_1/EGMS_L3_E27N18_100km_E_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)
ortho_v = pd.read_csv(ortho_v_file)
ortho_h = pd.read_csv(ortho_h_file)

# ==============================
# 2. Filtrar área de interesse
# ==============================
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

def filter_area(df):
    return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
              (df['easting'] >= este_min) & (df['easting'] <= este_max)]

asc = filter_area(asc)
desc = filter_area(desc)
ortho_v = filter_area(ortho_v)
ortho_h = filter_area(ortho_h)

# ==============================
# 3. Função melt para ASC/DESC
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]  # ajustar conforme colunas de datas
    long_df = df.melt(
        id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'],
        value_vars=disp_cols, var_name='date', value_name='disp'
    )
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)

common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(), desc_long['date'].max()),
    freq='MS'
)

# ==============================
# 4. Interpolação temporal linear
# ==============================
def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(
            pd.to_datetime(common_dates).astype(np.int64),
            group['date'].astype(np.int64),
            group['disp']
        )
        dfs.append(pd.DataFrame({
            'easting': x,
            'northing': y,
            'latitude': group['latitude'].iloc[0],
            'longitude': group['longitude'].iloc[0],
            'date': common_dates,
            'disp': interp,
            'incidence_angle': group['incidence_angle'].iloc[0],
            'track_angle': group['track_angle'].iloc[0]
        }))
    return pd.concat(dfs, ignore_index=True)

asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 5. Interpolação espacial IDW
# ==============================
def idw_interpolation_per_date(source_df, target_df, radius=150, power=2):
    out_list = []
    for date, src_group in source_df.groupby('date'):
        trg_group = target_df[target_df['date']==date].copy()
        if src_group.empty or trg_group.empty:
            continue
        src_points = np.array(list(zip(src_group['easting'], src_group['northing'])))
        trg_points = np.array(list(zip(trg_group['easting'], trg_group['northing'])))
        tree = cKDTree(src_points)
        dists, idxs = tree.query(trg_points, k=5, distance_upper_bound=radius)

        interpolated_disp = []
        interpolated_theta = []
        interpolated_alpha = []
        for dist, idx in zip(dists, idxs):
            mask = np.isfinite(dist)
            if not np.any(mask):
                interpolated_disp.append(np.nan)
                interpolated_theta.append(np.nan)
                interpolated_alpha.append(np.nan)
                continue
            weights = 1 / (dist[mask] ** power)
            interpolated_disp.append(np.sum(weights * src_group.iloc[idx[mask]]['disp']) / np.sum(weights))
            interpolated_theta.append(np.sum(weights * src_group.iloc[idx[mask]]['incidence_angle']) / np.sum(weights))
            interpolated_alpha.append(np.sum(weights * src_group.iloc[idx[mask]]['track_angle']) / np.sum(weights))

        trg_group['disp_idw'] = interpolated_disp
        trg_group['theta_desc'] = interpolated_theta
        trg_group['alpha_desc'] = interpolated_alpha
        out_list.append(trg_group)
    return pd.concat(out_list, ignore_index=True)

asc_interp = idw_interpolation_per_date(desc_interp, asc_interp)

# ==============================
# 6. Calcular β e γ
# ==============================
orbit_inclination = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orbit_inclination) * np.cos(np.deg2rad(asc_interp['latitude'])))
asc_interp['gamma'] = 0

# ==============================
# 7. Calcular dV e dH
# ==============================
def compute_dV_dH_real(row):
    θA = np.deg2rad(row['incidence_angle'])
    θD = np.deg2rad(row['theta_desc'])
    αA = np.deg2rad(row['track_angle'])
    αD = np.deg2rad(row['alpha_desc'])
    β = row['beta']
    γ = row['gamma']

    dASC_LOS = row['disp']
    dDESC_LOS = row['disp_idw']

    denom = (np.cos(θA)*np.sin(θD)*np.cos(β + γ) +
             np.cos(θD)*np.sin(θA)*np.cos(β - γ))

    dV = (dDESC_LOS*np.sin(θA)*np.cos(β - γ) +
          dASC_LOS*np.sin(θD)*np.cos(β + γ)) / denom
    dH = (dDESC_LOS*np.cos(θA) - dASC_LOS*np.cos(θD)) / denom
    return pd.Series({'dV': dV, 'dH': dH})

asc_interp[['dV','dH']] = asc_interp.apply(compute_dV_dH_real, axis=1)

# ==============================
# 8. Criar grelha centrada ORTHO
# ==============================
grid_size = 50
x_edges = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

asc_interp['cell_x'] = pd.cut(asc_interp['easting'], bins=x_edges_shifted, labels=False)
asc_interp['cell_y'] = pd.cut(asc_interp['northing'], bins=y_edges_shifted, labels=False)

# Remover NaNs antes de criar cell_id
asc_interp = asc_interp.dropna(subset=['cell_x','cell_y'])
asc_interp['cell_id'] = asc_interp['cell_x'].astype(int).astype(str) + "_" + asc_interp['cell_y'].astype(int).astype(str)

# Agrupar para ponto central de cada célula
points = asc_interp.groupby('cell_id').agg({'easting':'mean','northing':'mean'}).reset_index()
gdf_points = gpd.GeoDataFrame(
    points,
    geometry=gpd.points_from_xy(points['easting'], points['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

agg = asc_interp.groupby(['cell_x','cell_y','date']).agg(
    x_center=('easting','mean'),
    y_center=('northing','mean'),
    dV=('dV','mean'),
    dH=('dH','mean')
).reset_index()
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + "_" + agg['cell_y'].astype(int).astype(str)

# ==============================
# 9. Criar GeoDataFrame da grelha
# ==============================
grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({
            "cell_x": ix,
            "cell_y": iy,
            "cell_id": f"{ix}_{iy}",
            "geometry": box(x_edges_shifted[ix], y_edges_shifted[iy],
                            x_edges_shifted[ix+1], y_edges_shifted[iy+1])
        })
grid = gpd.GeoDataFrame(grid_data, crs="EPSG:3035").to_crs(epsg=3857)

# ==============================
# 10. Carregar precipitação
# ==============================
df_prec = pd.read_excel("data/prec.xlsx")
df_prec['data'] = pd.to_datetime(df_prec['data'])
# (sem suavização)

# ==============================
# 11. Clustering de dV
# ==============================
agg_pivot = agg.pivot(index='cell_id', columns='date', values='dV').fillna(0)
k = 3
kmeans = KMeans(n_clusters=k, random_state=0)
cluster_labels = kmeans.fit_predict(agg_pivot)
cluster_df = pd.DataFrame({'cell_id': agg_pivot.index, 'cluster': cluster_labels})

# Mapear cores
cluster_colors = {i: color for i, color in enumerate(['red','green','blue','orange','purple'])}
grid_sel = grid.merge(cluster_df, on='cell_id', how='left')

import matplotlib.dates as mdates

# ==============================
# 14. Figura única: mapa + clusters
# ==============================
clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)

# Criar figura com 2 linhas: mapa + séries temporais
fig = plt.figure(figsize=(20, 18))
gs = fig.add_gridspec(2, n_clusters, height_ratios=[2, 1.2])

# ------------------------------
# Linha 1: Mapa
# ------------------------------
ax_map = fig.add_subplot(gs[0, :])
grid.boundary.plot(ax=ax_map, color='lightgray', linewidth=0.5)
grid_sel.boundary.plot(ax=ax_map, color='black', linewidth=1, alpha=0.2)

for i, row in grid_sel.iterrows():
    if pd.notna(row['cluster']):
        gpd.GeoSeries([row['geometry']], crs=grid_sel.crs).plot(
            ax=ax_map,
            color=cluster_colors[int(row['cluster'])],
            alpha=0.4
        )
gdf_points.plot(ax=ax_map, color='white', edgecolor='black', markersize=40, label='Centro de massa ASC/DESC')

for cluster_id in clusters_present:
    color = cluster_colors[cluster_id]
    ax_map.scatter([], [], color=color, alpha=0.4, label=f'Cluster {cluster_id+1}')

ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)
ax_map.set_title(
    f"Clusters de séries temporais de deslocamento vertical.\n"
    f"K-Means, K={k}.\n"
    f"Grelha {grid_size} m × {grid_size} m.\n"
    f"Correlação entre a série temporal média de cada cluster e a precipitação total anual acumulada.",
    fontsize=16
)
ax_map.set_axis_off()
ax_map.legend(fontsize=10)

# --- Linha 2: Séries temporais ---
dV_min = agg['dV'].min()
dV_max = agg['dV'].max()
dV_margin = (dV_max - dV_min) * 0.1

for idx, cluster_id in enumerate(clusters_present):
    ax = fig.add_subplot(gs[1, idx])
    
    cluster_cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]

    # Séries dV
    for cid in cluster_cells:
        ax.plot(cluster_data.columns, cluster_data.loc[cid], color='lightgray', alpha=0.7)
    
    cluster_mean_dV = cluster_data.mean(axis=0)
    cluster_color = cluster_colors[cluster_id]
    ax.plot(cluster_data.columns, cluster_mean_dV, color=cluster_color, linewidth=2.5,
            label=f'Média Cluster {cluster_id + 1}')

    # --- Precipitação (barras + linha média) ---
    ax2 = ax.twinx()

    # Barras de precipitação
    bar_container = ax2.bar(
        df_prec['data'], df_prec['prec'],
        width=20, color='royalblue', alpha=0.35, label='Precipitação (mm)'
    )

    # Coordenadas dos centros das barras e respetivos valores
    bar_centers = [bar.get_x() + bar.get_width()/2 for bar in bar_container]
    bar_heights = [bar.get_height() for bar in bar_container]

    # Linha conectando os pontos médios das barras
    ax2.plot(bar_centers, bar_heights, color='navy', linewidth=2.0, label='Tendência da precipitação')

    # Eixos e rótulos
    ax2.set_ylabel("Precipitação (mm)", color='navy')
    ax2.tick_params(axis='y', labelcolor='navy')

    # --- Eixos e estilo ---
    ax.set_ylim(dV_min - dV_margin, dV_max + dV_margin)
    ax2.set_ylim(0, df_prec['prec'].max() * 1.1)
    ax.set_title(f'Cluster {cluster_id + 1} - {len(cluster_cells)} células', fontsize=12)
    ax.set_ylabel('dV (mm)')
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.grid(False)
    ax2.grid(False)

    # --- Legenda combinada ---
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, fontsize=9, loc='upper left')

plt.tight_layout()
plt.show()

### K=4

In [ ]:
# ==============================
# 0. Bibliotecas
# ==============================
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
from scipy.signal import savgol_filter
from sklearn.cluster import KMeans

# ==============================
# 1. Ler CSVs ASC, DESC e ORTHO
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"
ortho_v_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"
ortho_h_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_E_2019_2023_1/EGMS_L3_E27N18_100km_E_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)
ortho_v = pd.read_csv(ortho_v_file)
ortho_h = pd.read_csv(ortho_h_file)

# ==============================
# 2. Filtrar área de interesse
# ==============================
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

def filter_area(df):
    return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
              (df['easting'] >= este_min) & (df['easting'] <= este_max)]

asc = filter_area(asc)
desc = filter_area(desc)
ortho_v = filter_area(ortho_v)
ortho_h = filter_area(ortho_h)

# ==============================
# 3. Função melt para ASC/DESC
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]  # ajustar conforme colunas de datas
    long_df = df.melt(
        id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'],
        value_vars=disp_cols, var_name='date', value_name='disp'
    )
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)

common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(), desc_long['date'].max()),
    freq='MS'
)

# ==============================
# 4. Interpolação temporal linear
# ==============================
def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(
            pd.to_datetime(common_dates).astype(np.int64),
            group['date'].astype(np.int64),
            group['disp']
        )
        dfs.append(pd.DataFrame({
            'easting': x,
            'northing': y,
            'latitude': group['latitude'].iloc[0],
            'longitude': group['longitude'].iloc[0],
            'date': common_dates,
            'disp': interp,
            'incidence_angle': group['incidence_angle'].iloc[0],
            'track_angle': group['track_angle'].iloc[0]
        }))
    return pd.concat(dfs, ignore_index=True)

asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 5. Interpolação espacial IDW
# ==============================
def idw_interpolation_per_date(source_df, target_df, radius=150, power=2):
    out_list = []
    for date, src_group in source_df.groupby('date'):
        trg_group = target_df[target_df['date']==date].copy()
        if src_group.empty or trg_group.empty:
            continue
        src_points = np.array(list(zip(src_group['easting'], src_group['northing'])))
        trg_points = np.array(list(zip(trg_group['easting'], trg_group['northing'])))
        tree = cKDTree(src_points)
        dists, idxs = tree.query(trg_points, k=5, distance_upper_bound=radius)

        interpolated_disp = []
        interpolated_theta = []
        interpolated_alpha = []
        for dist, idx in zip(dists, idxs):
            mask = np.isfinite(dist)
            if not np.any(mask):
                interpolated_disp.append(np.nan)
                interpolated_theta.append(np.nan)
                interpolated_alpha.append(np.nan)
                continue
            weights = 1 / (dist[mask] ** power)
            interpolated_disp.append(np.sum(weights * src_group.iloc[idx[mask]]['disp']) / np.sum(weights))
            interpolated_theta.append(np.sum(weights * src_group.iloc[idx[mask]]['incidence_angle']) / np.sum(weights))
            interpolated_alpha.append(np.sum(weights * src_group.iloc[idx[mask]]['track_angle']) / np.sum(weights))

        trg_group['disp_idw'] = interpolated_disp
        trg_group['theta_desc'] = interpolated_theta
        trg_group['alpha_desc'] = interpolated_alpha
        out_list.append(trg_group)
    return pd.concat(out_list, ignore_index=True)

asc_interp = idw_interpolation_per_date(desc_interp, asc_interp)

# ==============================
# 6. Calcular β e γ
# ==============================
orbit_inclination = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orbit_inclination) * np.cos(np.deg2rad(asc_interp['latitude'])))
asc_interp['gamma'] = 0

# ==============================
# 7. Calcular dV e dH
# ==============================
def compute_dV_dH_real(row):
    θA = np.deg2rad(row['incidence_angle'])
    θD = np.deg2rad(row['theta_desc'])
    αA = np.deg2rad(row['track_angle'])
    αD = np.deg2rad(row['alpha_desc'])
    β = row['beta']
    γ = row['gamma']

    dASC_LOS = row['disp']
    dDESC_LOS = row['disp_idw']

    denom = (np.cos(θA)*np.sin(θD)*np.cos(β + γ) +
             np.cos(θD)*np.sin(θA)*np.cos(β - γ))

    dV = (dDESC_LOS*np.sin(θA)*np.cos(β - γ) +
          dASC_LOS*np.sin(θD)*np.cos(β + γ)) / denom
    dH = (dDESC_LOS*np.cos(θA) - dASC_LOS*np.cos(θD)) / denom
    return pd.Series({'dV': dV, 'dH': dH})

asc_interp[['dV','dH']] = asc_interp.apply(compute_dV_dH_real, axis=1)

# ==============================
# 8. Criar grelha centrada ORTHO
# ==============================
grid_size = 100
x_edges = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

asc_interp['cell_x'] = pd.cut(asc_interp['easting'], bins=x_edges_shifted, labels=False)
asc_interp['cell_y'] = pd.cut(asc_interp['northing'], bins=y_edges_shifted, labels=False)

# Remover NaNs antes de criar cell_id
asc_interp = asc_interp.dropna(subset=['cell_x','cell_y'])
asc_interp['cell_id'] = asc_interp['cell_x'].astype(int).astype(str) + "_" + asc_interp['cell_y'].astype(int).astype(str)

# Agrupar para ponto central de cada célula
points = asc_interp.groupby('cell_id').agg({'easting':'mean','northing':'mean'}).reset_index()
gdf_points = gpd.GeoDataFrame(
    points,
    geometry=gpd.points_from_xy(points['easting'], points['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

agg = asc_interp.groupby(['cell_x','cell_y','date']).agg(
    x_center=('easting','mean'),
    y_center=('northing','mean'),
    dV=('dV','mean'),
    dH=('dH','mean')
).reset_index()
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + "_" + agg['cell_y'].astype(int).astype(str)

# ==============================
# 9. Criar GeoDataFrame da grelha
# ==============================
grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({
            "cell_x": ix,
            "cell_y": iy,
            "cell_id": f"{ix}_{iy}",
            "geometry": box(x_edges_shifted[ix], y_edges_shifted[iy],
                            x_edges_shifted[ix+1], y_edges_shifted[iy+1])
        })
grid = gpd.GeoDataFrame(grid_data, crs="EPSG:3035").to_crs(epsg=3857)

# ==============================
# 10. Carregar precipitação
# ==============================
df_prec = pd.read_excel("data/prec.xlsx")
df_prec['data'] = pd.to_datetime(df_prec['data'])
# (sem suavização)

# ==============================
# 11. Clustering de dV
# ==============================
agg_pivot = agg.pivot(index='cell_id', columns='date', values='dV').fillna(0)
k = 4
kmeans = KMeans(n_clusters=k, random_state=0)
cluster_labels = kmeans.fit_predict(agg_pivot)
cluster_df = pd.DataFrame({'cell_id': agg_pivot.index, 'cluster': cluster_labels})

# Mapear cores
cluster_colors = {i: color for i, color in enumerate(['red','green','blue','orange','purple'])}
grid_sel = grid.merge(cluster_df, on='cell_id', how='left')

import matplotlib.dates as mdates

# ==============================
# 14. Figura única: mapa + clusters
# ==============================
clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)

# Criar figura com 2 linhas: mapa + séries temporais
fig = plt.figure(figsize=(20, 18))
gs = fig.add_gridspec(2, n_clusters, height_ratios=[2, 1.2])

# ------------------------------
# Linha 1: Mapa com legenda técnica
# ------------------------------
ax_map = fig.add_subplot(gs[0, :])
grid.boundary.plot(ax=ax_map, color='lightgray', linewidth=0.5)
grid_sel.boundary.plot(ax=ax_map, color='black', linewidth=1, alpha=0.2)

# --- Células coloridas por cluster
for i, row in grid_sel.iterrows():
    if pd.notna(row['cluster']):
        gpd.GeoSeries([row['geometry']], crs=grid_sel.crs).plot(
            ax=ax_map,
            color=cluster_colors[int(row['cluster'])],
            alpha=0.4
        )

# --- Pontos centrais (ASC/DESC)
gdf_points.plot(ax=ax_map, color='white', edgecolor='black', markersize=40)

# Adicionar item de legenda para os pontos
ax_map.scatter([], [], marker='o', color='white', edgecolor='black', s=120,
               label='Pontos ASC/DESC')

# --- Adicionar basemap
ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)

# --- Legenda dos clusters
for cluster_id in clusters_present:
    color = cluster_colors[cluster_id]
    n_cells = len(cluster_df[cluster_df["cluster"] == cluster_id])
    ax_map.scatter([], [], color=color, alpha=0.6,
                   label=f'Cluster {cluster_id + 1} - {n_cells} células')

# --- Caixa técnica (informações do processamento)
# textstr = '\n'.join((
#     f'Tamanho da grelha: {grid_size} x {grid_size} m',
#     f'Técnica de clustering: K-Means (k = {n_clusters})',
#     'Tipo de deslocamento: dV (vertical)',
#     'Base de dados: EGMS 2019–2023',
# ))
# ax_map.text(0.99, 0.01, textstr, transform=ax_map.transAxes,
#             fontsize=10, verticalalignment='bottom', horizontalalignment='right',
#             bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.7))

# --- Título e formatação
ax_map.set_title(
    f"Clusters de séries temporais de deslocamento vertical.\n"
    f"K-Means, K={k}.\n"
    f"Grelha {grid_size} m × {grid_size} m.\n"
    f"Correlação entre a série temporal média de cada cluster e a precipitação total.",
    fontsize=16
)
ax_map.set_axis_off()
ax_map.legend(fontsize=10, loc='upper left')

# ==============================
# Linha 2: Séries temporais (ajustada para precipitação)
# ==============================
dV_min = agg['dV'].min()
dV_max = agg['dV'].max()
dV_margin = (dV_max - dV_min) * 0.1

for idx, cluster_id in enumerate(clusters_present):
    ax = fig.add_subplot(gs[1, idx])
    
    cluster_cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]

    # --- Série dV ---
    for cid in cluster_cells:
        ax.plot(cluster_data.columns, cluster_data.loc[cid], color='lightgray', alpha=0.7)
    
    cluster_mean_dV = cluster_data.mean(axis=0)
    cluster_color = cluster_colors[cluster_id]
    ax.plot(cluster_data.columns, cluster_mean_dV, color=cluster_color, linewidth=2.5,
            label=f'Média Cluster {cluster_id + 1}')

    # --- Precipitação total como barras ---
    ax2 = ax.twinx()
    ax2.bar(df_prec['data'], df_prec['prec'], width=20, color='blue', alpha=0.3, label='Precipitação (mm)')
    ax2.set_ylabel("Precipitação (mm)", color='blue')
    ax2.tick_params(axis='y', labelcolor='blue')

    # --- Estética ---
    ax.set_ylim(dV_min - dV_margin, dV_max + dV_margin)
    ax2.set_ylim(0, df_prec['prec'].max() * 1.1)

    ax.set_title(f'Cluster {cluster_id + 1} - {len(cluster_cells)} células', fontsize=12)
    ax.set_ylabel('dV (mm)')
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

    ax.grid(False)
    ax2.grid(False)

    # --- Legenda combinada ---
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, fontsize=9, loc='upper left')

plt.tight_layout()
plt.show()

### K=5

In [ ]:
# ==============================
# 0. Bibliotecas
# ==============================
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
from scipy.signal import savgol_filter
from sklearn.cluster import KMeans

# ==============================
# 1. Ler CSVs ASC, DESC e ORTHO
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"
ortho_v_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"
ortho_h_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_E_2019_2023_1/EGMS_L3_E27N18_100km_E_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)
ortho_v = pd.read_csv(ortho_v_file)
ortho_h = pd.read_csv(ortho_h_file)

# ==============================
# 2. Filtrar área de interesse
# ==============================
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

def filter_area(df):
    return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
              (df['easting'] >= este_min) & (df['easting'] <= este_max)]

asc = filter_area(asc)
desc = filter_area(desc)
ortho_v = filter_area(ortho_v)
ortho_h = filter_area(ortho_h)

# ==============================
# 3. Função melt para ASC/DESC
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]  # ajustar conforme colunas de datas
    long_df = df.melt(
        id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'],
        value_vars=disp_cols, var_name='date', value_name='disp'
    )
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)

common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(), desc_long['date'].max()),
    freq='MS'
)

# ==============================
# 4. Interpolação temporal linear
# ==============================
def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(
            pd.to_datetime(common_dates).astype(np.int64),
            group['date'].astype(np.int64),
            group['disp']
        )
        dfs.append(pd.DataFrame({
            'easting': x,
            'northing': y,
            'latitude': group['latitude'].iloc[0],
            'longitude': group['longitude'].iloc[0],
            'date': common_dates,
            'disp': interp,
            'incidence_angle': group['incidence_angle'].iloc[0],
            'track_angle': group['track_angle'].iloc[0]
        }))
    return pd.concat(dfs, ignore_index=True)

asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 5. Interpolação espacial IDW
# ==============================
def idw_interpolation_per_date(source_df, target_df, radius=150, power=2):
    out_list = []
    for date, src_group in source_df.groupby('date'):
        trg_group = target_df[target_df['date']==date].copy()
        if src_group.empty or trg_group.empty:
            continue
        src_points = np.array(list(zip(src_group['easting'], src_group['northing'])))
        trg_points = np.array(list(zip(trg_group['easting'], trg_group['northing'])))
        tree = cKDTree(src_points)
        dists, idxs = tree.query(trg_points, k=5, distance_upper_bound=radius)

        interpolated_disp = []
        interpolated_theta = []
        interpolated_alpha = []
        for dist, idx in zip(dists, idxs):
            mask = np.isfinite(dist)
            if not np.any(mask):
                interpolated_disp.append(np.nan)
                interpolated_theta.append(np.nan)
                interpolated_alpha.append(np.nan)
                continue
            weights = 1 / (dist[mask] ** power)
            interpolated_disp.append(np.sum(weights * src_group.iloc[idx[mask]]['disp']) / np.sum(weights))
            interpolated_theta.append(np.sum(weights * src_group.iloc[idx[mask]]['incidence_angle']) / np.sum(weights))
            interpolated_alpha.append(np.sum(weights * src_group.iloc[idx[mask]]['track_angle']) / np.sum(weights))

        trg_group['disp_idw'] = interpolated_disp
        trg_group['theta_desc'] = interpolated_theta
        trg_group['alpha_desc'] = interpolated_alpha
        out_list.append(trg_group)
    return pd.concat(out_list, ignore_index=True)

asc_interp = idw_interpolation_per_date(desc_interp, asc_interp)

# ==============================
# 6. Calcular β e γ
# ==============================
orbit_inclination = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orbit_inclination) * np.cos(np.deg2rad(asc_interp['latitude'])))
asc_interp['gamma'] = 0

# ==============================
# 7. Calcular dV e dH
# ==============================
def compute_dV_dH_real(row):
    θA = np.deg2rad(row['incidence_angle'])
    θD = np.deg2rad(row['theta_desc'])
    αA = np.deg2rad(row['track_angle'])
    αD = np.deg2rad(row['alpha_desc'])
    β = row['beta']
    γ = row['gamma']

    dASC_LOS = row['disp']
    dDESC_LOS = row['disp_idw']

    denom = (np.cos(θA)*np.sin(θD)*np.cos(β + γ) +
             np.cos(θD)*np.sin(θA)*np.cos(β - γ))

    dV = (dDESC_LOS*np.sin(θA)*np.cos(β - γ) +
          dASC_LOS*np.sin(θD)*np.cos(β + γ)) / denom
    dH = (dDESC_LOS*np.cos(θA) - dASC_LOS*np.cos(θD)) / denom
    return pd.Series({'dV': dV, 'dH': dH})

asc_interp[['dV','dH']] = asc_interp.apply(compute_dV_dH_real, axis=1)

# ==============================
# 8. Criar grelha centrada ORTHO
# ==============================
grid_size = 100
x_edges = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

asc_interp['cell_x'] = pd.cut(asc_interp['easting'], bins=x_edges_shifted, labels=False)
asc_interp['cell_y'] = pd.cut(asc_interp['northing'], bins=y_edges_shifted, labels=False)

# Remover NaNs antes de criar cell_id
asc_interp = asc_interp.dropna(subset=['cell_x','cell_y'])
asc_interp['cell_id'] = asc_interp['cell_x'].astype(int).astype(str) + "_" + asc_interp['cell_y'].astype(int).astype(str)

# Agrupar para ponto central de cada célula
points = asc_interp.groupby('cell_id').agg({'easting':'mean','northing':'mean'}).reset_index()
gdf_points = gpd.GeoDataFrame(
    points,
    geometry=gpd.points_from_xy(points['easting'], points['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

agg = asc_interp.groupby(['cell_x','cell_y','date']).agg(
    x_center=('easting','mean'),
    y_center=('northing','mean'),
    dV=('dV','mean'),
    dH=('dH','mean')
).reset_index()
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + "_" + agg['cell_y'].astype(int).astype(str)

# ==============================
# 9. Criar GeoDataFrame da grelha
# ==============================
grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({
            "cell_x": ix,
            "cell_y": iy,
            "cell_id": f"{ix}_{iy}",
            "geometry": box(x_edges_shifted[ix], y_edges_shifted[iy],
                            x_edges_shifted[ix+1], y_edges_shifted[iy+1])
        })
grid = gpd.GeoDataFrame(grid_data, crs="EPSG:3035").to_crs(epsg=3857)

# ==============================
# 10. Carregar precipitação
# ==============================
df_prec = pd.read_excel("data/prec.xlsx")
df_prec['data'] = pd.to_datetime(df_prec['data'])
# (sem suavização)

# ==============================
# 11. Clustering de dV
# ==============================
agg_pivot = agg.pivot(index='cell_id', columns='date', values='dV').fillna(0)
k = 5
kmeans = KMeans(n_clusters=k, random_state=0)
cluster_labels = kmeans.fit_predict(agg_pivot)
cluster_df = pd.DataFrame({'cell_id': agg_pivot.index, 'cluster': cluster_labels})

# Mapear cores
cluster_colors = {i: color for i, color in enumerate(['red','green','blue','orange','purple'])}
grid_sel = grid.merge(cluster_df, on='cell_id', how='left')

import matplotlib.dates as mdates

# ==============================
# 14. Figura única: mapa + clusters
# ==============================
clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)

# Criar figura com 2 linhas: mapa + séries temporais
fig = plt.figure(figsize=(20, 18))
gs = fig.add_gridspec(2, n_clusters, height_ratios=[2, 1.2])

# ------------------------------
# Linha 1: Mapa com legenda técnica
# ------------------------------
ax_map = fig.add_subplot(gs[0, :])
grid.boundary.plot(ax=ax_map, color='lightgray', linewidth=0.5)
grid_sel.boundary.plot(ax=ax_map, color='black', linewidth=1, alpha=0.2)

# --- Células coloridas por cluster
for i, row in grid_sel.iterrows():
    if pd.notna(row['cluster']):
        gpd.GeoSeries([row['geometry']], crs=grid_sel.crs).plot(
            ax=ax_map,
            color=cluster_colors[int(row['cluster'])],
            alpha=0.4
        )

# --- Pontos centrais (ASC/DESC)
gdf_points.plot(ax=ax_map, color='white', edgecolor='black', markersize=40)

# Adicionar item de legenda para os pontos
ax_map.scatter([], [], marker='o', color='white', edgecolor='black', s=120,
               label='Pontos ASC/DESC')

# --- Adicionar basemap
ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)

# --- Legenda dos clusters
for cluster_id in clusters_present:
    color = cluster_colors[cluster_id]
    n_cells = len(cluster_df[cluster_df["cluster"] == cluster_id])
    ax_map.scatter([], [], color=color, alpha=0.6,
                   label=f'Cluster {cluster_id + 1} - {n_cells} células')

# --- Caixa técnica (informações do processamento)
# textstr = '\n'.join((
#     f'Tamanho da grelha: {grid_size} x {grid_size} m',
#     f'Técnica de clustering: K-Means (k = {n_clusters})',
#     'Tipo de deslocamento: dV (vertical)',
#     'Base de dados: EGMS 2019–2023',
# ))
# ax_map.text(0.99, 0.01, textstr, transform=ax_map.transAxes,
#             fontsize=10, verticalalignment='bottom', horizontalalignment='right',
#             bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.7))

# --- Título e formatação
ax_map.set_title(
    f"Clusters de séries temporais de deslocamento vertical.\n"
    f"K-Means, K={k}.\n"
    f"Grelha {grid_size} m × {grid_size} m.\n"
    f"Correlação entre a série temporal média de cada cluster e a precipitação total.",
    fontsize=16
)
ax_map.set_axis_off()
ax_map.legend(fontsize=10, loc='upper left')

# ==============================
# Linha 2: Séries temporais (ajustada para precipitação)
# ==============================
dV_min = agg['dV'].min()
dV_max = agg['dV'].max()
dV_margin = (dV_max - dV_min) * 0.1

for idx, cluster_id in enumerate(clusters_present):
    ax = fig.add_subplot(gs[1, idx])
    
    cluster_cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]

    # --- Série dV ---
    for cid in cluster_cells:
        ax.plot(cluster_data.columns, cluster_data.loc[cid], color='lightgray', alpha=0.7)
    
    cluster_mean_dV = cluster_data.mean(axis=0)
    cluster_color = cluster_colors[cluster_id]
    ax.plot(cluster_data.columns, cluster_mean_dV, color=cluster_color, linewidth=2.5,
            label=f'Média Cluster {cluster_id + 1}')

    # --- Precipitação total como barras ---
    ax2 = ax.twinx()
    ax2.bar(df_prec['data'], df_prec['prec'], width=20, color='blue', alpha=0.3, label='Precipitação (mm)')
    ax2.set_ylabel("Precipitação (mm)", color='blue')
    ax2.tick_params(axis='y', labelcolor='blue')

    # --- Estética ---
    ax.set_ylim(dV_min - dV_margin, dV_max + dV_margin)
    ax2.set_ylim(0, df_prec['prec'].max() * 1.1)

    ax.set_title(f'Cluster {cluster_id + 1} - {len(cluster_cells)} células', fontsize=12)
    ax.set_ylabel('dV (mm)')
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

    ax.grid(False)
    ax2.grid(False)

    # --- Legenda combinada ---
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, fontsize=9, loc='upper left')

plt.tight_layout()
plt.show()

## Precipitação total acumulada

### K=2

In [ ]:
# ==============================
# 0. Bibliotecas
# ==============================
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
from scipy.signal import savgol_filter
from sklearn.cluster import KMeans

# ==============================
# 1. Ler CSVs ASC, DESC e ORTHO
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"
ortho_v_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"
ortho_h_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_E_2019_2023_1/EGMS_L3_E27N18_100km_E_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)
ortho_v = pd.read_csv(ortho_v_file)
ortho_h = pd.read_csv(ortho_h_file)

# ==============================
# 2. Filtrar área de interesse
# ==============================
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

def filter_area(df):
    return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
              (df['easting'] >= este_min) & (df['easting'] <= este_max)]

asc = filter_area(asc)
desc = filter_area(desc)
ortho_v = filter_area(ortho_v)
ortho_h = filter_area(ortho_h)

# ==============================
# 3. Função melt para ASC/DESC
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]  # ajustar conforme colunas de datas
    long_df = df.melt(
        id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'],
        value_vars=disp_cols, var_name='date', value_name='disp'
    )
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)

common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(), desc_long['date'].max()),
    freq='MS'
)

# ==============================
# 4. Interpolação temporal linear
# ==============================
def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(
            pd.to_datetime(common_dates).astype(np.int64),
            group['date'].astype(np.int64),
            group['disp']
        )
        dfs.append(pd.DataFrame({
            'easting': x,
            'northing': y,
            'latitude': group['latitude'].iloc[0],
            'longitude': group['longitude'].iloc[0],
            'date': common_dates,
            'disp': interp,
            'incidence_angle': group['incidence_angle'].iloc[0],
            'track_angle': group['track_angle'].iloc[0]
        }))
    return pd.concat(dfs, ignore_index=True)

asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 5. Interpolação espacial IDW
# ==============================
def idw_interpolation_per_date(source_df, target_df, radius=150, power=2):
    out_list = []
    for date, src_group in source_df.groupby('date'):
        trg_group = target_df[target_df['date']==date].copy()
        if src_group.empty or trg_group.empty:
            continue
        src_points = np.array(list(zip(src_group['easting'], src_group['northing'])))
        trg_points = np.array(list(zip(trg_group['easting'], trg_group['northing'])))
        tree = cKDTree(src_points)
        dists, idxs = tree.query(trg_points, k=5, distance_upper_bound=radius)

        interpolated_disp = []
        interpolated_theta = []
        interpolated_alpha = []
        for dist, idx in zip(dists, idxs):
            mask = np.isfinite(dist)
            if not np.any(mask):
                interpolated_disp.append(np.nan)
                interpolated_theta.append(np.nan)
                interpolated_alpha.append(np.nan)
                continue
            weights = 1 / (dist[mask] ** power)
            interpolated_disp.append(np.sum(weights * src_group.iloc[idx[mask]]['disp']) / np.sum(weights))
            interpolated_theta.append(np.sum(weights * src_group.iloc[idx[mask]]['incidence_angle']) / np.sum(weights))
            interpolated_alpha.append(np.sum(weights * src_group.iloc[idx[mask]]['track_angle']) / np.sum(weights))

        trg_group['disp_idw'] = interpolated_disp
        trg_group['theta_desc'] = interpolated_theta
        trg_group['alpha_desc'] = interpolated_alpha
        out_list.append(trg_group)
    return pd.concat(out_list, ignore_index=True)

asc_interp = idw_interpolation_per_date(desc_interp, asc_interp)

# ==============================
# 6. Calcular β e γ
# ==============================
orbit_inclination = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orbit_inclination) * np.cos(np.deg2rad(asc_interp['latitude'])))
asc_interp['gamma'] = 0

# ==============================
# 7. Calcular dV e dH
# ==============================
def compute_dV_dH_real(row):
    θA = np.deg2rad(row['incidence_angle'])
    θD = np.deg2rad(row['theta_desc'])
    αA = np.deg2rad(row['track_angle'])
    αD = np.deg2rad(row['alpha_desc'])
    β = row['beta']
    γ = row['gamma']

    dASC_LOS = row['disp']
    dDESC_LOS = row['disp_idw']

    denom = (np.cos(θA)*np.sin(θD)*np.cos(β + γ) +
             np.cos(θD)*np.sin(θA)*np.cos(β - γ))

    dV = (dDESC_LOS*np.sin(θA)*np.cos(β - γ) +
          dASC_LOS*np.sin(θD)*np.cos(β + γ)) / denom
    dH = (dDESC_LOS*np.cos(θA) - dASC_LOS*np.cos(θD)) / denom
    return pd.Series({'dV': dV, 'dH': dH})

asc_interp[['dV','dH']] = asc_interp.apply(compute_dV_dH_real, axis=1)

# ==============================
# 8. Criar grelha centrada ORTHO
# ==============================
grid_size = 100
x_edges = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

asc_interp['cell_x'] = pd.cut(asc_interp['easting'], bins=x_edges_shifted, labels=False)
asc_interp['cell_y'] = pd.cut(asc_interp['northing'], bins=y_edges_shifted, labels=False)

# Remover NaNs antes de criar cell_id
asc_interp = asc_interp.dropna(subset=['cell_x','cell_y'])
asc_interp['cell_id'] = asc_interp['cell_x'].astype(int).astype(str) + "_" + asc_interp['cell_y'].astype(int).astype(str)

# Agrupar para ponto central de cada célula
points = asc_interp.groupby('cell_id').agg({'easting':'mean','northing':'mean'}).reset_index()
gdf_points = gpd.GeoDataFrame(
    points,
    geometry=gpd.points_from_xy(points['easting'], points['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

agg = asc_interp.groupby(['cell_x','cell_y','date']).agg(
    x_center=('easting','mean'),
    y_center=('northing','mean'),
    dV=('dV','mean'),
    dH=('dH','mean')
).reset_index()
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + "_" + agg['cell_y'].astype(int).astype(str)

# ==============================
# 9. Criar GeoDataFrame da grelha
# ==============================
grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({
            "cell_x": ix,
            "cell_y": iy,
            "cell_id": f"{ix}_{iy}",
            "geometry": box(x_edges_shifted[ix], y_edges_shifted[iy],
                            x_edges_shifted[ix+1], y_edges_shifted[iy+1])
        })
grid = gpd.GeoDataFrame(grid_data, crs="EPSG:3035").to_crs(epsg=3857)

# ==============================
# 10. Carregar precipitação
# ==============================
df_prec = pd.read_excel("data/prec.xlsx")
df_prec['data'] = pd.to_datetime(df_prec['data'])
# (sem suavização)

# ==============================
# 11. Clustering de dV
# ==============================
agg_pivot = agg.pivot(index='cell_id', columns='date', values='dV').fillna(0)
k = 2
kmeans = KMeans(n_clusters=k, random_state=0)
cluster_labels = kmeans.fit_predict(agg_pivot)
cluster_df = pd.DataFrame({'cell_id': agg_pivot.index, 'cluster': cluster_labels})

# Mapear cores
cluster_colors = {i: color for i, color in enumerate(['red','green','blue','orange','purple'])}
grid_sel = grid.merge(cluster_df, on='cell_id', how='left')

import matplotlib.dates as mdates

# ==============================
# 14. Figura única: mapa + clusters
# ==============================
clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)

# Criar figura com 2 linhas: mapa + séries temporais
fig = plt.figure(figsize=(20, 18))
gs = fig.add_gridspec(2, n_clusters, height_ratios=[2, 1.2])

# ------------------------------
# Linha 1: Mapa com legenda técnica
# ------------------------------
ax_map = fig.add_subplot(gs[0, :])
grid.boundary.plot(ax=ax_map, color='lightgray', linewidth=0.5)
grid_sel.boundary.plot(ax=ax_map, color='black', linewidth=1, alpha=0.2)

# --- Células coloridas por cluster
for i, row in grid_sel.iterrows():
    if pd.notna(row['cluster']):
        gpd.GeoSeries([row['geometry']], crs=grid_sel.crs).plot(
            ax=ax_map,
            color=cluster_colors[int(row['cluster'])],
            alpha=0.4
        )

# --- Pontos centrais (ASC/DESC)
gdf_points.plot(ax=ax_map, color='white', edgecolor='black', markersize=40)

# Adicionar item de legenda para os pontos
ax_map.scatter([], [], marker='o', color='white', edgecolor='black', s=120,
               label='Pontos ASC/DESC')

# --- Adicionar basemap
ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)

# --- Legenda dos clusters
for cluster_id in clusters_present:
    color = cluster_colors[cluster_id]
    n_cells = len(cluster_df[cluster_df["cluster"] == cluster_id])
    ax_map.scatter([], [], color=color, alpha=0.6,
                   label=f'Cluster {cluster_id + 1} - {n_cells} células')

# # --- Caixa técnica (informações do processamento)
# textstr = '\n'.join((
#     f'Tamanho da grelha: {grid_size} x {grid_size} m',
#     f'Técnica de clustering: K-Means (k = {n_clusters})',
#     'Tipo de deslocamento: dV (vertical)',
#     'Base de dados: EGMS 2019–2023',
# ))
# ax_map.text(0.99, 0.01, textstr, transform=ax_map.transAxes,
#             fontsize=10, verticalalignment='bottom', horizontalalignment='right',
#             bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.7))

# --- Título e formatação
ax_map.set_title(
    f"Clusters de séries temporais de deslocamento vertical.\n"
    f"K-Means, K={k}.\n"
    f"Grelha {grid_size} m × {grid_size} m.\n"
    f"Correlação entre a série temporal média de cada cluster e a precipitação total acumulada.",
    fontsize=16
)
ax_map.set_axis_off()
ax_map.legend(fontsize=10, loc='upper left')

# --- Linha 2: Séries temporais ---
dV_min = agg['dV'].min()
dV_max = agg['dV'].max()
dV_margin = (dV_max - dV_min) * 0.1

# Calcular precipitação acumulada
df_prec['prec_acum'] = df_prec['prec'].cumsum()

for idx, cluster_id in enumerate(clusters_present):
    ax = fig.add_subplot(gs[1, idx])
    
    # --- dV ---
    cluster_cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]

    for cid in cluster_cells:
        ax.plot(cluster_data.columns, cluster_data.loc[cid], color='lightgray', alpha=0.7)

    cluster_mean_dV = cluster_data.mean(axis=0)
    cluster_color = cluster_colors[cluster_id]
    ax.plot(cluster_data.columns, cluster_mean_dV, color=cluster_color, linewidth=2.5,
            label=f'Média Cluster {cluster_id + 1}')

    # --- Precipitação: barras + linha acumulada ---
    ax2 = ax.twinx()

    # Barras mensais
    ax2.bar(df_prec['data'], df_prec['prec'],
            width=20, color='royalblue', alpha=0.35, label='Precipitação mensal (mm)')

    # Linha de precipitação acumulada
    ax2.plot(df_prec['data'], df_prec['prec_acum'],
             color='navy', linewidth=2.2, label='Precipitação acumulada (mm)')

    # Eixos
    ax2.set_ylabel("Precipitação (mm)", color='navy')
    ax2.tick_params(axis='y', labelcolor='navy')

    # --- Limites ---
    ax.set_ylim(dV_min - dV_margin, dV_max + dV_margin)
    ax2.set_ylim(0, df_prec['prec_acum'].max() * 1.1)

    # --- Estilo ---
    ax.set_title(f'Cluster {cluster_id + 1} - {len(cluster_cells)} células', fontsize=12)
    ax.set_ylabel('dV (mm)')
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.grid(False)
    ax2.grid(False)

    # --- Legenda combinada ---
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, fontsize=9, loc='upper left')

plt.tight_layout()
plt.show()

### K=3

In [ ]:
# ==============================
# 0. Bibliotecas
# ==============================
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
from scipy.signal import savgol_filter
from sklearn.cluster import KMeans

# ==============================
# 1. Ler CSVs ASC, DESC e ORTHO
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"
ortho_v_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"
ortho_h_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_E_2019_2023_1/EGMS_L3_E27N18_100km_E_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)
ortho_v = pd.read_csv(ortho_v_file)
ortho_h = pd.read_csv(ortho_h_file)

# ==============================
# 2. Filtrar área de interesse
# ==============================
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

def filter_area(df):
    return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
              (df['easting'] >= este_min) & (df['easting'] <= este_max)]

asc = filter_area(asc)
desc = filter_area(desc)
ortho_v = filter_area(ortho_v)
ortho_h = filter_area(ortho_h)

# ==============================
# 3. Função melt para ASC/DESC
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]  # ajustar conforme colunas de datas
    long_df = df.melt(
        id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'],
        value_vars=disp_cols, var_name='date', value_name='disp'
    )
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)

common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(), desc_long['date'].max()),
    freq='MS'
)

# ==============================
# 4. Interpolação temporal linear
# ==============================
def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(
            pd.to_datetime(common_dates).astype(np.int64),
            group['date'].astype(np.int64),
            group['disp']
        )
        dfs.append(pd.DataFrame({
            'easting': x,
            'northing': y,
            'latitude': group['latitude'].iloc[0],
            'longitude': group['longitude'].iloc[0],
            'date': common_dates,
            'disp': interp,
            'incidence_angle': group['incidence_angle'].iloc[0],
            'track_angle': group['track_angle'].iloc[0]
        }))
    return pd.concat(dfs, ignore_index=True)

asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 5. Interpolação espacial IDW
# ==============================
def idw_interpolation_per_date(source_df, target_df, radius=150, power=2):
    out_list = []
    for date, src_group in source_df.groupby('date'):
        trg_group = target_df[target_df['date']==date].copy()
        if src_group.empty or trg_group.empty:
            continue
        src_points = np.array(list(zip(src_group['easting'], src_group['northing'])))
        trg_points = np.array(list(zip(trg_group['easting'], trg_group['northing'])))
        tree = cKDTree(src_points)
        dists, idxs = tree.query(trg_points, k=5, distance_upper_bound=radius)

        interpolated_disp = []
        interpolated_theta = []
        interpolated_alpha = []
        for dist, idx in zip(dists, idxs):
            mask = np.isfinite(dist)
            if not np.any(mask):
                interpolated_disp.append(np.nan)
                interpolated_theta.append(np.nan)
                interpolated_alpha.append(np.nan)
                continue
            weights = 1 / (dist[mask] ** power)
            interpolated_disp.append(np.sum(weights * src_group.iloc[idx[mask]]['disp']) / np.sum(weights))
            interpolated_theta.append(np.sum(weights * src_group.iloc[idx[mask]]['incidence_angle']) / np.sum(weights))
            interpolated_alpha.append(np.sum(weights * src_group.iloc[idx[mask]]['track_angle']) / np.sum(weights))

        trg_group['disp_idw'] = interpolated_disp
        trg_group['theta_desc'] = interpolated_theta
        trg_group['alpha_desc'] = interpolated_alpha
        out_list.append(trg_group)
    return pd.concat(out_list, ignore_index=True)

asc_interp = idw_interpolation_per_date(desc_interp, asc_interp)

# ==============================
# 6. Calcular β e γ
# ==============================
orbit_inclination = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orbit_inclination) * np.cos(np.deg2rad(asc_interp['latitude'])))
asc_interp['gamma'] = 0

# ==============================
# 7. Calcular dV e dH
# ==============================
def compute_dV_dH_real(row):
    θA = np.deg2rad(row['incidence_angle'])
    θD = np.deg2rad(row['theta_desc'])
    αA = np.deg2rad(row['track_angle'])
    αD = np.deg2rad(row['alpha_desc'])
    β = row['beta']
    γ = row['gamma']

    dASC_LOS = row['disp']
    dDESC_LOS = row['disp_idw']

    denom = (np.cos(θA)*np.sin(θD)*np.cos(β + γ) +
             np.cos(θD)*np.sin(θA)*np.cos(β - γ))

    dV = (dDESC_LOS*np.sin(θA)*np.cos(β - γ) +
          dASC_LOS*np.sin(θD)*np.cos(β + γ)) / denom
    dH = (dDESC_LOS*np.cos(θA) - dASC_LOS*np.cos(θD)) / denom
    return pd.Series({'dV': dV, 'dH': dH})

asc_interp[['dV','dH']] = asc_interp.apply(compute_dV_dH_real, axis=1)

# ==============================
# 8. Criar grelha centrada ORTHO
# ==============================
grid_size = 100
x_edges = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

asc_interp['cell_x'] = pd.cut(asc_interp['easting'], bins=x_edges_shifted, labels=False)
asc_interp['cell_y'] = pd.cut(asc_interp['northing'], bins=y_edges_shifted, labels=False)

# Remover NaNs antes de criar cell_id
asc_interp = asc_interp.dropna(subset=['cell_x','cell_y'])
asc_interp['cell_id'] = asc_interp['cell_x'].astype(int).astype(str) + "_" + asc_interp['cell_y'].astype(int).astype(str)

# Agrupar para ponto central de cada célula
points = asc_interp.groupby('cell_id').agg({'easting':'mean','northing':'mean'}).reset_index()
gdf_points = gpd.GeoDataFrame(
    points,
    geometry=gpd.points_from_xy(points['easting'], points['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

agg = asc_interp.groupby(['cell_x','cell_y','date']).agg(
    x_center=('easting','mean'),
    y_center=('northing','mean'),
    dV=('dV','mean'),
    dH=('dH','mean')
).reset_index()
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + "_" + agg['cell_y'].astype(int).astype(str)

# ==============================
# 9. Criar GeoDataFrame da grelha
# ==============================
grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({
            "cell_x": ix,
            "cell_y": iy,
            "cell_id": f"{ix}_{iy}",
            "geometry": box(x_edges_shifted[ix], y_edges_shifted[iy],
                            x_edges_shifted[ix+1], y_edges_shifted[iy+1])
        })
grid = gpd.GeoDataFrame(grid_data, crs="EPSG:3035").to_crs(epsg=3857)

# ==============================
# 10. Carregar precipitação
# ==============================
df_prec = pd.read_excel("data/prec.xlsx")
df_prec['data'] = pd.to_datetime(df_prec['data'])
# (sem suavização)

# ==============================
# 11. Clustering de dV
# ==============================
agg_pivot = agg.pivot(index='cell_id', columns='date', values='dV').fillna(0)
k = 3
kmeans = KMeans(n_clusters=k, random_state=0)
cluster_labels = kmeans.fit_predict(agg_pivot)
cluster_df = pd.DataFrame({'cell_id': agg_pivot.index, 'cluster': cluster_labels})

# Mapear cores
cluster_colors = {i: color for i, color in enumerate(['red','green','blue','orange','purple'])}
grid_sel = grid.merge(cluster_df, on='cell_id', how='left')

import matplotlib.dates as mdates

# ==============================
# 14. Figura única: mapa + clusters
# ==============================
clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)

# Criar figura com 2 linhas: mapa + séries temporais
fig = plt.figure(figsize=(20, 18))
gs = fig.add_gridspec(2, n_clusters, height_ratios=[2, 1.2])

# ------------------------------
# Linha 1: Mapa com legenda técnica
# ------------------------------
ax_map = fig.add_subplot(gs[0, :])
grid.boundary.plot(ax=ax_map, color='lightgray', linewidth=0.5)
grid_sel.boundary.plot(ax=ax_map, color='black', linewidth=1, alpha=0.2)

# --- Células coloridas por cluster
for i, row in grid_sel.iterrows():
    if pd.notna(row['cluster']):
        gpd.GeoSeries([row['geometry']], crs=grid_sel.crs).plot(
            ax=ax_map,
            color=cluster_colors[int(row['cluster'])],
            alpha=0.4
        )

# --- Pontos centrais (ASC/DESC)
gdf_points.plot(ax=ax_map, color='white', edgecolor='black', markersize=40)

# Adicionar item de legenda para os pontos
ax_map.scatter([], [], marker='o', color='white', edgecolor='black', s=120,
               label='Pontos ASC/DESC')

# --- Adicionar basemap
ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)

# --- Legenda dos clusters
for cluster_id in clusters_present:
    color = cluster_colors[cluster_id]
    n_cells = len(cluster_df[cluster_df["cluster"] == cluster_id])
    ax_map.scatter([], [], color=color, alpha=0.6,
                   label=f'Cluster {cluster_id + 1} - {n_cells} células')

# # --- Caixa técnica (informações do processamento)
# textstr = '\n'.join((
#     f'Tamanho da grelha: {grid_size} x {grid_size} m',
#     f'Técnica de clustering: K-Means (k = {n_clusters})',
#     'Tipo de deslocamento: dV (vertical)',
#     'Base de dados: EGMS 2019–2023',
# ))
# ax_map.text(0.99, 0.01, textstr, transform=ax_map.transAxes,
#             fontsize=10, verticalalignment='bottom', horizontalalignment='right',
#             bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.7))

# --- Título e formatação
ax_map.set_title(
    f"Clusters de séries temporais de deslocamento vertical.\n"
    f"K-Means, K={k}.\n"
    f"Grelha {grid_size} m × {grid_size} m.\n"
    f"Correlação entre a série temporal média de cada cluster e a precipitação total acumulada.",
    fontsize=16
)
ax_map.set_axis_off()
ax_map.legend(fontsize=10, loc='upper left')

# --- Linha 2: Séries temporais ---
dV_min = agg['dV'].min()
dV_max = agg['dV'].max()
dV_margin = (dV_max - dV_min) * 0.1

# Calcular precipitação acumulada
df_prec['prec_acum'] = df_prec['prec'].cumsum()

for idx, cluster_id in enumerate(clusters_present):
    ax = fig.add_subplot(gs[1, idx])
    
    # --- dV ---
    cluster_cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]

    for cid in cluster_cells:
        ax.plot(cluster_data.columns, cluster_data.loc[cid], color='lightgray', alpha=0.7)

    cluster_mean_dV = cluster_data.mean(axis=0)
    cluster_color = cluster_colors[cluster_id]
    ax.plot(cluster_data.columns, cluster_mean_dV, color=cluster_color, linewidth=2.5,
            label=f'Média Cluster {cluster_id + 1}')

    # --- Precipitação: barras + linha acumulada ---
    ax2 = ax.twinx()

    # Barras mensais
    ax2.bar(df_prec['data'], df_prec['prec'],
            width=20, color='royalblue', alpha=0.35, label='Precipitação mensal (mm)')

    # Linha de precipitação acumulada
    ax2.plot(df_prec['data'], df_prec['prec_acum'],
             color='navy', linewidth=2.2, label='Precipitação acumulada (mm)')

    # Eixos
    ax2.set_ylabel("Precipitação (mm)", color='navy')
    ax2.tick_params(axis='y', labelcolor='navy')

    # --- Limites ---
    ax.set_ylim(dV_min - dV_margin, dV_max + dV_margin)
    ax2.set_ylim(0, df_prec['prec_acum'].max() * 1.1)

    # --- Estilo ---
    ax.set_title(f'Cluster {cluster_id + 1} - {len(cluster_cells)} células', fontsize=12)
    ax.set_ylabel('dV (mm)')
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.grid(False)
    ax2.grid(False)

    # --- Legenda combinada ---
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, fontsize=9, loc='upper left')

plt.tight_layout()
plt.show()

In [ ]:
# ==============================
# 0. Bibliotecas
# ==============================
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
from scipy.signal import savgol_filter
from sklearn.cluster import KMeans

# ==============================
# 1. Ler CSVs ASC, DESC e ORTHO
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"
ortho_v_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"
ortho_h_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_E_2019_2023_1/EGMS_L3_E27N18_100km_E_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)
ortho_v = pd.read_csv(ortho_v_file)
ortho_h = pd.read_csv(ortho_h_file)

# ==============================
# 2. Filtrar área de interesse
# ==============================
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

def filter_area(df):
    return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
              (df['easting'] >= este_min) & (df['easting'] <= este_max)]

asc = filter_area(asc)
desc = filter_area(desc)
ortho_v = filter_area(ortho_v)
ortho_h = filter_area(ortho_h)

# ==============================
# 3. Função melt para ASC/DESC
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]  # ajustar conforme colunas de datas
    long_df = df.melt(
        id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'],
        value_vars=disp_cols, var_name='date', value_name='disp'
    )
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)

common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(), desc_long['date'].max()),
    freq='MS'
)

# ==============================
# 4. Interpolação temporal linear
# ==============================
def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(
            pd.to_datetime(common_dates).astype(np.int64),
            group['date'].astype(np.int64),
            group['disp']
        )
        dfs.append(pd.DataFrame({
            'easting': x,
            'northing': y,
            'latitude': group['latitude'].iloc[0],
            'longitude': group['longitude'].iloc[0],
            'date': common_dates,
            'disp': interp,
            'incidence_angle': group['incidence_angle'].iloc[0],
            'track_angle': group['track_angle'].iloc[0]
        }))
    return pd.concat(dfs, ignore_index=True)

asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 5. Interpolação espacial IDW
# ==============================
def idw_interpolation_per_date(source_df, target_df, radius=150, power=2):
    out_list = []
    for date, src_group in source_df.groupby('date'):
        trg_group = target_df[target_df['date']==date].copy()
        if src_group.empty or trg_group.empty:
            continue
        src_points = np.array(list(zip(src_group['easting'], src_group['northing'])))
        trg_points = np.array(list(zip(trg_group['easting'], trg_group['northing'])))
        tree = cKDTree(src_points)
        dists, idxs = tree.query(trg_points, k=5, distance_upper_bound=radius)

        interpolated_disp = []
        interpolated_theta = []
        interpolated_alpha = []
        for dist, idx in zip(dists, idxs):
            mask = np.isfinite(dist)
            if not np.any(mask):
                interpolated_disp.append(np.nan)
                interpolated_theta.append(np.nan)
                interpolated_alpha.append(np.nan)
                continue
            weights = 1 / (dist[mask] ** power)
            interpolated_disp.append(np.sum(weights * src_group.iloc[idx[mask]]['disp']) / np.sum(weights))
            interpolated_theta.append(np.sum(weights * src_group.iloc[idx[mask]]['incidence_angle']) / np.sum(weights))
            interpolated_alpha.append(np.sum(weights * src_group.iloc[idx[mask]]['track_angle']) / np.sum(weights))

        trg_group['disp_idw'] = interpolated_disp
        trg_group['theta_desc'] = interpolated_theta
        trg_group['alpha_desc'] = interpolated_alpha
        out_list.append(trg_group)
    return pd.concat(out_list, ignore_index=True)

asc_interp = idw_interpolation_per_date(desc_interp, asc_interp)

# ==============================
# 6. Calcular β e γ
# ==============================
orbit_inclination = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orbit_inclination) * np.cos(np.deg2rad(asc_interp['latitude'])))
asc_interp['gamma'] = 0

# ==============================
# 7. Calcular dV e dH
# ==============================
def compute_dV_dH_real(row):
    θA = np.deg2rad(row['incidence_angle'])
    θD = np.deg2rad(row['theta_desc'])
    αA = np.deg2rad(row['track_angle'])
    αD = np.deg2rad(row['alpha_desc'])
    β = row['beta']
    γ = row['gamma']

    dASC_LOS = row['disp']
    dDESC_LOS = row['disp_idw']

    denom = (np.cos(θA)*np.sin(θD)*np.cos(β + γ) +
             np.cos(θD)*np.sin(θA)*np.cos(β - γ))

    dV = (dDESC_LOS*np.sin(θA)*np.cos(β - γ) +
          dASC_LOS*np.sin(θD)*np.cos(β + γ)) / denom
    dH = (dDESC_LOS*np.cos(θA) - dASC_LOS*np.cos(θD)) / denom
    return pd.Series({'dV': dV, 'dH': dH})

asc_interp[['dV','dH']] = asc_interp.apply(compute_dV_dH_real, axis=1)

# ==============================
# 8. Criar grelha centrada ORTHO
# ==============================
grid_size = 50
x_edges = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

asc_interp['cell_x'] = pd.cut(asc_interp['easting'], bins=x_edges_shifted, labels=False)
asc_interp['cell_y'] = pd.cut(asc_interp['northing'], bins=y_edges_shifted, labels=False)

# Remover NaNs antes de criar cell_id
asc_interp = asc_interp.dropna(subset=['cell_x','cell_y'])
asc_interp['cell_id'] = asc_interp['cell_x'].astype(int).astype(str) + "_" + asc_interp['cell_y'].astype(int).astype(str)

# Agrupar para ponto central de cada célula
points = asc_interp.groupby('cell_id').agg({'easting':'mean','northing':'mean'}).reset_index()
gdf_points = gpd.GeoDataFrame(
    points,
    geometry=gpd.points_from_xy(points['easting'], points['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

agg = asc_interp.groupby(['cell_x','cell_y','date']).agg(
    x_center=('easting','mean'),
    y_center=('northing','mean'),
    dV=('dV','mean'),
    dH=('dH','mean')
).reset_index()
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + "_" + agg['cell_y'].astype(int).astype(str)

# ==============================
# 9. Criar GeoDataFrame da grelha
# ==============================
grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({
            "cell_x": ix,
            "cell_y": iy,
            "cell_id": f"{ix}_{iy}",
            "geometry": box(x_edges_shifted[ix], y_edges_shifted[iy],
                            x_edges_shifted[ix+1], y_edges_shifted[iy+1])
        })
grid = gpd.GeoDataFrame(grid_data, crs="EPSG:3035").to_crs(epsg=3857)

# ==============================
# 10. Carregar precipitação
# ==============================
df_prec = pd.read_excel("data/prec.xlsx")
df_prec['data'] = pd.to_datetime(df_prec['data'])
# (sem suavização)

# ==============================
# 11. Clustering de dV
# ==============================
agg_pivot = agg.pivot(index='cell_id', columns='date', values='dV').fillna(0)
k = 3
kmeans = KMeans(n_clusters=k, random_state=0)
cluster_labels = kmeans.fit_predict(agg_pivot)
cluster_df = pd.DataFrame({'cell_id': agg_pivot.index, 'cluster': cluster_labels})

# Mapear cores
cluster_colors = {i: color for i, color in enumerate(['red','green','blue','orange','purple'])}
grid_sel = grid.merge(cluster_df, on='cell_id', how='left')

import matplotlib.dates as mdates

# ==============================
# 14. Figura única: mapa + clusters
# ==============================
clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)

# Criar figura com 2 linhas: mapa + séries temporais
fig = plt.figure(figsize=(20, 18))
gs = fig.add_gridspec(2, n_clusters, height_ratios=[2, 1.2])

# ------------------------------
# Linha 1: Mapa
# ------------------------------
ax_map = fig.add_subplot(gs[0, :])
grid.boundary.plot(ax=ax_map, color='lightgray', linewidth=0.5)
grid_sel.boundary.plot(ax=ax_map, color='black', linewidth=1, alpha=0.2)

for i, row in grid_sel.iterrows():
    if pd.notna(row['cluster']):
        gpd.GeoSeries([row['geometry']], crs=grid_sel.crs).plot(
            ax=ax_map,
            color=cluster_colors[int(row['cluster'])],
            alpha=0.4
        )
gdf_points.plot(ax=ax_map, color='white', edgecolor='black', markersize=40, label='Centro de massa ASC/DESC')

for cluster_id in clusters_present:
    color = cluster_colors[cluster_id]
    ax_map.scatter([], [], color=color, alpha=0.4, label=f'Cluster {cluster_id+1}')

ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)
ax_map.set_title("Mapa de Clusters de dV com Centro de Células ASC/DESC", fontsize=16)
ax_map.set_axis_off()
ax_map.legend(fontsize=10)

# --- Linha 2: Séries temporais ---
dV_min = agg['dV'].min()
dV_max = agg['dV'].max()
dV_margin = (dV_max - dV_min) * 0.1

# Calcular precipitação acumulada
df_prec['prec_acum'] = df_prec['prec'].cumsum()

for idx, cluster_id in enumerate(clusters_present):
    ax = fig.add_subplot(gs[1, idx])
    
    # --- dV ---
    cluster_cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]

    for cid in cluster_cells:
        ax.plot(cluster_data.columns, cluster_data.loc[cid], color='lightgray', alpha=0.7)

    cluster_mean_dV = cluster_data.mean(axis=0)
    cluster_color = cluster_colors[cluster_id]
    ax.plot(cluster_data.columns, cluster_mean_dV, color=cluster_color, linewidth=2.5,
            label=f'Média Cluster {cluster_id + 1}')

    # --- Precipitação: barras + linha acumulada ---
    ax2 = ax.twinx()

    # Barras mensais
    ax2.bar(df_prec['data'], df_prec['prec'],
            width=20, color='royalblue', alpha=0.35, label='Precipitação mensal (mm)')

    # Linha de precipitação acumulada
    ax2.plot(df_prec['data'], df_prec['prec_acum'],
             color='navy', linewidth=2.2, label='Precipitação acumulada (mm)')

    # Eixos
    ax2.set_ylabel("Precipitação (mm)", color='navy')
    ax2.tick_params(axis='y', labelcolor='navy')

    # --- Limites ---
    ax.set_ylim(dV_min - dV_margin, dV_max + dV_margin)
    ax2.set_ylim(0, df_prec['prec_acum'].max() * 1.1)

    # --- Estilo ---
    ax.set_title(f'Cluster {cluster_id + 1} - {len(cluster_cells)} células', fontsize=12)
    ax.set_ylabel('dV (mm)')
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.grid(False)
    ax2.grid(False)

    # --- Legenda combinada ---
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, fontsize=9, loc='upper left')

plt.tight_layout()
plt.show()

## Precipitação total anual acumulada

### K=2

In [ ]:
# ==============================
# 0. Bibliotecas
# ==============================
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
from scipy.signal import savgol_filter
from sklearn.cluster import KMeans

# ==============================
# 1. Ler CSVs ASC, DESC e ORTHO
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"
ortho_v_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"
ortho_h_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_E_2019_2023_1/EGMS_L3_E27N18_100km_E_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)
ortho_v = pd.read_csv(ortho_v_file)
ortho_h = pd.read_csv(ortho_h_file)

# ==============================
# 2. Filtrar área de interesse
# ==============================
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

def filter_area(df):
    return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
              (df['easting'] >= este_min) & (df['easting'] <= este_max)]

asc = filter_area(asc)
desc = filter_area(desc)
ortho_v = filter_area(ortho_v)
ortho_h = filter_area(ortho_h)

# ==============================
# 3. Função melt para ASC/DESC
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]  # ajustar conforme colunas de datas
    long_df = df.melt(
        id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'],
        value_vars=disp_cols, var_name='date', value_name='disp'
    )
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)

common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(), desc_long['date'].max()),
    freq='MS'
)

# ==============================
# 4. Interpolação temporal linear
# ==============================
def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(
            pd.to_datetime(common_dates).astype(np.int64),
            group['date'].astype(np.int64),
            group['disp']
        )
        dfs.append(pd.DataFrame({
            'easting': x,
            'northing': y,
            'latitude': group['latitude'].iloc[0],
            'longitude': group['longitude'].iloc[0],
            'date': common_dates,
            'disp': interp,
            'incidence_angle': group['incidence_angle'].iloc[0],
            'track_angle': group['track_angle'].iloc[0]
        }))
    return pd.concat(dfs, ignore_index=True)

asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 5. Interpolação espacial IDW
# ==============================
def idw_interpolation_per_date(source_df, target_df, radius=150, power=2):
    out_list = []
    for date, src_group in source_df.groupby('date'):
        trg_group = target_df[target_df['date']==date].copy()
        if src_group.empty or trg_group.empty:
            continue
        src_points = np.array(list(zip(src_group['easting'], src_group['northing'])))
        trg_points = np.array(list(zip(trg_group['easting'], trg_group['northing'])))
        tree = cKDTree(src_points)
        dists, idxs = tree.query(trg_points, k=5, distance_upper_bound=radius)

        interpolated_disp = []
        interpolated_theta = []
        interpolated_alpha = []
        for dist, idx in zip(dists, idxs):
            mask = np.isfinite(dist)
            if not np.any(mask):
                interpolated_disp.append(np.nan)
                interpolated_theta.append(np.nan)
                interpolated_alpha.append(np.nan)
                continue
            weights = 1 / (dist[mask] ** power)
            interpolated_disp.append(np.sum(weights * src_group.iloc[idx[mask]]['disp']) / np.sum(weights))
            interpolated_theta.append(np.sum(weights * src_group.iloc[idx[mask]]['incidence_angle']) / np.sum(weights))
            interpolated_alpha.append(np.sum(weights * src_group.iloc[idx[mask]]['track_angle']) / np.sum(weights))

        trg_group['disp_idw'] = interpolated_disp
        trg_group['theta_desc'] = interpolated_theta
        trg_group['alpha_desc'] = interpolated_alpha
        out_list.append(trg_group)
    return pd.concat(out_list, ignore_index=True)

asc_interp = idw_interpolation_per_date(desc_interp, asc_interp)

# ==============================
# 6. Calcular β e γ
# ==============================
orbit_inclination = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orbit_inclination) * np.cos(np.deg2rad(asc_interp['latitude'])))
asc_interp['gamma'] = 0

# ==============================
# 7. Calcular dV e dH
# ==============================
def compute_dV_dH_real(row):
    θA = np.deg2rad(row['incidence_angle'])
    θD = np.deg2rad(row['theta_desc'])
    αA = np.deg2rad(row['track_angle'])
    αD = np.deg2rad(row['alpha_desc'])
    β = row['beta']
    γ = row['gamma']

    dASC_LOS = row['disp']
    dDESC_LOS = row['disp_idw']

    denom = (np.cos(θA)*np.sin(θD)*np.cos(β + γ) +
             np.cos(θD)*np.sin(θA)*np.cos(β - γ))

    dV = (dDESC_LOS*np.sin(θA)*np.cos(β - γ) +
          dASC_LOS*np.sin(θD)*np.cos(β + γ)) / denom
    dH = (dDESC_LOS*np.cos(θA) - dASC_LOS*np.cos(θD)) / denom
    return pd.Series({'dV': dV, 'dH': dH})

asc_interp[['dV','dH']] = asc_interp.apply(compute_dV_dH_real, axis=1)

# ==============================
# 8. Criar grelha centrada ORTHO
# ==============================
grid_size = 100
x_edges = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

asc_interp['cell_x'] = pd.cut(asc_interp['easting'], bins=x_edges_shifted, labels=False)
asc_interp['cell_y'] = pd.cut(asc_interp['northing'], bins=y_edges_shifted, labels=False)

# Remover NaNs antes de criar cell_id
asc_interp = asc_interp.dropna(subset=['cell_x','cell_y'])
asc_interp['cell_id'] = asc_interp['cell_x'].astype(int).astype(str) + "_" + asc_interp['cell_y'].astype(int).astype(str)

# Agrupar para ponto central de cada célula
points = asc_interp.groupby('cell_id').agg({'easting':'mean','northing':'mean'}).reset_index()
gdf_points = gpd.GeoDataFrame(
    points,
    geometry=gpd.points_from_xy(points['easting'], points['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

agg = asc_interp.groupby(['cell_x','cell_y','date']).agg(
    x_center=('easting','mean'),
    y_center=('northing','mean'),
    dV=('dV','mean'),
    dH=('dH','mean')
).reset_index()
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + "_" + agg['cell_y'].astype(int).astype(str)

# ==============================
# 9. Criar GeoDataFrame da grelha
# ==============================
grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({
            "cell_x": ix,
            "cell_y": iy,
            "cell_id": f"{ix}_{iy}",
            "geometry": box(x_edges_shifted[ix], y_edges_shifted[iy],
                            x_edges_shifted[ix+1], y_edges_shifted[iy+1])
        })
grid = gpd.GeoDataFrame(grid_data, crs="EPSG:3035").to_crs(epsg=3857)

# ==============================
# 10. Carregar precipitação (acumulado anual)
# ==============================
df_prec = pd.read_excel("data/prec.xlsx")
df_prec['data'] = pd.to_datetime(df_prec['data'])
df_prec['ano_hidrologico'] = df_prec['data'].apply(
    lambda x: x.year if x.month < 10 else x.year + 1
)

# Calcular acumulado dentro de cada ano hidrológico
df_prec['prec_acumulada'] = df_prec.groupby('ano_hidrologico')['prec'].cumsum()

# ==============================
# 11. Clustering de dV
# ==============================
agg_pivot = agg.pivot(index='cell_id', columns='date', values='dV').fillna(0)
k = 2
kmeans = KMeans(n_clusters=k, random_state=0)
cluster_labels = kmeans.fit_predict(agg_pivot)
cluster_df = pd.DataFrame({'cell_id': agg_pivot.index, 'cluster': cluster_labels})

# Mapear cores
cluster_colors = {i: color for i, color in enumerate(['red','green','blue','orange','purple'])}
grid_sel = grid.merge(cluster_df, on='cell_id', how='left')

import matplotlib.dates as mdates

# ==============================
# 14. Figura única: mapa + clusters (com precipitação anual acumulada)
# ==============================
clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)

fig = plt.figure(figsize=(20, 18))
gs = fig.add_gridspec(2, n_clusters, height_ratios=[2, 1.2])

# ------------------------------
# Linha 1: Mapa com legenda técnica
# ------------------------------
ax_map = fig.add_subplot(gs[0, :])
grid.boundary.plot(ax=ax_map, color='lightgray', linewidth=0.5)
grid_sel.boundary.plot(ax=ax_map, color='black', linewidth=1, alpha=0.2)

# --- Células coloridas por cluster
for i, row in grid_sel.iterrows():
    if pd.notna(row['cluster']):
        gpd.GeoSeries([row['geometry']], crs=grid_sel.crs).plot(
            ax=ax_map,
            color=cluster_colors[int(row['cluster'])],
            alpha=0.4
        )

# --- Pontos centrais (ASC/DESC)
gdf_points.plot(ax=ax_map, color='white', edgecolor='black', markersize=40)

# Adicionar item de legenda para os pontos
ax_map.scatter([], [], marker='o', color='white', edgecolor='black', s=120,
               label='Pontos ASC/DESC')

# --- Adicionar basemap
ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)

# --- Legenda dos clusters
for cluster_id in clusters_present:
    color = cluster_colors[cluster_id]
    n_cells = len(cluster_df[cluster_df["cluster"] == cluster_id])
    ax_map.scatter([], [], color=color, alpha=0.6,
                   label=f'Cluster {cluster_id + 1} - {n_cells} células')

# --- Caixa técnica (informações do processamento)
# textstr = '\n'.join((
#     f'Tamanho da grelha: {grid_size} x {grid_size} m',
#     f'Técnica de clustering: K-Means (k = {n_clusters})',
#     'Tipo de deslocamento: dV (vertical)',
#     'Base de dados: EGMS 2019–2023',
# ))
# ax_map.text(0.99, 0.01, textstr, transform=ax_map.transAxes,
#             fontsize=10, verticalalignment='bottom', horizontalalignment='right',
#             bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.7))

# --- Título e formatação
ax_map.set_title(
    f"Clusters de séries temporais de deslocamento vertical.\n"
    f"K-Means, K={k}.\n"
    f"Grelha {grid_size} m × {grid_size} m.\n"
    f"Correlação entre a série temporal média de cada cluster e a precipitação total anual acumulada.",
    fontsize=16
)
ax_map.set_axis_off()
ax_map.legend(fontsize=10, loc='upper left')

# Linha 2: Séries temporais
dV_min = agg['dV'].min()
dV_max = agg['dV'].max()
dV_margin = (dV_max - dV_min) * 0.1

for idx, cluster_id in enumerate(clusters_present):
    ax = fig.add_subplot(gs[1, idx])

    cluster_cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]

    # Linhas cinzentas individuais
    for cid in cluster_cells:
        ax.plot(cluster_data.columns, cluster_data.loc[cid], color='lightgray', alpha=0.7)

    # Média do cluster
    cluster_mean_dV = cluster_data.mean(axis=0)
    cluster_color = cluster_colors[cluster_id]
    ax.plot(cluster_data.columns, cluster_mean_dV, color=cluster_color, linewidth=2.5,
            label=f'Média Cluster {cluster_id + 1}')

    # ---------- Precipitação ----------
    ax2 = ax.twinx()

    # Barras = precipitação mensal
    ax2.bar(df_prec['data'], df_prec['prec'], color='deepskyblue', alpha=0.5,
            width=15, label='Precipitação (mm)')

    # Linha = acumulado anual
    for ano, grupo in df_prec.groupby('ano_hidrologico'):
        ax2.plot(grupo['data'], grupo['prec_acumulada'],
                 color='blue', linewidth=2, alpha=0.9,
                 label=f'Acumulado {ano}')

    # Eixos
    ax.set_ylim(dV_min - dV_margin, dV_max + dV_margin)
    ax2.set_ylim(0, df_prec['prec_acumulada'].max() * 1.1)

    ax.set_title(f'Cluster {cluster_id + 1} - {len(cluster_cells)} células', fontsize=12)
    ax.set_ylabel('dV (mm)')
    ax2.set_ylabel('Precipitação (mm)', color='blue')
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

    ax.grid(False)
    ax2.grid(False)

    # Legenda combinada
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, fontsize=8, loc='upper left')

plt.tight_layout()
plt.show()

### K=3

In [ ]:
# ==============================
# 0. Bibliotecas
# ==============================
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
from scipy.signal import savgol_filter
from sklearn.cluster import KMeans

# ==============================
# 1. Ler CSVs ASC, DESC e ORTHO
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"
ortho_v_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"
ortho_h_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_E_2019_2023_1/EGMS_L3_E27N18_100km_E_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)
ortho_v = pd.read_csv(ortho_v_file)
ortho_h = pd.read_csv(ortho_h_file)

# ==============================
# 2. Filtrar área de interesse
# ==============================
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

def filter_area(df):
    return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
              (df['easting'] >= este_min) & (df['easting'] <= este_max)]

asc = filter_area(asc)
desc = filter_area(desc)
ortho_v = filter_area(ortho_v)
ortho_h = filter_area(ortho_h)

# ==============================
# 3. Função melt para ASC/DESC
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]  # ajustar conforme colunas de datas
    long_df = df.melt(
        id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'],
        value_vars=disp_cols, var_name='date', value_name='disp'
    )
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)

common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(), desc_long['date'].max()),
    freq='MS'
)

# ==============================
# 4. Interpolação temporal linear
# ==============================
def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(
            pd.to_datetime(common_dates).astype(np.int64),
            group['date'].astype(np.int64),
            group['disp']
        )
        dfs.append(pd.DataFrame({
            'easting': x,
            'northing': y,
            'latitude': group['latitude'].iloc[0],
            'longitude': group['longitude'].iloc[0],
            'date': common_dates,
            'disp': interp,
            'incidence_angle': group['incidence_angle'].iloc[0],
            'track_angle': group['track_angle'].iloc[0]
        }))
    return pd.concat(dfs, ignore_index=True)

asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 5. Interpolação espacial IDW
# ==============================
def idw_interpolation_per_date(source_df, target_df, radius=150, power=2):
    out_list = []
    for date, src_group in source_df.groupby('date'):
        trg_group = target_df[target_df['date']==date].copy()
        if src_group.empty or trg_group.empty:
            continue
        src_points = np.array(list(zip(src_group['easting'], src_group['northing'])))
        trg_points = np.array(list(zip(trg_group['easting'], trg_group['northing'])))
        tree = cKDTree(src_points)
        dists, idxs = tree.query(trg_points, k=5, distance_upper_bound=radius)

        interpolated_disp = []
        interpolated_theta = []
        interpolated_alpha = []
        for dist, idx in zip(dists, idxs):
            mask = np.isfinite(dist)
            if not np.any(mask):
                interpolated_disp.append(np.nan)
                interpolated_theta.append(np.nan)
                interpolated_alpha.append(np.nan)
                continue
            weights = 1 / (dist[mask] ** power)
            interpolated_disp.append(np.sum(weights * src_group.iloc[idx[mask]]['disp']) / np.sum(weights))
            interpolated_theta.append(np.sum(weights * src_group.iloc[idx[mask]]['incidence_angle']) / np.sum(weights))
            interpolated_alpha.append(np.sum(weights * src_group.iloc[idx[mask]]['track_angle']) / np.sum(weights))

        trg_group['disp_idw'] = interpolated_disp
        trg_group['theta_desc'] = interpolated_theta
        trg_group['alpha_desc'] = interpolated_alpha
        out_list.append(trg_group)
    return pd.concat(out_list, ignore_index=True)

asc_interp = idw_interpolation_per_date(desc_interp, asc_interp)

# ==============================
# 6. Calcular β e γ
# ==============================
orbit_inclination = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orbit_inclination) * np.cos(np.deg2rad(asc_interp['latitude'])))
asc_interp['gamma'] = 0

# ==============================
# 7. Calcular dV e dH
# ==============================
def compute_dV_dH_real(row):
    θA = np.deg2rad(row['incidence_angle'])
    θD = np.deg2rad(row['theta_desc'])
    αA = np.deg2rad(row['track_angle'])
    αD = np.deg2rad(row['alpha_desc'])
    β = row['beta']
    γ = row['gamma']

    dASC_LOS = row['disp']
    dDESC_LOS = row['disp_idw']

    denom = (np.cos(θA)*np.sin(θD)*np.cos(β + γ) +
             np.cos(θD)*np.sin(θA)*np.cos(β - γ))

    dV = (dDESC_LOS*np.sin(θA)*np.cos(β - γ) +
          dASC_LOS*np.sin(θD)*np.cos(β + γ)) / denom
    dH = (dDESC_LOS*np.cos(θA) - dASC_LOS*np.cos(θD)) / denom
    return pd.Series({'dV': dV, 'dH': dH})

asc_interp[['dV','dH']] = asc_interp.apply(compute_dV_dH_real, axis=1)

# ==============================
# 8. Criar grelha centrada ORTHO
# ==============================
grid_size = 100
x_edges = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

asc_interp['cell_x'] = pd.cut(asc_interp['easting'], bins=x_edges_shifted, labels=False)
asc_interp['cell_y'] = pd.cut(asc_interp['northing'], bins=y_edges_shifted, labels=False)

# Remover NaNs antes de criar cell_id
asc_interp = asc_interp.dropna(subset=['cell_x','cell_y'])
asc_interp['cell_id'] = asc_interp['cell_x'].astype(int).astype(str) + "_" + asc_interp['cell_y'].astype(int).astype(str)

# Agrupar para ponto central de cada célula
points = asc_interp.groupby('cell_id').agg({'easting':'mean','northing':'mean'}).reset_index()
gdf_points = gpd.GeoDataFrame(
    points,
    geometry=gpd.points_from_xy(points['easting'], points['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

agg = asc_interp.groupby(['cell_x','cell_y','date']).agg(
    x_center=('easting','mean'),
    y_center=('northing','mean'),
    dV=('dV','mean'),
    dH=('dH','mean')
).reset_index()
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + "_" + agg['cell_y'].astype(int).astype(str)

# ==============================
# 9. Criar GeoDataFrame da grelha
# ==============================
grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({
            "cell_x": ix,
            "cell_y": iy,
            "cell_id": f"{ix}_{iy}",
            "geometry": box(x_edges_shifted[ix], y_edges_shifted[iy],
                            x_edges_shifted[ix+1], y_edges_shifted[iy+1])
        })
grid = gpd.GeoDataFrame(grid_data, crs="EPSG:3035").to_crs(epsg=3857)

# ==============================
# 10. Carregar precipitação (acumulado anual)
# ==============================
df_prec = pd.read_excel("data/prec.xlsx")
df_prec['data'] = pd.to_datetime(df_prec['data'])
df_prec['ano_hidrologico'] = df_prec['data'].apply(
    lambda x: x.year if x.month < 10 else x.year + 1
)

# Calcular acumulado dentro de cada ano hidrológico
df_prec['prec_acumulada'] = df_prec.groupby('ano_hidrologico')['prec'].cumsum()

# ==============================
# 11. Clustering de dV
# ==============================
agg_pivot = agg.pivot(index='cell_id', columns='date', values='dV').fillna(0)
k = 3
kmeans = KMeans(n_clusters=k, random_state=0)
cluster_labels = kmeans.fit_predict(agg_pivot)
cluster_df = pd.DataFrame({'cell_id': agg_pivot.index, 'cluster': cluster_labels})

# Mapear cores
cluster_colors = {i: color for i, color in enumerate(['red','green','blue','orange','purple'])}
grid_sel = grid.merge(cluster_df, on='cell_id', how='left')

import matplotlib.dates as mdates

# ==============================
# 14. Figura única: mapa + clusters (com precipitação anual acumulada)
# ==============================
clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)

fig = plt.figure(figsize=(20, 18))
gs = fig.add_gridspec(2, n_clusters, height_ratios=[2, 1.2])

# ------------------------------
# Linha 1: Mapa com legenda técnica
# ------------------------------
ax_map = fig.add_subplot(gs[0, :])
grid.boundary.plot(ax=ax_map, color='lightgray', linewidth=0.5)
grid_sel.boundary.plot(ax=ax_map, color='black', linewidth=1, alpha=0.2)

# --- Células coloridas por cluster
for i, row in grid_sel.iterrows():
    if pd.notna(row['cluster']):
        gpd.GeoSeries([row['geometry']], crs=grid_sel.crs).plot(
            ax=ax_map,
            color=cluster_colors[int(row['cluster'])],
            alpha=0.4
        )

# --- Pontos centrais (ASC/DESC)
gdf_points.plot(ax=ax_map, color='white', edgecolor='black', markersize=40)

# Adicionar item de legenda para os pontos
ax_map.scatter([], [], marker='o', color='white', edgecolor='black', s=120,
               label='Pontos ASC/DESC')

# --- Adicionar basemap
ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)

# --- Legenda dos clusters
for cluster_id in clusters_present:
    color = cluster_colors[cluster_id]
    n_cells = len(cluster_df[cluster_df["cluster"] == cluster_id])
    ax_map.scatter([], [], color=color, alpha=0.6,
                   label=f'Cluster {cluster_id + 1} - {n_cells} células')

# # --- Caixa técnica (informações do processamento)
# textstr = '\n'.join((
#     f'Tamanho da grelha: {grid_size} x {grid_size} m',
#     f'Técnica de clustering: K-Means (k = {n_clusters})',
#     'Tipo de deslocamento: dV (vertical)',
#     'Base de dados: EGMS 2019–2023',
# ))
# ax_map.text(0.99, 0.01, textstr, transform=ax_map.transAxes,
#             fontsize=10, verticalalignment='bottom', horizontalalignment='right',
#             bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.7))

# --- Título e formatação
ax_map.set_title(
    f"Clusters de séries temporais de deslocamento vertical.\n"
    f"K-Means, K={k}.\n"
    f"Grelha {grid_size} m × {grid_size} m.\n"
    f"Correlação entre a série temporal média de cada cluster e a precipitação total anual acumulada.",
    fontsize=16
)
ax_map.set_axis_off()
ax_map.legend(fontsize=10, loc='upper left')

# Linha 2: Séries temporais
dV_min = agg['dV'].min()
dV_max = agg['dV'].max()
dV_margin = (dV_max - dV_min) * 0.1

for idx, cluster_id in enumerate(clusters_present):
    ax = fig.add_subplot(gs[1, idx])

    cluster_cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]

    # Linhas cinzentas individuais
    for cid in cluster_cells:
        ax.plot(cluster_data.columns, cluster_data.loc[cid], color='lightgray', alpha=0.7)

    # Média do cluster
    cluster_mean_dV = cluster_data.mean(axis=0)
    cluster_color = cluster_colors[cluster_id]
    ax.plot(cluster_data.columns, cluster_mean_dV, color=cluster_color, linewidth=2.5,
            label=f'Média Cluster {cluster_id + 1}')

    # ---------- Precipitação ----------
    ax2 = ax.twinx()

    # Barras = precipitação mensal
    ax2.bar(df_prec['data'], df_prec['prec'], color='deepskyblue', alpha=0.5,
            width=15, label='Precipitação (mm)')

    # Linha = acumulado anual
    for ano, grupo in df_prec.groupby('ano_hidrologico'):
        ax2.plot(grupo['data'], grupo['prec_acumulada'],
                 color='blue', linewidth=2, alpha=0.9,
                 label=f'Acumulado {ano}')

    # Eixos
    ax.set_ylim(dV_min - dV_margin, dV_max + dV_margin)
    ax2.set_ylim(0, df_prec['prec_acumulada'].max() * 1.1)

    ax.set_title(f'Cluster {cluster_id + 1} - {len(cluster_cells)} células', fontsize=12)
    ax.set_ylabel('dV (mm)')
    ax2.set_ylabel('Precipitação (mm)', color='blue')
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

    ax.grid(False)
    ax2.grid(False)

    # Legenda combinada
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, fontsize=8, loc='upper left')

plt.tight_layout()
plt.show()

In [ ]:
# ==============================
# 0. Bibliotecas
# ==============================
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
from scipy.signal import savgol_filter
from sklearn.cluster import KMeans

# ==============================
# 1. Ler CSVs ASC, DESC e ORTHO
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"
ortho_v_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"
ortho_h_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_E_2019_2023_1/EGMS_L3_E27N18_100km_E_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)
ortho_v = pd.read_csv(ortho_v_file)
ortho_h = pd.read_csv(ortho_h_file)

# ==============================
# 2. Filtrar área de interesse
# ==============================
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

def filter_area(df):
    return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
              (df['easting'] >= este_min) & (df['easting'] <= este_max)]

asc = filter_area(asc)
desc = filter_area(desc)
ortho_v = filter_area(ortho_v)
ortho_h = filter_area(ortho_h)

# ==============================
# 3. Função melt para ASC/DESC
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]  # ajustar conforme colunas de datas
    long_df = df.melt(
        id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'],
        value_vars=disp_cols, var_name='date', value_name='disp'
    )
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)

common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(), desc_long['date'].max()),
    freq='MS'
)

# ==============================
# 4. Interpolação temporal linear
# ==============================
def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(
            pd.to_datetime(common_dates).astype(np.int64),
            group['date'].astype(np.int64),
            group['disp']
        )
        dfs.append(pd.DataFrame({
            'easting': x,
            'northing': y,
            'latitude': group['latitude'].iloc[0],
            'longitude': group['longitude'].iloc[0],
            'date': common_dates,
            'disp': interp,
            'incidence_angle': group['incidence_angle'].iloc[0],
            'track_angle': group['track_angle'].iloc[0]
        }))
    return pd.concat(dfs, ignore_index=True)

asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 5. Interpolação espacial IDW
# ==============================
def idw_interpolation_per_date(source_df, target_df, radius=150, power=2):
    out_list = []
    for date, src_group in source_df.groupby('date'):
        trg_group = target_df[target_df['date']==date].copy()
        if src_group.empty or trg_group.empty:
            continue
        src_points = np.array(list(zip(src_group['easting'], src_group['northing'])))
        trg_points = np.array(list(zip(trg_group['easting'], trg_group['northing'])))
        tree = cKDTree(src_points)
        dists, idxs = tree.query(trg_points, k=5, distance_upper_bound=radius)

        interpolated_disp = []
        interpolated_theta = []
        interpolated_alpha = []
        for dist, idx in zip(dists, idxs):
            mask = np.isfinite(dist)
            if not np.any(mask):
                interpolated_disp.append(np.nan)
                interpolated_theta.append(np.nan)
                interpolated_alpha.append(np.nan)
                continue
            weights = 1 / (dist[mask] ** power)
            interpolated_disp.append(np.sum(weights * src_group.iloc[idx[mask]]['disp']) / np.sum(weights))
            interpolated_theta.append(np.sum(weights * src_group.iloc[idx[mask]]['incidence_angle']) / np.sum(weights))
            interpolated_alpha.append(np.sum(weights * src_group.iloc[idx[mask]]['track_angle']) / np.sum(weights))

        trg_group['disp_idw'] = interpolated_disp
        trg_group['theta_desc'] = interpolated_theta
        trg_group['alpha_desc'] = interpolated_alpha
        out_list.append(trg_group)
    return pd.concat(out_list, ignore_index=True)

asc_interp = idw_interpolation_per_date(desc_interp, asc_interp)

# ==============================
# 6. Calcular β e γ
# ==============================
orbit_inclination = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orbit_inclination) * np.cos(np.deg2rad(asc_interp['latitude'])))
asc_interp['gamma'] = 0

# ==============================
# 7. Calcular dV e dH
# ==============================
def compute_dV_dH_real(row):
    θA = np.deg2rad(row['incidence_angle'])
    θD = np.deg2rad(row['theta_desc'])
    αA = np.deg2rad(row['track_angle'])
    αD = np.deg2rad(row['alpha_desc'])
    β = row['beta']
    γ = row['gamma']

    dASC_LOS = row['disp']
    dDESC_LOS = row['disp_idw']

    denom = (np.cos(θA)*np.sin(θD)*np.cos(β + γ) +
             np.cos(θD)*np.sin(θA)*np.cos(β - γ))

    dV = (dDESC_LOS*np.sin(θA)*np.cos(β - γ) +
          dASC_LOS*np.sin(θD)*np.cos(β + γ)) / denom
    dH = (dDESC_LOS*np.cos(θA) - dASC_LOS*np.cos(θD)) / denom
    return pd.Series({'dV': dV, 'dH': dH})

asc_interp[['dV','dH']] = asc_interp.apply(compute_dV_dH_real, axis=1)

# ==============================
# 8. Criar grelha centrada ORTHO
# ==============================
grid_size = 50
x_edges = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

asc_interp['cell_x'] = pd.cut(asc_interp['easting'], bins=x_edges_shifted, labels=False)
asc_interp['cell_y'] = pd.cut(asc_interp['northing'], bins=y_edges_shifted, labels=False)

# Remover NaNs antes de criar cell_id
asc_interp = asc_interp.dropna(subset=['cell_x','cell_y'])
asc_interp['cell_id'] = asc_interp['cell_x'].astype(int).astype(str) + "_" + asc_interp['cell_y'].astype(int).astype(str)

# Agrupar para ponto central de cada célula
points = asc_interp.groupby('cell_id').agg({'easting':'mean','northing':'mean'}).reset_index()
gdf_points = gpd.GeoDataFrame(
    points,
    geometry=gpd.points_from_xy(points['easting'], points['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

agg = asc_interp.groupby(['cell_x','cell_y','date']).agg(
    x_center=('easting','mean'),
    y_center=('northing','mean'),
    dV=('dV','mean'),
    dH=('dH','mean')
).reset_index()
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + "_" + agg['cell_y'].astype(int).astype(str)

# ==============================
# 9. Criar GeoDataFrame da grelha
# ==============================
grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({
            "cell_x": ix,
            "cell_y": iy,
            "cell_id": f"{ix}_{iy}",
            "geometry": box(x_edges_shifted[ix], y_edges_shifted[iy],
                            x_edges_shifted[ix+1], y_edges_shifted[iy+1])
        })
grid = gpd.GeoDataFrame(grid_data, crs="EPSG:3035").to_crs(epsg=3857)

# ==============================
# 10. Carregar precipitação (acumulado anual)
# ==============================
df_prec = pd.read_excel("data/prec.xlsx")
df_prec['data'] = pd.to_datetime(df_prec['data'])
df_prec['ano_hidrologico'] = df_prec['data'].apply(
    lambda x: x.year if x.month < 10 else x.year + 1
)

# Calcular acumulado dentro de cada ano hidrológico
df_prec['prec_acumulada'] = df_prec.groupby('ano_hidrologico')['prec'].cumsum()

# ==============================
# 11. Clustering de dV
# ==============================
agg_pivot = agg.pivot(index='cell_id', columns='date', values='dV').fillna(0)
k = 3
kmeans = KMeans(n_clusters=k, random_state=0)
cluster_labels = kmeans.fit_predict(agg_pivot)
cluster_df = pd.DataFrame({'cell_id': agg_pivot.index, 'cluster': cluster_labels})

# Mapear cores
cluster_colors = {i: color for i, color in enumerate(['red','green','blue','orange','purple'])}
grid_sel = grid.merge(cluster_df, on='cell_id', how='left')

import matplotlib.dates as mdates

# ==============================
# 14. Figura única: mapa + clusters (com precipitação anual acumulada)
# ==============================
clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)

fig = plt.figure(figsize=(20, 18))
gs = fig.add_gridspec(2, n_clusters, height_ratios=[2, 1.2])

# Linha 1: Mapa
ax_map = fig.add_subplot(gs[0, :])
grid.boundary.plot(ax=ax_map, color='lightgray', linewidth=0.5)
grid_sel.boundary.plot(ax=ax_map, color='black', linewidth=1, alpha=0.2)
for i, row in grid_sel.iterrows():
    if pd.notna(row['cluster']):
        gpd.GeoSeries([row['geometry']], crs=grid_sel.crs).plot(ax=ax_map,
                                                               color=cluster_colors[int(row['cluster'])],
                                                               alpha=0.4)
gdf_points.plot(ax=ax_map, color='white', edgecolor='black', markersize=40, label='Centro de massa ASC/DESC')

for cluster_id in clusters_present:
    ax_map.scatter([], [], color=cluster_colors[cluster_id], alpha=0.4, label=f'Cluster {cluster_id+1}')

ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)
ax_map.set_title("Mapa de Clusters de dV com Centro de Células ASC/DESC", fontsize=16)
ax_map.set_axis_off()
ax_map.legend(fontsize=10)

# Linha 2: Séries temporais
dV_min = agg['dV'].min()
dV_max = agg['dV'].max()
dV_margin = (dV_max - dV_min) * 0.1

for idx, cluster_id in enumerate(clusters_present):
    ax = fig.add_subplot(gs[1, idx])

    cluster_cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]

    # Linhas cinzentas individuais
    for cid in cluster_cells:
        ax.plot(cluster_data.columns, cluster_data.loc[cid], color='lightgray', alpha=0.7)

    # Média do cluster
    cluster_mean_dV = cluster_data.mean(axis=0)
    cluster_color = cluster_colors[cluster_id]
    ax.plot(cluster_data.columns, cluster_mean_dV, color=cluster_color, linewidth=2.5,
            label=f'Média Cluster {cluster_id + 1}')

    # ---------- Precipitação ----------
    ax2 = ax.twinx()

    # Barras = precipitação mensal
    ax2.bar(df_prec['data'], df_prec['prec'], color='deepskyblue', alpha=0.5,
            width=15, label='Precipitação (mm)')

    # Linha = acumulado anual
    for ano, grupo in df_prec.groupby('ano_hidrologico'):
        ax2.plot(grupo['data'], grupo['prec_acumulada'],
                 color='blue', linewidth=2, alpha=0.9,
                 label=f'Acumulado {ano}')

    # Eixos
    ax.set_ylim(dV_min - dV_margin, dV_max + dV_margin)
    ax2.set_ylim(0, df_prec['prec_acumulada'].max() * 1.1)

    ax.set_title(f'Cluster {cluster_id + 1} - {len(cluster_cells)} células', fontsize=12)
    ax.set_ylabel('dV (mm)')
    ax2.set_ylabel('Precipitação (mm)', color='blue')
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

    ax.grid(False)
    ax2.grid(False)

    # Legenda combinada
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, fontsize=8, loc='upper left')

plt.tight_layout()
plt.show()